In [1]:
# =============================================================================
# APTOS 2019 DR grading | EfficientNet-B1 | Pipeline v3 = v2 (steps 1-6) + 5-fold CV
#
#   Training recipe, preprocessing, loss, EMA, selection and temperature scaling
#   are IDENTICAL to v2. What is new:
#
#   DESIGN
#   - train.csv + valid.csv are POOLED (3294 images) and split into 5 stratified
#     folds. Each fold model trains on 4/5 and uses the held-out 1/5 for exactly
#     what v2 used "val" for (epoch selection by QWK+macro-F1 on the EMA model,
#     and fitting the temperature).
#   - The original test set (366 images) is NEVER used for selection or
#     calibration. Each fold model predicts it once; results are reported per
#     fold (mean/SD) and as a 5-model ensemble (reported separately).
#   - Master seed 42; fold k uses seed 42+k. No multi-seed repeats.
#
#   STATISTICS
#   - Out-of-fold (OOF) predictions over all 3294 images -> pooled metrics with
#     stratified-bootstrap 95% CIs (1000 reps).
#   - Cross-fitted temperature scaling for OOF calibration (T for fold k is fit
#     on the OTHER folds' OOF predictions) -> honest OOF ECE/NLL/Brier.
#   - Per-fold mean, SD, min/max; naive and Nadeau-Bengio corrected t-intervals
#     across folds (val side).
#   - Test: per-fold models (mean/SD) + ensemble with bootstrap CIs.
#   - Paired bootstrap TTA - noTTA on OOF predictions (is TTA worth it?).
#   - Fold results are cached to disk so an interrupted run can resume.
#
#   Everything is written to ONE CSV (column "fold" added).
# =============================================================================

import os, time, json, random, math, warnings, platform, gc, hashlib, copy, pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF

from scipy import stats as sps
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, average_precision_score,
    cohen_kappa_score, silhouette_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")


# -----------------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------------
class CFG:
    # ---------- paths ----------
    ROOT      = "/kaggle/input/datasets/mariaherrerot/aptos2019"
    TRAIN_DIR = f"{ROOT}/train_images/train_images"
    VAL_DIR   = f"{ROOT}/val_images/val_images"
    TEST_DIR  = f"{ROOT}/test_images/test_images"
    TRAIN_CSV = f"{ROOT}/train_1.csv"
    VAL_CSV   = f"{ROOT}/valid.csv"
    TEST_CSV  = f"{ROOT}/test.csv"
    OUT_DIR   = "/kaggle/working"
    CACHE_DIR = "/tmp/dr_cache"
    FIG_DIR   = f"{OUT_DIR}/figures_v3"

    # ---------- experiment identity ----------
    TAG  = "v3_cv5_seed42"
    SEED = 42

    # ---------- v3: cross-validation ----------
    N_FOLDS = 5
    RESUME  = True                         # reuse finished folds if config unchanged

    # ---------- data ----------
    IMG_SIZE    = 320
    NUM_CLASSES = 5
    CLASS_NAMES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative"]

    # ---------- paper-specified (unchanged) ----------
    EPOCHS          = 30
    BATCH_SIZE      = 16
    WEIGHT_DECAY    = 1e-4
    LABEL_SMOOTHING = 0.05

    # ---------- preprocessing ----------
    USE_CROP       = True
    CROP_TOL       = 7
    CROP_MIN_FRAC  = 0.05
    USE_CLAHE      = True
    CLAHE_CLIP     = 2.0
    CLAHE_TILE     = 8

    # ---------- optimisation ----------
    BACKBONE_LR = 2e-4
    HEAD_LR     = 1e-3
    PCT_START   = 0.1
    GRAD_CLIP   = 1.0
    EMA_DECAY   = 0.998

    # ---------- regularisation / imbalance ----------
    FOCAL_GAMMA  = 1.0
    WEIGHT_POWER = 0.5
    MIXUP_ALPHA  = 0.4
    MIX_PROB     = 0.25
    DROPOUT      = 0.2
    HIDDEN       = 512

    # ---------- model selection ----------
    SELECT_MIN_EPOCH = 8

    # ---------- other assumptions ----------
    TTA_ANGLE      = 10
    AUG_DEGREES    = 20
    AUG_TRANSLATE  = 0.05
    AUG_SCALE      = (0.9, 1.1)
    AUG_BRIGHTNESS = 0.15
    AUG_CONTRAST   = 0.15

    USE_AMP     = True
    NUM_WORKERS = 4
    N_JOBS      = 4

    INFER_NO_TTA = "noTTA"
    INFER_TTA    = "TTA"

    BOOTSTRAP_REPS = 1000


os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.CACHE_DIR, exist_ok=True)
os.makedirs(CFG.FIG_DIR, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
AMP_ON    = CFG.USE_AMP and DEVICE.type == "cuda"
MODES     = (CFG.INFER_NO_TTA, CFG.INFER_TTA)
K         = CFG.N_FOLDS


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def cfg_signature():
    d = {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]


CFG_SIG = cfg_signature()
set_seed(CFG.SEED)
print(f"Device {DEVICE} | GPUs {GPU_NAMES} | torch {torch.__version__} | py {platform.python_version()} | cfg {CFG_SIG}")


# -----------------------------------------------------------------------------
# 2. Data loading, split integrity, preprocessing cache
# -----------------------------------------------------------------------------
def load_df(csv_path):
    df = pd.read_csv(csv_path)
    id_col  = next((c for c in df.columns if c.lower() in
                    ("id_code", "image", "id", "image_id", "filename")), df.columns[0])
    lab_col = next((c for c in df.columns if c.lower() in
                    ("diagnosis", "label", "level", "dr_grade")), df.columns[1])
    df = df[[id_col, lab_col]].rename(columns={id_col: "id", lab_col: "label"})
    df["id"]    = df["id"].astype(str)
    df["label"] = df["label"].astype(int)
    assert df["label"].isin(range(CFG.NUM_CLASSES)).all(), f"bad labels in {csv_path}"
    return df


TRAIN_DF = load_df(CFG.TRAIN_CSV)
VAL_DF   = load_df(CFG.VAL_CSV)
TEST_DF  = load_df(CFG.TEST_CSV)

for name, df in (("train", TRAIN_DF), ("val", VAL_DF), ("test", TEST_DF)):
    assert df["id"].is_unique, f"duplicate IDs in {name}"
tr, va, te = map(set, (TRAIN_DF["id"], VAL_DF["id"], TEST_DF["id"]))
assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), "split overlap"
print("Split integrity OK.")

PAPER_COUNTS = {
    "train": [1434, 300, 808, 154, 234],
    "val":   [171, 40, 103, 22, 28],
    "test":  [199, 30, 87, 17, 33],
}
for name, df in (("train", TRAIN_DF), ("val", VAL_DF), ("test", TEST_DF)):
    got = np.bincount(df["label"].values, minlength=CFG.NUM_CLASSES).tolist()
    print(f"{name:5s} n={len(df):5d} | counts {got} | paper {PAPER_COUNTS[name]} | "
          f"{'MATCH' if got == PAPER_COUNTS[name] else 'DIFFERS'}")


def find_image(img_dir, img_id):
    for ext in ("", ".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        p = os.path.join(img_dir, f"{img_id}{ext}")
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"{img_id!r} not found in {img_dir}")


def crop_retina(rgb, tol=CFG.CROP_TOL, min_frac=CFG.CROP_MIN_FRAC):
    mask = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) > tol
    if mask.mean() < min_frac:
        return rgb
    h, w = mask.shape
    rows = np.flatnonzero(mask.sum(1) > max(2, 0.005 * w))
    cols = np.flatnonzero(mask.sum(0) > max(2, 0.005 * h))
    if len(rows) == 0 or len(cols) == 0:
        return rgb
    return rgb[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def pad_to_square(rgb):
    h, w = rgb.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    return cv2.copyMakeBorder(rgb, top, s - h - top, left, s - w - left,
                              cv2.BORDER_CONSTANT, value=(0, 0, 0))


def apply_clahe_lab(rgb, clip=CFG.CLAHE_CLIP, tile=CFG.CLAHE_TILE):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)


def load_and_preprocess(path, size):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"cannot read {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if CFG.USE_CROP:
        rgb = crop_retina(rgb)
    if CFG.USE_CLAHE:
        rgb = apply_clahe_lab(rgb)
    rgb = pad_to_square(rgb)
    interp = cv2.INTER_AREA if rgb.shape[0] >= size else cv2.INTER_CUBIC
    return cv2.resize(rgb, (size, size), interpolation=interp)


PREP_VERSION = "v2_crop_clahe_pad"          # unchanged -> v2 image caches are reused


def cache_fingerprint(df, paths, csv_path, tag):
    h = hashlib.sha1()
    h.update(json.dumps({
        "tag": tag, "prep": PREP_VERSION,
        "use_crop": CFG.USE_CROP, "crop_tol": CFG.CROP_TOL, "crop_min_frac": CFG.CROP_MIN_FRAC,
        "use_clahe": CFG.USE_CLAHE, "clahe_clip": CFG.CLAHE_CLIP, "clahe_tile": CFG.CLAHE_TILE,
        "img_size": CFG.IMG_SIZE, "n": len(df),
    }, sort_keys=True).encode())
    with open(csv_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    for p in paths:
        st = os.stat(p)
        h.update(f"{os.path.basename(p)}|{st.st_size}|{st.st_mtime_ns}".encode())
    return h.hexdigest()[:12]


def build_cache(df, img_dir, tag, csv_path):
    paths = [find_image(img_dir, i) for i in df["id"]]
    sig   = cache_fingerprint(df, paths, csv_path, tag)
    fn    = os.path.join(CFG.CACHE_DIR, f"{tag}_{PREP_VERSION}_s{CFG.IMG_SIZE}_{sig}.npy")
    if os.path.exists(fn):
        arr = np.load(fn)
        if arr.shape == (len(df), CFG.IMG_SIZE, CFG.IMG_SIZE, 3) and arr.dtype == np.uint8:
            print(f"cache hit: {fn}")
            return arr
        os.remove(fn)
    arr = Parallel(n_jobs=CFG.N_JOBS)(
        delayed(load_and_preprocess)(p, CFG.IMG_SIZE) for p in tqdm(paths, desc=f"cache {tag}"))
    arr = np.stack(arr).astype(np.uint8)
    np.save(fn, arr)
    print(f"cache saved: {fn}")
    return arr


X_TRAIN = build_cache(TRAIN_DF, CFG.TRAIN_DIR, "train", CFG.TRAIN_CSV)
X_VAL   = build_cache(VAL_DF,   CFG.VAL_DIR,   "val",   CFG.VAL_CSV)
X_TEST  = build_cache(TEST_DF,  CFG.TEST_DIR,  "test",  CFG.TEST_CSV)
Y_TEST  = TEST_DF["label"].values

# ---- v3: pool train + val for cross-validation; test stays untouched ----------
X_POOL  = np.concatenate([X_TRAIN, X_VAL])
Y_POOL  = np.concatenate([TRAIN_DF["label"].values, VAL_DF["label"].values])
ID_POOL = np.concatenate([TRAIN_DF["id"].values, VAL_DF["id"].values])
del X_TRAIN, X_VAL
gc.collect()
N_POOL = len(Y_POOL)
print(f"pool n={N_POOL} counts {np.bincount(Y_POOL, minlength=CFG.NUM_CLASSES).tolist()} | "
      f"test n={len(Y_TEST)} | X_POOL {X_POOL.shape}")

SKF   = StratifiedKFold(n_splits=K, shuffle=True, random_state=CFG.SEED)
FOLDS = list(SKF.split(np.zeros(N_POOL), Y_POOL))
FOLD_OF = np.empty(N_POOL, dtype=int)
for f, (_, va_i) in enumerate(FOLDS):
    FOLD_OF[va_i] = f
    print(f"fold {f}: train {len(FOLDS[f][0])} | val {len(va_i)} | "
          f"val counts {np.bincount(Y_POOL[va_i], minlength=CFG.NUM_CLASSES).tolist()}")


def save_preprocess_preview(X, y, tag, per_class=2):
    fig, ax = plt.subplots(CFG.NUM_CLASSES, per_class, figsize=(3 * per_class, 3 * CFG.NUM_CLASSES),
                           squeeze=False)
    rng = np.random.default_rng(0)
    for c in range(CFG.NUM_CLASSES):
        idx = np.flatnonzero(y == c)
        pick = rng.choice(idx, size=min(per_class, len(idx)), replace=False) if len(idx) else []
        for j in range(per_class):
            ax[c, j].axis("off")
            if j < len(pick):
                ax[c, j].imshow(X[pick[j]])
                ax[c, j].set_title(CFG.CLASS_NAMES[c], fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{CFG.FIG_DIR}/{tag}.png", dpi=100)
    plt.close()


save_preprocess_preview(X_POOL, Y_POOL, f"{CFG.TAG}_preprocess_preview")


# -----------------------------------------------------------------------------
# 3. Dataset, augmentations, model, loss, Mixup, TTA, EMA   (identical to v2)
# -----------------------------------------------------------------------------
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
BLACK_FILL = (-IMNET_MEAN / IMNET_STD).flatten().tolist()


class DRDataset(Dataset):
    def __init__(self, images, labels, train):
        self.images, self.labels, self.train = images, labels, train
        self.aug = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.5),
            T.RandomAffine(degrees=CFG.AUG_DEGREES,
                           translate=(CFG.AUG_TRANSLATE, CFG.AUG_TRANSLATE),
                           scale=CFG.AUG_SCALE),
            T.ColorJitter(brightness=CFG.AUG_BRIGHTNESS, contrast=CFG.AUG_CONTRAST),
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).permute(2, 0, 1).float() / 255.0
        if self.train:
            img = self.aug(img).clamp(0, 1)
        img = (img - IMNET_MEAN) / IMNET_STD
        return img, int(self.labels[i])


def make_loader(X, y, train):
    return DataLoader(
        DRDataset(X, y, train),
        batch_size=CFG.BATCH_SIZE if train else 64,
        shuffle=train, drop_last=train,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        persistent_workers=(train and CFG.NUM_WORKERS > 0),
    )


def mixup_batch(x, y, prob):
    if random.random() > prob:
        return x, y, y, 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    lam = float(np.random.beta(CFG.MIXUP_ALPHA, CFG.MIXUP_ALPHA))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma, label_smoothing):
        super().__init__()
        self.register_buffer("alpha", alpha.float())
        self.gamma, self.ls = gamma, label_smoothing

    def forward(self, logits, target):
        logits = logits.float()
        ce_s = F.cross_entropy(logits, target, reduction="none", label_smoothing=self.ls)
        pt   = torch.exp(-F.cross_entropy(logits, target, reduction="none"))
        return (self.alpha[target] * (1 - pt) ** self.gamma * ce_s).mean()


def class_weights(y):
    counts = np.bincount(y, minlength=CFG.NUM_CLASSES).astype(np.float64)
    w = 1.0 / (np.maximum(counts, 1) ** CFG.WEIGHT_POWER)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)


class DRNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.EfficientNet_B1_Weights.IMAGENET1K_V1 if pretrained else None
        m = torchvision.models.efficientnet_b1(weights=weights)
        self.features, self.pool = m.features, m.avgpool
        layers = []
        if CFG.DROPOUT > 0:
            layers.append(nn.Dropout(CFG.DROPOUT))
        layers += [nn.Linear(1280, CFG.HIDDEN), nn.ReLU(inplace=True)]
        if CFG.DROPOUT > 0:
            layers.append(nn.Dropout(CFG.DROPOUT))
        layers.append(nn.Linear(CFG.HIDDEN, CFG.NUM_CLASSES))
        self.head = nn.Sequential(*layers)

    def embed(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.head(self.embed(x))


class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.module = copy.deepcopy(model).eval()
        self._src = [v for v in model.state_dict().values() if v.dtype.is_floating_point]
        self._dst = [v for v in self.module.state_dict().values() if v.dtype.is_floating_point]

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(self._dst, d)
        torch._foreach_add_(self._dst, self._src, alpha=1 - d)


def tta_views(x, mode):
    if mode is None:
        return [x]
    views = [x, torch.flip(x, dims=[3]), torch.flip(x, dims=[2])]
    if mode == "TTA":
        views += [TF.rotate(x,  CFG.TTA_ANGLE, fill=BLACK_FILL),
                  TF.rotate(x, -CFG.TTA_ANGLE, fill=BLACK_FILL)]
    return views


@torch.no_grad()
def predict_probs(model, loader, mode=None):
    model.eval()
    out = []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", enabled=AMP_ON):
            p = torch.stack([F.softmax(model(v).float(), 1) for v in tta_views(x, mode)]).mean(0)
        out.append(p.cpu())
    return torch.cat(out).numpy() if out else np.empty((0, CFG.NUM_CLASSES), np.float32)


def predict_both(net, X, y):
    loader = make_loader(X, y, False)
    out = {CFG.INFER_NO_TTA: predict_probs(net, loader, None),
           CFG.INFER_TTA:    predict_probs(net, loader, "TTA")}
    del loader
    return out


# -----------------------------------------------------------------------------
# 4. Metrics, calibration, bootstrap, CV statistics
# -----------------------------------------------------------------------------
def _log(p):
    return np.log(np.clip(p, 1e-12, 1.0))


def apply_temperature(probs, T_):
    z = _log(probs) / T_
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def nll_score(probs, y):
    return float(-_log(probs)[np.arange(len(y)), y].mean())


def brier_score(probs, y):
    return float(((probs - np.eye(CFG.NUM_CLASSES)[y]) ** 2).sum(1).mean())


def fit_temperature(probs, y):
    return float(minimize_scalar(lambda t: nll_score(apply_temperature(probs, t), y),
                                 bounds=(0.3, 5.0), method="bounded").x)


def ece_score(probs, y, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bins_out = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            gap = abs(acc[m].mean() - conf[m].mean())
            ece += m.mean() * gap
            bins_out.append((float(lo), float(hi), int(m.sum()),
                             float(conf[m].mean()), float(acc[m].mean()), float(gap)))
    return float(ece), bins_out


def _sens_spec(yt, yp):
    """Binary sensitivity (TPR) and specificity (TNR); nan if a denominator is 0."""
    tp = int(((yt == 1) & (yp == 1)).sum()); fn = int(((yt == 1) & (yp == 0)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum()); fp = int(((yt == 0) & (yp == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    return sens, spec


def full_metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    ece, _ = ece_score(probs, y)
    m = {
        "acc": accuracy_score(y, pred),
        "macro_precision": p, "macro_recall": r, "macro_f1": f1,
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
        "qwk": cohen_kappa_score(y, pred, weights="quadratic"),
        "ece": ece,
    }
    try:
        yb = label_binarize(y, classes=list(range(CFG.NUM_CLASSES)))
        m["macro_auc"] = roc_auc_score(yb, probs, average="macro")
        m["macro_ap"]  = average_precision_score(yb, probs, average="macro")
    except Exception:
        m["macro_auc"] = m["macro_ap"] = float("nan")
    m["nll"]   = nll_score(probs, y)
    m["brier"] = brier_score(probs, y)
    # ---- sensitivity / specificity ----
    spec_c = [_sens_spec((y == c).astype(int), (pred == c).astype(int))[1]
              for c in range(CFG.NUM_CLASSES) if (y == c).any()]
    m["macro_specificity"] = float(np.nanmean(spec_c))          # one-vs-rest, macro; macro sensitivity == macro_recall
    for name, thr in (("ref", 2), ("any", 1)):                  # referable DR: grade>=2 ; any DR: grade>=1
        m[f"{name}_sens"], m[f"{name}_spec"] = _sens_spec((y >= thr).astype(int), (pred >= thr).astype(int))
    try:
        m["ref_auc"] = roc_auc_score((y >= 2).astype(int), probs[:, 2:].sum(1))
    except Exception:
        m["ref_auc"] = float("nan")
    return m


METRIC_KEYS = ["acc", "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
               "qwk", "ece", "macro_auc", "macro_ap", "nll", "brier",
               "macro_specificity", "ref_sens", "ref_spec", "ref_auc", "any_sens", "any_spec"]


def per_class_metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f1, sup = precision_recall_fscore_support(
        y, pred, labels=list(range(CFG.NUM_CLASSES)), zero_division=0)
    yb = label_binarize(y, classes=list(range(CFG.NUM_CLASSES)))
    rows = []
    for c in range(CFG.NUM_CLASSES):
        pos, neg = yb[:, c].sum() > 0, yb[:, c].sum() < len(yb)
        auc = roc_auc_score(yb[:, c], probs[:, c]) if pos and neg else float("nan")
        ap  = average_precision_score(yb[:, c], probs[:, c]) if pos else float("nan")
        cm = confusion_matrix(y, pred, labels=list(range(CFG.NUM_CLASSES)))
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + tp
        spec = tn / (tn + fp) if (tn + fp) else float("nan")
        rows.append({"cls": CFG.CLASS_NAMES[c], "precision": p[c], "recall": r[c],
                     "sensitivity": r[c], "specificity": spec,
                     "f1": f1[c], "support": int(sup[c]), "auc": auc, "ap": ap})
    return rows


def confidence_summary(probs, y):
    return {"mean_confidence": float(probs.max(1).mean()),
            "accuracy": float((probs.argmax(1) == y).mean()),
            "gap_conf_minus_acc": float(probs.max(1).mean() - (probs.argmax(1) == y).mean())}


def wilson_ci(k, n, z=1.96):
    if n <= 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def stratified_bootstrap_indices(y, rng):
    parts = []
    for cls in range(CFG.NUM_CLASSES):
        idx = np.flatnonzero(y == cls)
        if len(idx):
            parts.append(rng.choice(idx, size=len(idx), replace=True))
    return np.concatenate(parts) if parts else np.array([], dtype=int)


def bootstrap_metric_cis(y, probs, reps=1000, seed=42):
    rng = np.random.default_rng(seed)
    samples = {k: [] for k in METRIC_KEYS}
    for _ in range(reps):
        idx = stratified_bootstrap_indices(y, rng)
        m = full_metrics(y[idx], probs[idx])
        for k in METRIC_KEYS:
            v = m.get(k, np.nan)
            if np.isfinite(v):
                samples[k].append(float(v))
    out = {}
    for k in METRIC_KEYS:
        v = np.asarray(samples[k])
        out[k] = (float(np.quantile(v, .025)), float(np.quantile(v, .975))) if len(v) else (float("nan"),) * 2
    return out


QUICK_KEYS = ("acc", "macro_f1", "qwk", "ece", "nll")


def quick_metrics(y, probs):
    pred = probs.argmax(1)
    return {"acc": float((pred == y).mean()),
            "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
            "qwk": float(cohen_kappa_score(y, pred, weights="quadratic")),
            "ece": ece_score(probs, y)[0],
            "nll": nll_score(probs, y)}


def paired_bootstrap(y, probs_a, probs_b, reps=1000, seed=42, keys=QUICK_KEYS):
    """Paired stratified bootstrap of metric(A) - metric(B) on the SAME resamples."""
    rng = np.random.default_rng(seed)
    full_a, full_b = quick_metrics(y, probs_a), quick_metrics(y, probs_b)
    diffs = {k: [] for k in keys}
    for _ in range(reps):
        idx = stratified_bootstrap_indices(y, rng)
        a, b = quick_metrics(y[idx], probs_a[idx]), quick_metrics(y[idx], probs_b[idx])
        for k in keys:
            diffs[k].append(a[k] - b[k])
    out = {}
    for k in keys:
        v = np.asarray(diffs[k])
        out[k] = {"diff": full_a[k] - full_b[k],
                  "lo": float(np.quantile(v, .025)), "hi": float(np.quantile(v, .975)),
                  "p_gt0": float((v > 0).mean())}
    return out


def cv_stats(values, n_tr, n_va):
    """Across-fold summary. 'nb' = Nadeau-Bengio corrected t-interval, which
    accounts for the overlap of training sets between folds (the naive interval
    is too narrow). Only meaningful for metrics measured on the held-out fold."""
    v = np.asarray(values, dtype=float)
    k = len(v)
    mean, sd = float(np.nanmean(v)), float(np.nanstd(v, ddof=1))
    t = float(sps.t.ppf(0.975, k - 1))
    se_naive = sd / math.sqrt(k)
    se_nb    = sd * math.sqrt(1.0 / k + n_va / n_tr)
    return {"mean": mean, "sd": sd, "min": float(np.nanmin(v)), "max": float(np.nanmax(v)),
            "ci95_naive_low": mean - t * se_naive, "ci95_naive_high": mean + t * se_naive,
            "ci95_nb_low":    mean - t * se_nb,    "ci95_nb_high":    mean + t * se_nb}


# -----------------------------------------------------------------------------
# 5. Training one fold (identical recipe to v2)
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, crit):
    model.eval()
    lg_list, y_list = [], []
    for x, y in loader:
        with torch.autocast("cuda", enabled=AMP_ON):
            lg = model(x.to(DEVICE, non_blocking=True))
        lg_list.append(lg.float().cpu())
        y_list.append(y)
    logits, labels = torch.cat(lg_list), torch.cat(y_list)
    probs = F.softmax(logits, 1).numpy()
    m = full_metrics(labels.numpy(), probs)
    m["loss"] = crit(logits.to(DEVICE), labels.to(DEVICE)).item()
    m["nll"]  = F.cross_entropy(logits, labels).item()
    return m


def train_fold(fold, Xtr, ytr, Xva, yva):
    set_seed(CFG.SEED + fold)
    net  = DRNet().to(DEVICE)
    crit = FocalLoss(class_weights(ytr), CFG.FOCAL_GAMMA, CFG.LABEL_SMOOTHING).to(DEVICE)
    print(f"[fold {fold}] class weights:", [round(float(w), 3) for w in crit.alpha.cpu()])

    opt = torch.optim.AdamW(
        [{"params": net.features.parameters(), "lr": CFG.BACKBONE_LR},
         {"params": net.head.parameters(),     "lr": CFG.HEAD_LR}],
        weight_decay=CFG.WEIGHT_DECAY)
    train_loader = make_loader(Xtr, ytr, True)
    val_loader   = make_loader(Xva, yva, False)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[CFG.BACKBONE_LR, CFG.HEAD_LR],
        total_steps=CFG.EPOCHS * len(train_loader),
        pct_start=CFG.PCT_START, cycle_momentum=False)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ON)
    ema = EMA(net, CFG.EMA_DECAY)

    hist = {k: [] for k in [
        "train_loss", "train_acc", "train_acc_clean",
        "val_loss", "val_nll", "val_acc", "val_macro_f1", "val_qwk", "val_ece",
        "val_acc_raw", "val_macro_f1_raw", "val_qwk_raw",
        "select_score", "lr_backbone", "lr_head", "epoch_sec"]}
    best_score, best_ep, best_state, best_m = -1e9, 0, None, None
    ckpt = f"{CFG.FIG_DIR}/{CFG.TAG}_fold{fold}_best.pt"

    for ep in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        net.train()
        loss_sum, correct, seen = 0.0, 0, 0
        c_correct, c_seen = 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            xm, ya, yb, lam = mixup_batch(x, y, CFG.MIX_PROB)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=AMP_ON):
                logits = net(xm)
            loss = crit(logits, ya) if lam == 1.0 else \
                   lam * crit(logits, ya) + (1 - lam) * crit(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update()

            bs = x.size(0)
            loss_sum += loss.item() * bs
            tgt = ya if lam >= 0.5 else yb
            n_ok = (logits.argmax(1) == tgt).sum().item()
            correct += n_ok; seen += bs
            if lam == 1.0:
                c_correct += n_ok; c_seen += bs

        vm_ema = eval_epoch(ema.module, val_loader, crit)
        vm_raw = eval_epoch(net,        val_loader, crit)
        score  = vm_ema["qwk"] + vm_ema["macro_f1"]

        hist["train_loss"].append(loss_sum / max(seen, 1))
        hist["train_acc"].append(100 * correct / max(seen, 1))
        hist["train_acc_clean"].append(100 * c_correct / c_seen if c_seen else float("nan"))
        hist["val_loss"].append(vm_ema["loss"])
        hist["val_nll"].append(vm_ema["nll"])
        hist["val_acc"].append(100 * vm_ema["acc"])
        hist["val_macro_f1"].append(vm_ema["macro_f1"])
        hist["val_qwk"].append(vm_ema["qwk"])
        hist["val_ece"].append(vm_ema["ece"])
        hist["val_acc_raw"].append(100 * vm_raw["acc"])
        hist["val_macro_f1_raw"].append(vm_raw["macro_f1"])
        hist["val_qwk_raw"].append(vm_raw["qwk"])
        hist["select_score"].append(score)
        hist["lr_backbone"].append(opt.param_groups[0]["lr"])
        hist["lr_head"].append(opt.param_groups[1]["lr"])
        hist["epoch_sec"].append(time.time() - t0)

        msg = (f"[f{fold} ep {ep:02d}] tr_loss {hist['train_loss'][-1]:.4f} "
               f"tr_acc_clean {hist['train_acc_clean'][-1]:.1f} | "
               f"EMA val_acc {100*vm_ema['acc']:.2f} F1 {vm_ema['macro_f1']:.4f} "
               f"QWK {vm_ema['qwk']:.4f} NLL {vm_ema['nll']:.4f} ECE {vm_ema['ece']:.4f} | "
               f"raw acc {100*vm_raw['acc']:.2f} QWK {vm_raw['qwk']:.4f} | "
               f"lr {opt.param_groups[0]['lr']:.1e}/{opt.param_groups[1]['lr']:.1e}")
        if ep >= CFG.SELECT_MIN_EPOCH and score > best_score:
            best_score, best_ep, best_m = score, ep, vm_ema
            best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
            torch.save(best_state, ckpt)
            msg += "  <-- best"
        print(msg + f" | {hist['epoch_sec'][-1]:.0f}s")

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
        torch.save(best_state, ckpt)
        best_ep, best_score, best_m = CFG.EPOCHS, hist["select_score"][-1], vm_ema
    ema.module.load_state_dict(best_state)
    best_m = {k: float(v) for k, v in best_m.items()}
    del train_loader, val_loader, net
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ema.module, hist, best_ep, best_score, best_m, ckpt


# -----------------------------------------------------------------------------
# 6. Run the 5 folds (each fold model predicts held-out fold + test, once)
# -----------------------------------------------------------------------------
FOLD = {}
t_all = time.time()
for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    state_fn = f"{CFG.FIG_DIR}/{CFG.TAG}_fold{fold}_state.pkl"
    st = None
    if CFG.RESUME and os.path.exists(state_fn):
        try:
            with open(state_fn, "rb") as fh:
                st = pickle.load(fh)
            if (st.get("cfg_sig") != CFG_SIG or not os.path.exists(st["ckpt"])
                    or not np.array_equal(st["va_idx"], va_idx)):
                st = None
            else:
                print(f"[fold {fold}] resumed from {state_fn}")
        except Exception:
            st = None

    if st is None:
        print(f"\n{'=' * 25} FOLD {fold + 1}/{K} | train {len(tr_idx)} | val {len(va_idx)} {'=' * 25}")
        t0 = time.time()
        Xtr, ytr = X_POOL[tr_idx], Y_POOL[tr_idx]
        Xva, yva = X_POOL[va_idx], Y_POOL[va_idx]
        net, hist, best_ep, best_score, best_m, ckpt = train_fold(fold, Xtr, ytr, Xva, yva)
        train_sec = time.time() - t0
        val_probs  = predict_both(net, Xva, yva)
        test_probs = predict_both(net, X_TEST, Y_TEST)     # this model's one test evaluation
        st = {"cfg_sig": CFG_SIG, "va_idx": va_idx, "hist": hist, "best_ep": best_ep,
              "best_score": float(best_score), "best_m": best_m, "ckpt": ckpt,
              "train_sec": train_sec, "val_probs": val_probs, "test_probs": test_probs}
        with open(state_fn, "wb") as fh:
            pickle.dump(st, fh)
        print(f"[fold {fold}] best epoch {best_ep} | select score {best_score:.4f} | "
              f"val acc {100 * best_m['acc']:.2f}% | {train_sec / 60:.1f} min")
        del net, Xtr, Xva
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    FOLD[fold] = st

TOTAL_SEC = time.time() - t_all
print(f"\nAll folds done in {TOTAL_SEC / 60:.1f} min")


# -----------------------------------------------------------------------------
# 7. Statistics: per-fold, OOF, ensemble, bootstrap CIs, paired bootstrap
# -----------------------------------------------------------------------------
BLOCKS = []


def make_block(fold, split, inf, y, raw, cal, note=""):
    b = {"fold": fold, "split": split, "inf": inf, "y": y, "raw": raw, "cal": cal, "note": note,
         "m_raw": full_metrics(y, raw), "m_cal": full_metrics(y, cal)}
    BLOCKS.append(b)
    return b


T_FOLD = {f: {} for f in range(K)}
for f in range(K):
    st = FOLD[f]
    yva = Y_POOL[st["va_idx"]]
    for inf in MODES:
        Tk = fit_temperature(st["val_probs"][inf], yva)           # fit on this fold's val only
        T_FOLD[f][inf] = Tk
        make_block(f, "val", inf, yva, st["val_probs"][inf],
                   apply_temperature(st["val_probs"][inf], Tk),
                   note="T fit on this val fold (in-sample, optimistic)")
        make_block(f, "test", inf, Y_TEST, st["test_probs"][inf],
                   apply_temperature(st["test_probs"][inf], Tk),
                   note="T from this fold's val; test never used for fitting")

# ---- OOF predictions over all pooled images --------------------------------
OOF_RAW = {inf: np.zeros((N_POOL, CFG.NUM_CLASSES), np.float32) for inf in MODES}
for f in range(K):
    for inf in MODES:
        OOF_RAW[inf][FOLD[f]["va_idx"]] = FOLD[f]["val_probs"][inf]

T_CROSS = {f: {} for f in range(K)}
OOF_CAL = {inf: np.zeros_like(OOF_RAW[inf]) for inf in MODES}
for inf in MODES:
    for f in range(K):
        other = FOLD_OF != f
        Tx = fit_temperature(OOF_RAW[inf][other], Y_POOL[other])  # fit on the OTHER folds
        T_CROSS[f][inf] = Tx
        OOF_CAL[inf][~other] = apply_temperature(OOF_RAW[inf][~other], Tx)
    make_block("all", "oof", inf, Y_POOL, OOF_RAW[inf], OOF_CAL[inf],
               note="calibrated = cross-fitted T (honest)")

# ---- 5-model ensemble on the fixed test set --------------------------------
ENS_RAW, ENS_CAL = {}, {}
for inf in MODES:
    ENS_RAW[inf] = np.mean([FOLD[f]["test_probs"][inf] for f in range(K)], axis=0)
    ENS_CAL[inf] = np.mean([[b for b in BLOCKS if b["fold"] == f and b["split"] == "test"
                             and b["inf"] == inf][0]["cal"] for f in range(K)], axis=0)
    make_block("ensemble", "test", inf, Y_TEST, ENS_RAW[inf], ENS_CAL[inf],
               note="raw = mean of fold probs; calibrated = mean of per-fold calibrated probs")

# ---- bootstrap CIs (OOF + ensemble) ----------------------------------------
print("\nBootstrapping (this takes a few minutes)...")
for b in BLOCKS:
    if b["fold"] in ("all", "ensemble"):
        b["ci_raw"] = bootstrap_metric_cis(b["y"], b["raw"], reps=CFG.BOOTSTRAP_REPS, seed=CFG.SEED)
        b["ci_cal"] = bootstrap_metric_cis(b["y"], b["cal"], reps=CFG.BOOTSTRAP_REPS, seed=CFG.SEED)

# ---- paired bootstrap: TTA - noTTA on OOF (raw) ----------------------------
PAIRED_TTA = paired_bootstrap(Y_POOL, OOF_RAW[CFG.INFER_TTA], OOF_RAW[CFG.INFER_NO_TTA],
                              reps=CFG.BOOTSTRAP_REPS, seed=CFG.SEED)

# ---- across-fold statistics -------------------------------------------------
N_TR = float(np.mean([len(FOLDS[f][0]) for f in range(K)]))
N_VA = float(np.mean([len(FOLDS[f][1]) for f in range(K)]))
CV = {}
for split in ("val", "test"):
    for inf in MODES:
        for variant in ("raw", "cal"):
            for k in METRIC_KEYS:
                vals = [[b for b in BLOCKS if b["fold"] == f and b["split"] == split
                         and b["inf"] == inf][0]["m_" + variant][k] for f in range(K)]
                CV[(split, inf, variant, k)] = cv_stats(vals, N_TR, N_VA)


def blk(fold, split, inf):
    return [b for b in BLOCKS if b["fold"] == fold and b["split"] == split and b["inf"] == inf][0]


def fm(m, keys=("acc", "macro_f1", "qwk", "ece", "nll")):
    names = {"acc": "acc", "macro_f1": "F1", "qwk": "QWK", "ece": "ECE", "nll": "NLL"}
    return " | ".join(f"{names[k]} {m[k]:.4f}" for k in keys)


def fci(ci):
    return f"[{ci[0]:.4f}, {ci[1]:.4f}]"


def sens_line(m, ci):
    return (f"{'':10s} macro-spec {m['macro_specificity']:.4f} {fci(ci['macro_specificity'])} | "
            f"referable-DR sens {m['ref_sens']:.4f} {fci(ci['ref_sens'])} "
            f"spec {m['ref_spec']:.4f} {fci(ci['ref_spec'])} AUC {m['ref_auc']:.4f} | "
            f"any-DR sens {m['any_sens']:.4f} spec {m['any_spec']:.4f}")


print("\n" + "=" * 100)
print("PER-FOLD RESULTS (raw probabilities; T = fold temperature)")
for f in range(K):
    print(f"fold {f}: best epoch {FOLD[f]['best_ep']} | T noTTA {T_FOLD[f][CFG.INFER_NO_TTA]:.3f} "
          f"TTA {T_FOLD[f][CFG.INFER_TTA]:.3f}")
    for inf in MODES:
        print(f"   val  {inf:5s} | {fm(blk(f, 'val', inf)['m_raw'])}")
        print(f"   test {inf:5s} | {fm(blk(f, 'test', inf)['m_raw'])}"
              f"  || cal ECE {blk(f, 'test', inf)['m_cal']['ece']:.4f} NLL {blk(f, 'test', inf)['m_cal']['nll']:.4f}")

print("\n" + "=" * 100)
print("ACROSS-FOLD SUMMARY (mean ± SD). Val CI = Nadeau-Bengio corrected 95% t-interval.")
for inf in MODES:
    for split in ("val", "test"):
        parts = []
        for k, nm in (("acc", "acc"), ("macro_f1", "F1"), ("qwk", "QWK"), ("ece", "ECE"), ("nll", "NLL")):
            s = CV[(split, inf, "raw", k)]
            txt = f"{nm} {s['mean']:.4f}±{s['sd']:.4f}"
            if split == "val" and k in ("acc", "qwk"):
                txt += f" [{s['ci95_nb_low']:.4f}, {s['ci95_nb_high']:.4f}]"
            parts.append(txt)
        print(f"{split:4s} {inf:5s} | " + " | ".join(parts))
        s = CV[(split, inf, "cal", "ece")]
        n = CV[(split, inf, "cal", "nll")]
        print(f"{'':10s} calibrated: ECE {s['mean']:.4f}±{s['sd']:.4f} | NLL {n['mean']:.4f}±{n['sd']:.4f}")

print("\n" + "=" * 100)
print("OUT-OF-FOLD (all 3294 pooled images) with stratified-bootstrap 95% CI")
for inf in MODES:
    b = blk("all", "oof", inf)
    for variant, label in (("raw", "raw"), ("cal", "cross-fitted calibrated")):
        m, ci = b["m_" + variant], b["ci_" + variant]
        print(f"OOF {inf:5s} {label:24s} | "
              f"acc {m['acc']:.4f} {fci(ci['acc'])} | F1 {m['macro_f1']:.4f} {fci(ci['macro_f1'])} | "
              f"QWK {m['qwk']:.4f} {fci(ci['qwk'])} | ECE {m['ece']:.4f} {fci(ci['ece'])} | NLL {m['nll']:.4f}")
        print(sens_line(m, ci))
    cs = confidence_summary(b["raw"], Y_POOL)
    print(f"{'':10s} raw mean_conf {cs['mean_confidence']:.4f} acc {cs['accuracy']:.4f} "
          f"conf-acc {cs['gap_conf_minus_acc']:+.4f}")

print("\n" + "=" * 100)
print("TEST SET: 5-model ENSEMBLE with stratified-bootstrap 95% CI (reported separately from single models)")
for inf in MODES:
    b = blk("ensemble", "test", inf)
    for variant, label in (("raw", "raw"), ("cal", "calibrated")):
        m, ci = b["m_" + variant], b["ci_" + variant]
        print(f"ENS {inf:5s} {label:10s} | "
              f"acc {m['acc']:.4f} {fci(ci['acc'])} | F1 {m['macro_f1']:.4f} {fci(ci['macro_f1'])} | "
              f"QWK {m['qwk']:.4f} {fci(ci['qwk'])} | ECE {m['ece']:.4f} {fci(ci['ece'])} | NLL {m['nll']:.4f}")
        print(sens_line(m, ci))
    cs = confidence_summary(b["cal"], Y_TEST)
    print(f"{'':10s} calibrated conf-acc {cs['gap_conf_minus_acc']:+.4f} "
          f"(+ overconfident, - underconfident)")

print("\n" + "=" * 100)
print("PAIRED BOOTSTRAP on OOF: TTA minus noTTA (CI excluding 0 => real difference)")
for k in QUICK_KEYS:
    r = PAIRED_TTA[k]
    print(f"  {k:9s} diff {r['diff']:+.4f}  95% CI [{r['lo']:+.4f}, {r['hi']:+.4f}]  P(diff>0)={r['p_gt0']:.3f}")


# -----------------------------------------------------------------------------
# 8. Grad-CAM + t-SNE (single model: the fold with the best VALIDATION score)
# -----------------------------------------------------------------------------
class GradCAM:
    def __init__(self, net):
        self.net = net.eval()
        self.acts = None
        self.grads = None
        self.handle = net.features[-1].register_forward_hook(self._fwd)

    def _fwd(self, m, i, o):
        if o.requires_grad:
            self.acts = o
            o.register_hook(lambda g: setattr(self, "grads", g))

    def remove(self):
        self.handle.remove()

    def __call__(self, x, cls=None):
        self.net.zero_grad(set_to_none=True)
        logits = self.net(x)
        cls = int(logits.argmax(1).item()) if cls is None else int(cls)
        logits[0, cls].backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls


def save_gradcam_figure(net, X, y, pred, tag):
    net = net.to(DEVICE).float().eval()
    cam_fn = GradCAM(net)
    picks = []
    for c in range(CFG.NUM_CLASSES):
        idx = np.where((y == c) & (pred == c))[0]
        if len(idx):
            picks.append((idx[0], "correct"))
    wrong = np.where(y != pred)[0]
    picks += [(i, "WRONG") for i in wrong[:4]]
    if not picks:
        cam_fn.remove()
        return
    fig, ax = plt.subplots(2, len(picks), figsize=(3 * len(picks), 6), squeeze=False)
    for j, (i, kind) in enumerate(picks):
        x = ((torch.from_numpy(X[i]).permute(2, 0, 1).float() / 255
              - IMNET_MEAN) / IMNET_STD).unsqueeze(0).to(DEVICE)
        cam, cam_pred = cam_fn(x)
        heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8),
                                              cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay = (0.55 * X[i].astype(np.float32) + 0.45 * heat.astype(np.float32)
                   ).clip(0, 255).astype(np.uint8)
        ax[0, j].imshow(X[i])
        ax[0, j].set_title(f"True: {CFG.CLASS_NAMES[y[i]]}", fontsize=9)
        ax[0, j].axis("off")
        ax[1, j].imshow(overlay)
        ax[1, j].set_title(f"Pred: {CFG.CLASS_NAMES[cam_pred]} ({kind})", fontsize=9)
        ax[1, j].axis("off")
    plt.tight_layout()
    plt.savefig(f"{CFG.FIG_DIR}/{tag}.png", dpi=120)
    plt.close()
    cam_fn.remove()


@torch.no_grad()
def extract_embeddings(net, X):
    net = net.to(DEVICE).eval()
    feats = []
    for xb, _ in make_loader(X, np.zeros(len(X), dtype=int), False):
        with torch.autocast("cuda", enabled=AMP_ON):
            feats.append(net.embed(xb.to(DEVICE)).float().cpu())
    return torch.cat(feats).numpy()


def space_stats(Z, y, metric, suffix):
    stats_ = {f"silhouette_gt_{suffix}": float(silhouette_score(Z, y, metric=metric))}
    present = [c for c in range(CFG.NUM_CLASSES) if (y == c).any()]
    cent = {c: Z[y == c].mean(0) for c in present}
    stats_[f"mean_inter_centroid_distance_{suffix}"] = float(np.mean([
        np.linalg.norm(cent[a] - cent[b]) for i, a in enumerate(present) for b in present[i + 1:]]))
    for c in present:
        stats_[f"intra_class_spread_{CFG.CLASS_NAMES[c]}_{suffix}"] = float(
            np.linalg.norm(Z[y == c] - cent[c], axis=1).mean())
    return stats_


VIS_FOLD = int(max(FOLD, key=lambda f: FOLD[f]["best_score"]))      # chosen on val score, not test
print(f"\nGrad-CAM / t-SNE use fold {VIS_FOLD} (highest validation selection score).")
VIS_NET = DRNet(pretrained=False)
VIS_NET.load_state_dict(torch.load(FOLD[VIS_FOLD]["ckpt"], map_location="cpu"))
VIS_NET = VIS_NET.to(DEVICE).eval()
VIS_PRED = FOLD[VIS_FOLD]["test_probs"][CFG.INFER_TTA].argmax(1)

save_gradcam_figure(VIS_NET, X_TEST, Y_TEST, VIS_PRED, tag=f"{CFG.TAG}_gradcam_fold{VIS_FOLD}")

EMB       = extract_embeddings(VIS_NET, X_TEST)
EMB_STATS = space_stats(EMB, Y_TEST, "cosine", "original_1280d")
perplexity = float(min(30, max(2, (len(EMB) - 1) // 3)))
Z2 = TSNE(n_components=2, perplexity=perplexity, init="pca",
          random_state=CFG.SEED).fit_transform(EMB)
EMB_STATS.update(space_stats(Z2, Y_TEST, "euclidean", "tsne_coords"))

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for c in range(CFG.NUM_CLASSES):
    ax[0].scatter(*Z2[Y_TEST == c].T, s=12, label=CFG.CLASS_NAMES[c])
    ax[1].scatter(*Z2[VIS_PRED == c].T, s=12, label=CFG.CLASS_NAMES[c])
ax[0].set_title(f"t-SNE GT (fold {VIS_FOLD} model)")
ax[1].set_title("t-SNE pred")
ax[0].legend(); ax[1].legend()
plt.tight_layout()
plt.savefig(f"{CFG.FIG_DIR}/{CFG.TAG}_tsne_fold{VIS_FOLD}.png", dpi=120)
plt.close()
print("Grad-CAM and t-SNE figures saved.")


# -----------------------------------------------------------------------------
# 9. Write EVERYTHING to ONE CSV
# -----------------------------------------------------------------------------
ROWS = []


def add(section, metric, value, split="", inference="", cls="", epoch="", note="", fold=""):
    ROWS.append({"section": section, "fold": fold, "split": split, "inference": inference,
                 "class": cls, "epoch": epoch, "metric": metric, "value": value, "note": note})


V_NOTES = {
    "IMG_SIZE": "v2 step 1 (v1/paper: 224)",
    "USE_CROP": "v2 step 2", "CROP_TOL": "v2 step 2", "CROP_MIN_FRAC": "v2 step 2",
    "BACKBONE_LR": "v2 step 3", "HEAD_LR": "v2 step 3", "PCT_START": "v2 step 3",
    "GRAD_CLIP": "v2 step 3", "EMA_DECAY": "v2 step 3",
    "FOCAL_GAMMA": "v2 step 4 (v1/paper: 2.0)", "WEIGHT_POWER": "v2 step 4 (v1: 1.0)",
    "MIXUP_ALPHA": "v2 step 4 (CutMix removed)", "MIX_PROB": "v2 step 4 (v1: 0.5)",
    "DROPOUT": "v2 step 4 (v1: 0.0)",
    "SELECT_MIN_EPOCH": "v2 step 5; selection score = val QWK + macro-F1 on EMA model",
    "N_FOLDS": "v3 new: stratified K-fold on pooled train+val",
    "RESUME": "v3 new",
}
assumption_keys = {
    "CLAHE_CLIP", "CLAHE_TILE", "HIDDEN", "TTA_ANGLE", "AUG_DEGREES", "AUG_TRANSLATE",
    "AUG_SCALE", "AUG_BRIGHTNESS", "AUG_CONTRAST", "USE_AMP", "USE_CLAHE",
    "NUM_WORKERS", "N_JOBS", "BOOTSTRAP_REPS", "SEED", "TAG",
    "OUT_DIR", "CACHE_DIR", "FIG_DIR", "ROOT",
    "TRAIN_DIR", "VAL_DIR", "TEST_DIR", "TRAIN_CSV", "VAL_CSV", "TEST_CSV",
}
for k, v in vars(CFG).items():
    if not k.startswith("_") and not callable(v):
        add("config", k, v,
            note=V_NOTES.get(k, "assumption" if k in assumption_keys else "paper-specified"))
add("config", "CLASS_WEIGHT_IMPLEMENTATION", "inv_freq^WEIGHT_POWER_mean1",
    note="weights = 1/count^0.5 rescaled to class-mean 1, recomputed per fold from its training labels")
add("config", "PREPROCESS_ORDER", "crop -> CLAHE(LAB-L) -> pad_to_square -> resize")
add("config", "FOLD_SEED_RULE", "SEED + fold_index", note="master seed 42; stratified split random_state=42")
add("config", "CV_DESIGN", "pooled train+val, stratified 5-fold; test untouched",
    note="test never used for epoch selection or temperature fitting")

add("run_info", "torch_version", torch.__version__)
add("run_info", "python_version", platform.python_version())
add("run_info", "gpus_visible", "; ".join(GPU_NAMES) if GPU_NAMES else "cpu")
add("run_info", "config_signature", CFG_SIG)
add("run_info", "pool_size", N_POOL)
add("run_info", "test_size", int(len(Y_TEST)))
add("run_info", "total_run_seconds_this_session", round(TOTAL_SEC, 1),
    note="folds resumed from cache contribute ~0 s")
add("run_info", "vis_fold_gradcam_tsne", VIS_FOLD, note="fold with best validation selection score")
for f in range(K):
    st = FOLD[f]
    add("run_info", "best_epoch", st["best_ep"], fold=f)
    add("run_info", "best_select_score_qwk_plus_macro_f1", st["best_score"], fold=f)
    add("run_info", "val_acc_at_best_epoch_ema", st["best_m"]["acc"], fold=f)
    add("run_info", "val_qwk_at_best_epoch_ema", st["best_m"]["qwk"], fold=f)
    add("run_info", "val_macro_f1_at_best_epoch_ema", st["best_m"]["macro_f1"], fold=f)
    add("run_info", "training_seconds", round(st["train_sec"], 1), fold=f)
    add("run_info", "n_train", int(len(FOLDS[f][0])), fold=f)
    add("run_info", "n_val", int(len(FOLDS[f][1])), fold=f)

for i in range(N_POOL):
    add("fold_assignment", str(ID_POOL[i]), int(FOLD_OF[i]), cls=CFG.CLASS_NAMES[Y_POOL[i]])

for f in range(K):
    for i, ep in enumerate(range(1, CFG.EPOCHS + 1)):
        for k in FOLD[f]["hist"]:
            add("history", k, FOLD[f]["hist"][k][i], epoch=ep, fold=f)


def report_block(b):
    fold, split, inf, y = b["fold"], b["split"], b["inf"], b["y"]
    kw = dict(fold=fold, split=split, inference=inf)
    for variant in ("raw", "cal"):
        probs, m = b[variant], b["m_" + variant]
        sfx = "" if variant == "raw" else "_calibrated"
        note = b["note"] if variant == "cal" else ""
        for k in METRIC_KEYS:
            add("overall" + sfx, k, m[k], note=note, **kw)
        lo, hi = wilson_ci(int((probs.argmax(1) == y).sum()), len(y))
        add("overall" + sfx, "acc_wilson_ci95_low",  lo, **kw)
        add("overall" + sfx, "acc_wilson_ci95_high", hi, **kw)
        for row in per_class_metrics(y, probs):
            for k in ("precision", "recall", "sensitivity", "specificity", "f1", "support", "auc", "ap"):
                add("per_class" + sfx, k, row[k], cls=row["cls"], **kw)
        cm = confusion_matrix(y, probs.argmax(1), labels=range(CFG.NUM_CLASSES))
        for i, tname in enumerate(CFG.CLASS_NAMES):
            for j, pname in enumerate(CFG.CLASS_NAMES):
                add("confusion" + sfx, f"pred_{pname}", int(cm[i, j]), cls=tname, **kw)
        _, bins = ece_score(probs, y)
        for lo_, hi_, cnt, mc, ac, gp in bins:
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_count",     cnt, **kw)
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_mean_conf", mc,  **kw)
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_accuracy",  ac,  **kw)
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_gap",       gp,  **kw)
        for k, v in confidence_summary(probs, y).items():
            add("calibration_summary" + sfx, k, v, **kw)
        ci = b.get("ci_" + variant)
        if ci:
            for metric, (lo_, hi_) in ci.items():
                add("bootstrap_ci" + sfx, f"{metric}_ci95_low",  lo_, **kw)
                add("bootstrap_ci" + sfx, f"{metric}_ci95_high", hi_, **kw)


for b in BLOCKS:
    report_block(b)

# temperatures
for f in range(K):
    for inf in MODES:
        add("temperature_scaling", "temperature_fold_val_fit", T_FOLD[f][inf], fold=f, inference=inf,
            note="fit on this fold's val; applied to this fold's val and test")
        add("temperature_scaling", "temperature_cross_fit_oof", T_CROSS[f][inf], fold=f, inference=inf,
            note="fit on the other folds' OOF; used only for honest OOF calibration")

# across-fold statistics
for (split, inf, variant, k), s in CV.items():
    sec = "cv_summary" if variant == "raw" else "cv_summary_calibrated"
    for stat, val in s.items():
        if split == "test" and stat.startswith("ci95"):
            continue                                   # test CI across folds ignores test-sampling noise
        add(sec, f"{k}_{stat}", val, fold="cv", split=split, inference=inf,
            note=("Nadeau-Bengio corrected" if stat.startswith("ci95_nb") else
                  "naive t-interval (too narrow, folds overlap)" if stat.startswith("ci95_naive") else
                  "variation across fold-trained models on the fixed test set" if split == "test" else ""))

# paired bootstrap TTA - noTTA
for k, r in PAIRED_TTA.items():
    add("paired_bootstrap_oof", f"{k}_diff_TTA_minus_noTTA", r["diff"], fold="all", split="oof")
    add("paired_bootstrap_oof", f"{k}_ci95_low",  r["lo"],    fold="all", split="oof")
    add("paired_bootstrap_oof", f"{k}_ci95_high", r["hi"],    fold="all", split="oof")
    add("paired_bootstrap_oof", f"{k}_p_diff_gt0", r["p_gt0"], fold="all", split="oof")

for k, v in EMB_STATS.items():
    add("embedding", k, v, split="test", inference=CFG.INFER_TTA, fold=VIS_FOLD)

# ---- references --------------------------------------------------------------
BASELINE_V1 = {
    ("val",  CFG.INFER_NO_TTA): {"acc": 0.8607, "macro_f1": 0.7324, "weighted_f1": 0.8589, "qwk": 0.9179, "ece": 0.0654},
    ("val",  CFG.INFER_TTA):    {"acc": 0.8607, "macro_f1": 0.7573, "weighted_f1": 0.8607, "qwk": 0.9204, "ece": 0.0666},
    ("test", CFG.INFER_NO_TTA): {"acc": 0.8251, "macro_f1": 0.6389, "weighted_f1": 0.8261, "qwk": 0.9030, "ece": 0.0351},
    ("test", CFG.INFER_TTA):    {"acc": 0.8197, "macro_f1": 0.6285, "weighted_f1": 0.8199, "qwk": 0.9036, "ece": 0.0484},
}
BASELINE_V2 = {
    ("val",  CFG.INFER_NO_TTA): {"acc": 0.8552, "macro_f1": 0.7325, "weighted_f1": 0.8518, "qwk": 0.9158, "ece": 0.0790},
    ("val",  CFG.INFER_TTA):    {"acc": 0.8388, "macro_f1": 0.7064, "weighted_f1": 0.8350, "qwk": 0.9041, "ece": 0.0745},
    ("test", CFG.INFER_NO_TTA): {"acc": 0.8525, "macro_f1": 0.6863, "weighted_f1": 0.8476, "qwk": 0.9062, "ece": 0.0822},
    ("test", CFG.INFER_TTA):    {"acc": 0.8579, "macro_f1": 0.6969, "weighted_f1": 0.8543, "qwk": 0.9104, "ece": 0.0587},
}
for sec, ref, nt in (("baseline_v1_reference", BASELINE_V1, "v1 pipeline, 224px, single split, seed 42"),
                     ("baseline_v2_reference", BASELINE_V2, "v2 pipeline (steps 1-5), 320px, single split, seed 42")):
    for (split, inf), r in ref.items():
        for k, v in r.items():
            add(sec, k, v, split=split, inference=inf, note=nt)

PAPER_OVERALL = {
    ("val",  CFG.INFER_NO_TTA): {"acc": 0.8407, "macro_f1": 0.6962},
    ("test", CFG.INFER_NO_TTA): {"acc": 0.8361, "macro_f1": 0.6434, "weighted_f1": 0.8262,
                                 "macro_precision": 0.7122, "macro_recall": 0.6217,
                                 "macro_auc": 0.8949, "macro_ap": 0.6836, "ece": 0.0327},
    ("test", CFG.INFER_TTA):    {"acc": 0.8497, "macro_f1": 0.6627, "weighted_f1": 0.8400,
                                 "macro_precision": 0.7354, "macro_recall": 0.6375,
                                 "macro_auc": 0.9099, "macro_ap": 0.6767, "ece": 0.0239},
}
for (split, inf), ref in PAPER_OVERALL.items():
    for k, v in ref.items():
        add("paper_reference", k, v, split=split, inference=inf, note="paper Tables 3/4/7")

PAPER_CLASS = {
    CFG.INFER_NO_TTA: {
        "No_DR": (0.9751, 0.9849, 0.9800), "Mild": (0.6154, 0.5333, 0.5714),
        "Moderate": (0.6847, 0.8736, 0.7677), "Severe": (0.4286, 0.3529, 0.3871),
        "Proliferative": (0.8571, 0.3636, 0.5106),
    },
    CFG.INFER_TTA: {
        "No_DR": (0.9799, 0.9799, 0.9799), "Mild": (0.7500, 0.6000, 0.6667),
        "Moderate": (0.6957, 0.9195, 0.7921), "Severe": (0.3846, 0.2941, 0.3333),
        "Proliferative": (0.8667, 0.3939, 0.5417),
    },
}
for inf, table in PAPER_CLASS.items():
    for cname, (p, r, f1) in table.items():
        add("paper_reference_per_class", "precision", p,  split="test", inference=inf, cls=cname)
        add("paper_reference_per_class", "recall",    r,  split="test", inference=inf, cls=cname)
        add("paper_reference_per_class", "f1",        f1, split="test", inference=inf, cls=cname)

CSV_PATH = os.path.join(CFG.OUT_DIR, f"{CFG.TAG}_results.csv")
pd.DataFrame(ROWS, columns=["section", "fold", "split", "inference", "class",
                            "epoch", "metric", "value", "note"]
             ).to_csv(CSV_PATH, index=False)
print(f"\nSaved single CSV: {CSV_PATH}")
print(f"Rows: {len(ROWS)}")
print("Done. Send me the CSV + console log.")

Device cuda | GPUs ['Tesla T4', 'Tesla T4'] | torch 2.11.0+cu128 | py 3.13.15 | cfg b651d46cce14
Split integrity OK.
train n= 2930 | counts [1434, 300, 808, 154, 234] | paper [1434, 300, 808, 154, 234] | MATCH
val   n=  366 | counts [172, 40, 104, 22, 28] | paper [171, 40, 103, 22, 28] | DIFFERS
test  n=  366 | counts [199, 30, 87, 17, 33] | paper [199, 30, 87, 17, 33] | MATCH


cache train:   0%|          | 0/2930 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/train_v2_crop_clahe_pad_s320_39e7dc8e75db.npy


cache val:   0%|          | 0/366 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/val_v2_crop_clahe_pad_s320_45aa701f176d.npy


cache test:   0%|          | 0/366 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/test_v2_crop_clahe_pad_s320_b8e22ce90a39.npy
pool n=3296 counts [1606, 340, 912, 176, 262] | test n=366 | X_POOL (3296, 320, 320, 3)
fold 0: train 2636 | val 660 | val counts [321, 68, 183, 36, 52]
fold 1: train 2637 | val 659 | val counts [321, 68, 183, 35, 52]
fold 2: train 2637 | val 659 | val counts [321, 68, 182, 35, 53]
fold 3: train 2637 | val 659 | val counts [321, 68, 182, 35, 53]
fold 4: train 2637 | val 659 | val counts [322, 68, 182, 35, 52]

========================= FOLD 1/5 | train 2636 | val 660 =========================
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 105MB/s] 


[fold 0] class weights: [0.5, 1.086, 0.664, 1.514, 1.236]
[f0 ep 01] tr_loss 0.7221 tr_acc_clean 61.8 | EMA val_acc 75.45 F1 0.4310 QWK 0.8070 NLL 0.7533 ECE 0.1540 | raw acc 76.52 QWK 0.8238 | lr 5.6e-05/2.8e-04 | 100s
[f0 ep 02] tr_loss 0.4896 tr_acc_clean 74.5 | EMA val_acc 77.27 F1 0.5681 QWK 0.8551 NLL 0.5858 ECE 0.0551 | raw acc 77.58 QWK 0.8758 | lr 1.5e-04/7.6e-04 | 25s
[f0 ep 03] tr_loss 0.4646 tr_acc_clean 77.1 | EMA val_acc 80.00 F1 0.6050 QWK 0.8815 NLL 0.5092 ECE 0.0414 | raw acc 77.27 QWK 0.8637 | lr 2.0e-04/1.0e-03 | 25s
[f0 ep 04] tr_loss 0.4005 tr_acc_clean 80.8 | EMA val_acc 80.61 F1 0.6339 QWK 0.8905 NLL 0.4911 ECE 0.0469 | raw acc 78.94 QWK 0.8646 | lr 2.0e-04/1.0e-03 | 26s
[f0 ep 05] tr_loss 0.3524 tr_acc_clean 83.3 | EMA val_acc 81.97 F1 0.6673 QWK 0.9016 NLL 0.4661 ECE 0.0250 | raw acc 77.42 QWK 0.8721 | lr 2.0e-04/9.9e-04 | 25s
[f0 ep 06] tr_loss 0.3251 tr_acc_clean 84.7 | EMA val_acc 83.48 F1 0.6926 QWK 0.9022 NLL 0.4593 ECE 0.0318 | raw acc 81.82 QWK 0.8786 | 

In [1]:
# =============================================================================
# APTOS 2019 | EfficientNet-B1 | Optical-robustness study: Regime A and Regime B
#
#   Same model / recipe as v3 (OneCycle, EMA, selection by QWK+macro-F1, 320 px,
#   crop+pad, 5-fold CV on pooled train+val, fixed test set).
#   Only the DATA side changes:
#
#   Order of operations (acquisition -> processing):
#       raw fundus -> crop -> pad-to-square -> resize 320   (cached, NO CLAHE)
#                  -> [corruption]                          (acquisition)
#                  -> CLAHE (LAB-L)                         (processing)
#                  -> augmentation -> normalise
#
#   REGIME "A": train on clean images only.          select on CLEAN val fold.
#   REGIME "B": each training image corrupted with probability 0.5 (random type,
#               random severity 1-3, BEFORE CLAHE).  select on a fixed MIXED
#               val fold (half clean, half corrupted).
#
#   Both regimes: same folds, same seeds (SEED+fold), same hyper-parameters.
#   Every fold model is evaluated on the SAME 10 conditions
#       clean + {defocus, haze, noise} x severity {1,2,3}
#   on (i) its held-out fold (-> pooled out-of-fold predictions) and
#       (ii) the fixed test set.
#   Corrupted evaluation images are generated deterministically per
#   (EVAL_SEED, condition, image id) so every model sees IDENTICAL images.
#
#   METRICS (the only addition relative to the previous v4 script)
#   Every metric of the v3 pipeline is now computed for EVERY one of the 10
#   conditions and written to ONE CSV per regime ({TAG}_results.csv), in the same
#   long layout as v3 (section / fold / split / inference / class / epoch /
#   metric / value / note) plus one extra column "condition":
#     overall, per_class, confusion, calibration_bin, calibration_summary,
#     bootstrap_ci (+ the "_calibrated" twin of each), temperature_scaling,
#     cv_summary (+ calibrated, Nadeau-Bengio), embedding (t-SNE/silhouette on the
#     clean test set), history, run_info, fold_assignment, config, references.
#   Blocks: per-fold held-out ("val"), per-fold test ("test"), pooled
#   out-of-fold ("oof", bootstrap CIs, cross-fitted temperature) and the 5-model
#   test ensemble ("ensemble", bootstrap CIs).  Temperature scaling is fitted per
#   condition exactly as v3 fitted it per inference mode.  v4 has no TTA, so the
#   "inference" column is always "noTTA" and the v3 TTA-vs-noTTA paired bootstrap
#   does not apply.  Grad-CAM / t-SNE figures use the clean test set.
#   Nothing about preprocessing, corruption, model, training or seeds changed.
#
#   NEW (figures only, Regime B only, no effect on training or results):
#   section 4b saves two paper figures showing what the corrupted TRAINING images
#   look like -> {TAG}_samples_pipeline_stages.png and {TAG}_samples_training_batch.png
#   in FIG_DIR.  Runs automatically when REGIME == "B" (also in GRID_ONLY / SMOKE).
#
#   RUN ORDER   1) GRID_ONLY=True  -> look at Figure 1, lock corruption values
#               2) SMOKE=True      -> tiny end-to-end test (REGIME A, then B)
#               3) REGIME="A"      -> full 5-fold run   -> v4_robust_A_probs.pkl
#               4) REGIME="B"      -> full 5-fold run   -> v4_robust_B_probs.pkl
#               5) run dr_robustness_analysis.py
#   Change ONLY the three switches at the top of CFG between runs.
# =============================================================================

import os, time, json, random, math, warnings, platform, gc, hashlib, copy, pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

from scipy import stats as sps
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, average_precision_score,
    cohen_kappa_score, silhouette_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")


# -----------------------------------------------------------------------------
# 1. Config  (freeze this before the final runs)
# -----------------------------------------------------------------------------
class CFG:
    # ---------- the ONLY switches you change between runs ----------
    REGIME    = "A"        # "A" = clean-trained, "B" = corruption-trained
    GRID_ONLY = False      # True: build caches, save Figure 1 + sanity stats, stop
    SMOKE     = False      # True: tiny end-to-end test (fold 0 only, 3 epochs)

    # ---------- paths ----------
    ROOT      = "/kaggle/input/datasets/mariaherrerot/aptos2019"
    TRAIN_DIR = f"{ROOT}/train_images/train_images"
    VAL_DIR   = f"{ROOT}/val_images/val_images"
    TEST_DIR  = f"{ROOT}/test_images/test_images"
    TRAIN_CSV = f"{ROOT}/train_1.csv"
    VAL_CSV   = f"{ROOT}/valid.csv"
    TEST_CSV  = f"{ROOT}/test.csv"
    OUT_DIR   = "/kaggle/working"
    CACHE_DIR = "/tmp/dr_cache"
    FIG_DIR   = f"{OUT_DIR}/figures_v4"

    # ---------- identity / seeds ----------
    TAG        = f"v4_robust_{REGIME}" + ("_smoke" if SMOKE else "")
    SEED       = 42        # fold k trains with seed SEED + k (same for A and B)
    SPLIT_SEED = 42        # fold assignment (never change -> same folds for A and B)
    N_FOLDS    = 5
    RESUME     = True

    # ---------- data ----------
    IMG_SIZE    = 320
    NUM_CLASSES = 5
    CLASS_NAMES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative"]

    # ---------- corruption model (LOCK after viewing Figure 1) ----------
    BASE_RES     = 224                   # severities below are defined at 224 px
    CORR_TYPES   = ("defocus", "haze", "noise")
    DEFOCUS_SIGMA = (1.0, 2.0, 3.0)      # Gaussian blur sigma (px at 224) -> scaled to IMG_SIZE
    HAZE_T        = (0.8, 0.6, 0.4)      # transmission;  I = J*t + A*(1-t)
    HAZE_A        = 0.8                  # airlight
    NOISE_STD     = (0.03, 0.06, 0.10)   # additive Gaussian, image in [0,1], then clipped
    P_CORRUPT     = {"A": 0.0, "B": 0.5}[REGIME]
    EVAL_SEED     = 20260101             # fixed corrupted evaluation sets
    VALMIX_SEED   = 777                  # fixed mixed validation fold (Regime B)

    # ---------- training recipe (identical to v3) ----------
    EPOCHS          = 3 if SMOKE else 30
    BATCH_SIZE      = 16
    WEIGHT_DECAY    = 1e-4
    LABEL_SMOOTHING = 0.05
    CROP_TOL        = 7
    CROP_MIN_FRAC   = 0.05
    CLAHE_CLIP      = 2.0
    CLAHE_TILE      = 8
    BACKBONE_LR     = 2e-4
    HEAD_LR         = 1e-3
    PCT_START       = 0.1
    GRAD_CLIP       = 1.0
    EMA_DECAY       = 0.998
    FOCAL_GAMMA     = 1.0
    WEIGHT_POWER    = 0.5
    MIXUP_ALPHA     = 0.4
    MIX_PROB        = 0.25
    DROPOUT         = 0.2
    HIDDEN          = 512
    SELECT_MIN_EPOCH = 1 if SMOKE else 8
    AUG_DEGREES     = 20
    AUG_TRANSLATE   = 0.05
    AUG_SCALE       = (0.9, 1.1)
    AUG_BRIGHTNESS  = 0.15
    AUG_CONTRAST    = 0.15
    USE_AMP         = True
    NUM_WORKERS     = 4
    N_JOBS          = 4

    # ---------- analysis only (does not affect training; excluded from CFG_SIG) ----------
    BOOTSTRAP_REPS  = 1000


os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.CACHE_DIR, exist_ok=True)
os.makedirs(CFG.FIG_DIR, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
AMP_ON    = CFG.USE_AMP and DEVICE.type == "cuda"
K         = CFG.N_FOLDS
NC        = CFG.NUM_CLASSES

CONDITIONS = ["clean"] + [f"{k}_s{l}" for k in CFG.CORR_TYPES for l in (1, 2, 3)]
COND_ID    = {c: i for i, c in enumerate(CONDITIONS)}

# Seed for the Regime-B sample figures (section 4b). Deliberately a module-level constant and
# NOT a CFG attribute: adding a CFG key would change CFG_SIG and invalidate resumed fold states.
SAMPLE_SEED = 2026


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


# analysis-only keys are left out of the signature so that fold states trained with the
# previous version of this script (same signature) can still be resumed.
ANALYSIS_ONLY_KEYS = {"BOOTSTRAP_REPS"}


def cfg_signature():
    d = {k: v for k, v in vars(CFG).items()
         if not k.startswith("_") and not callable(v) and k not in ANALYSIS_ONLY_KEYS}
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]


CFG_SIG = cfg_signature()
set_seed(CFG.SEED)
print(f"REGIME {CFG.REGIME} | p_corrupt {CFG.P_CORRUPT} | smoke {CFG.SMOKE} | grid_only {CFG.GRID_ONLY} | "
      f"device {DEVICE} | torch {torch.__version__} | cfg {CFG_SIG}")


# -----------------------------------------------------------------------------
# 2. Corruption model  (acquisition stage; images are uint8 RGB at IMG_SIZE)
# -----------------------------------------------------------------------------
def defocus(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigmaX=float(sigma))


def haze(img, t, A=CFG.HAZE_A):
    return np.clip(img * t + A * (1.0 - t), 0.0, 1.0)


def noise(img, s, rng):
    return np.clip(img + rng.normal(0.0, s, img.shape).astype(np.float32), 0.0, 1.0)


def corrupt(img, kind, level, rng):
    """img float32 in [0,1]; level in {1,2,3}."""
    if kind == "defocus":   # keep the SAME relative blur when the working resolution is not 224
        return defocus(img, CFG.DEFOCUS_SIGMA[level - 1] * CFG.IMG_SIZE / CFG.BASE_RES)
    if kind == "haze":
        return haze(img, CFG.HAZE_T[level - 1])
    if kind == "noise":
        return noise(img, CFG.NOISE_STD[level - 1], rng)
    raise ValueError(kind)


def corrupt_u8(raw_u8, kind, level, rng):
    img = raw_u8.astype(np.float32) / 255.0
    out = corrupt(img, kind, level, rng)
    return (np.clip(out, 0.0, 1.0) * 255.0 + 0.5).astype(np.uint8)


def parse_cond(cond):
    kind, lv = cond.rsplit("_s", 1)
    return kind, int(lv)


def apply_clahe_lab(rgb, clip=CFG.CLAHE_CLIP, tile=CFG.CLAHE_TILE):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)


def condition_image(raw_u8, cond, gid):
    """Deterministic model input (uint8, post-CLAHE) for a condition. gid = global image id."""
    if cond == "clean":
        return apply_clahe_lab(raw_u8)
    kind, level = parse_cond(cond)
    rng = np.random.default_rng([CFG.EVAL_SEED, COND_ID[cond], int(gid)])
    return apply_clahe_lab(corrupt_u8(raw_u8, kind, level, rng))


# -----------------------------------------------------------------------------
# 3. Data: frozen splits, RAW (pre-CLAHE) cache, CLAHE arrays, folds
# -----------------------------------------------------------------------------
def load_df(csv_path):
    df = pd.read_csv(csv_path)
    id_col  = next((c for c in df.columns if c.lower() in
                    ("id_code", "image", "id", "image_id", "filename")), df.columns[0])
    lab_col = next((c for c in df.columns if c.lower() in
                    ("diagnosis", "label", "level", "dr_grade")), df.columns[1])
    df = df[[id_col, lab_col]].rename(columns={id_col: "id", lab_col: "label"})
    df["id"]    = df["id"].astype(str)
    df["label"] = df["label"].astype(int)
    assert df["label"].isin(range(NC)).all(), f"bad labels in {csv_path}"
    return df


TRAIN_DF, VAL_DF, TEST_DF = load_df(CFG.TRAIN_CSV), load_df(CFG.VAL_CSV), load_df(CFG.TEST_CSV)
for name, df in (("train", TRAIN_DF), ("val", VAL_DF), ("test", TEST_DF)):
    assert df["id"].is_unique, f"duplicate IDs in {name}"
_tr, _va, _te = map(set, (TRAIN_DF["id"], VAL_DF["id"], TEST_DF["id"]))
assert _tr.isdisjoint(_va) and _tr.isdisjoint(_te) and _va.isdisjoint(_te), "split overlap"
print("Split integrity OK.")


def find_image(img_dir, img_id):
    for ext in ("", ".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        p = os.path.join(img_dir, f"{img_id}{ext}")
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"{img_id!r} not found in {img_dir}")


def crop_retina(rgb, tol=CFG.CROP_TOL, min_frac=CFG.CROP_MIN_FRAC):
    mask = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) > tol
    if mask.mean() < min_frac:
        return rgb
    h, w = mask.shape
    rows = np.flatnonzero(mask.sum(1) > max(2, 0.005 * w))
    cols = np.flatnonzero(mask.sum(0) > max(2, 0.005 * h))
    if len(rows) == 0 or len(cols) == 0:
        return rgb
    return rgb[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def pad_to_square(rgb):
    h, w = rgb.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    return cv2.copyMakeBorder(rgb, top, s - h - top, left, s - w - left,
                              cv2.BORDER_CONSTANT, value=(0, 0, 0))


def load_raw(path, size):
    """crop -> pad -> resize.  NO CLAHE (CLAHE comes after the corruption)."""
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"cannot read {path}")
    rgb = pad_to_square(crop_retina(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
    interp = cv2.INTER_AREA if rgb.shape[0] >= size else cv2.INTER_CUBIC
    return cv2.resize(rgb, (size, size), interpolation=interp)


PREP_VERSION = "v4_raw_crop_pad_resize"


def cache_fingerprint(df, paths, csv_path, tag):
    h = hashlib.sha1()
    h.update(json.dumps({"tag": tag, "prep": PREP_VERSION, "img_size": CFG.IMG_SIZE,
                         "crop_tol": CFG.CROP_TOL, "crop_min_frac": CFG.CROP_MIN_FRAC,
                         "n": len(df)}, sort_keys=True).encode())
    with open(csv_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    for p in paths:
        st = os.stat(p)
        h.update(f"{os.path.basename(p)}|{st.st_size}|{st.st_mtime_ns}".encode())
    return h.hexdigest()[:12]


def build_cache(df, img_dir, tag, csv_path):
    paths = [find_image(img_dir, i) for i in df["id"]]
    sig = cache_fingerprint(df, paths, csv_path, tag)
    fn = os.path.join(CFG.CACHE_DIR, f"{tag}_{PREP_VERSION}_s{CFG.IMG_SIZE}_{sig}.npy")
    if os.path.exists(fn):
        arr = np.load(fn)
        if arr.shape == (len(df), CFG.IMG_SIZE, CFG.IMG_SIZE, 3) and arr.dtype == np.uint8:
            print(f"cache hit: {fn}")
            return arr
        os.remove(fn)
    arr = Parallel(n_jobs=CFG.N_JOBS)(
        delayed(load_raw)(p, CFG.IMG_SIZE) for p in tqdm(paths, desc=f"cache {tag}"))
    arr = np.stack(arr).astype(np.uint8)
    np.save(fn, arr)
    print(f"cache saved: {fn}")
    return arr


def clahe_stack(X):
    return np.stack(Parallel(n_jobs=CFG.N_JOBS)(delayed(apply_clahe_lab)(x) for x in X)).astype(np.uint8)


_xtr = build_cache(TRAIN_DF, CFG.TRAIN_DIR, "train", CFG.TRAIN_CSV)
_xva = build_cache(VAL_DF,   CFG.VAL_DIR,   "val",   CFG.VAL_CSV)
X_RAW_POOL = np.concatenate([_xtr, _xva]); del _xtr, _xva
X_RAW_TEST = build_cache(TEST_DF, CFG.TEST_DIR, "test", CFG.TEST_CSV)
Y_POOL  = np.concatenate([TRAIN_DF["label"].values, VAL_DF["label"].values])
ID_POOL = np.concatenate([TRAIN_DF["id"].values, VAL_DF["id"].values])
Y_TEST  = TEST_DF["label"].values
N_POOL  = len(Y_POOL)
TEST_GIDS = 100000 + np.arange(len(Y_TEST))          # separate id space for deterministic seeding
X_CLAHE_POOL = clahe_stack(X_RAW_POOL)               # clean model input (post-CLAHE)
X_CLAHE_TEST = clahe_stack(X_RAW_TEST)
print(f"pool n={N_POOL} counts {np.bincount(Y_POOL, minlength=NC).tolist()} | "
      f"test n={len(Y_TEST)} counts {np.bincount(Y_TEST, minlength=NC).tolist()}")

FOLDS = list(StratifiedKFold(n_splits=K, shuffle=True, random_state=CFG.SPLIT_SEED)
             .split(np.zeros(N_POOL), Y_POOL))
FOLD_OF = np.empty(N_POOL, dtype=int)
for f, (_, va_i) in enumerate(FOLDS):
    FOLD_OF[va_i] = f


# ---- Figure 1 + corruption sanity checks (Step 3 "check") --------------------
def save_corruption_grid(raw_img, tag):
    cells = {"clean": raw_img}
    for kind in CFG.CORR_TYPES:
        for lv in (1, 2, 3):
            cond = f"{kind}_s{lv}"
            cells[cond] = corrupt_u8(raw_img, kind, lv, np.random.default_rng([CFG.EVAL_SEED, COND_ID[cond], 0]))
    for post, name in ((False, "acquired (before CLAHE)"), (True, "model input (after CLAHE)")):
        fig, ax = plt.subplots(len(CFG.CORR_TYPES), 4, figsize=(12, 9))
        for r, kind in enumerate(CFG.CORR_TYPES):
            for c in range(4):
                cond = "clean" if c == 0 else f"{kind}_s{c}"
                im = apply_clahe_lab(cells[cond]) if post else cells[cond]
                ax[r, c].imshow(im); ax[r, c].axis("off"); ax[r, c].set_title(cond, fontsize=10)
        fig.suptitle(f"Figure 1 - {name}")
        plt.tight_layout()
        plt.savefig(f"{CFG.FIG_DIR}/{tag}_{'after' if post else 'before'}_clahe.png", dpi=110)
        plt.close()


def corruption_sanity(n=24):
    rng = np.random.default_rng(0)
    pick = rng.choice(N_POOL, size=n, replace=False)
    print("\nCorruption sanity (mean |corrupted - clean| on 0-255 scale, before CLAHE; must grow with severity):")
    ok = True
    for kind in CFG.CORR_TYPES:
        vals = []
        for lv in (1, 2, 3):
            cond = f"{kind}_s{lv}"
            d = [np.abs(corrupt_u8(X_RAW_POOL[i], kind, lv, np.random.default_rng([CFG.EVAL_SEED, COND_ID[cond], int(i)])).astype(np.float32)
                        - X_RAW_POOL[i].astype(np.float32)).mean() for i in pick]
            vals.append(float(np.mean(d)))
        mono = vals[0] < vals[1] < vals[2]
        ok &= mono
        print(f"  {kind:8s} s1 {vals[0]:6.2f} | s2 {vals[1]:6.2f} | s3 {vals[2]:6.2f} | monotonic {mono}")
    return ok


_i_demo = int(np.flatnonzero(Y_POOL == 2)[0])                # first Moderate image (fixed rule)
save_corruption_grid(X_RAW_POOL[_i_demo], f"{CFG.TAG}_figure1")
SANITY_OK = corruption_sanity()
print("Figure 1 saved to", CFG.FIG_DIR)


# -----------------------------------------------------------------------------
# 4. Datasets and loaders
# -----------------------------------------------------------------------------
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def to_tensor_norm(img_u8):
    x = torch.from_numpy(np.ascontiguousarray(img_u8)).permute(2, 0, 1).float() / 255.0
    return x


class DRTrainSet(Dataset):
    """Training set. With prob P_CORRUPT an image is corrupted (random type, random
    severity 1-3) from the RAW cache BEFORE CLAHE; otherwise the cached clean CLAHE
    image is used. Same augmentation for both regimes."""
    def __init__(self, indices, p_corrupt):
        self.indices, self.p = np.asarray(indices), float(p_corrupt)
        self._rng = None
        self.aug = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.5),
            T.RandomAffine(degrees=CFG.AUG_DEGREES,
                           translate=(CFG.AUG_TRANSLATE, CFG.AUG_TRANSLATE),
                           scale=CFG.AUG_SCALE),
            T.ColorJitter(brightness=CFG.AUG_BRIGHTNESS, contrast=CFG.AUG_CONTRAST),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        if self._rng is None:                               # per-worker stream, reproducible from the global seed
            self._rng = np.random.default_rng(torch.initial_seed() % (2 ** 32))
        g = int(self.indices[i])
        if self.p > 0 and self._rng.random() < self.p:
            kind = CFG.CORR_TYPES[int(self._rng.integers(len(CFG.CORR_TYPES)))]
            level = int(self._rng.integers(1, 4))
            img = apply_clahe_lab(corrupt_u8(X_RAW_POOL[g], kind, level, self._rng))
        else:
            img = X_CLAHE_POOL[g]
        x = self.aug(to_tensor_norm(img)).clamp(0, 1)
        return (x - IMNET_MEAN) / IMNET_STD, int(Y_POOL[g])


class ArrayDataset(Dataset):
    """Already-processed (post-CLAHE) uint8 images, no augmentation (validation)."""
    def __init__(self, images, labels):
        self.images, self.labels = images, labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (to_tensor_norm(self.images[i]) - IMNET_MEAN) / IMNET_STD, int(self.labels[i])


class CondDataset(Dataset):
    """Deterministic evaluation set for one condition."""
    def __init__(self, raw, clahe, labels, gids, cond):
        self.raw, self.clahe, self.labels, self.gids, self.cond = raw, clahe, labels, gids, cond

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        img = self.clahe[i] if self.cond == "clean" else condition_image(self.raw[i], self.cond, self.gids[i])
        return (to_tensor_norm(img) - IMNET_MEAN) / IMNET_STD, int(self.labels[i])


def eval_loader(ds):
    return DataLoader(ds, batch_size=64, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)


def build_mixed_val(fold, va_idx):
    """Regime B selection set: FIXED, half clean / half corrupted. The corrupted half is a
    random (seeded) subset with the 9 corruption x severity combinations used in balance."""
    rng = np.random.default_rng([CFG.VALMIX_SEED, fold])
    order = rng.permutation(len(va_idx))
    combos = [(k, l) for k in CFG.CORR_TYPES for l in (1, 2, 3)]
    out = X_CLAHE_POOL[va_idx].copy()
    for j, pos in enumerate(order[: len(va_idx) // 2]):
        kind, lvl = combos[j % len(combos)]
        r = np.random.default_rng([CFG.VALMIX_SEED, fold, int(va_idx[pos])])
        out[pos] = apply_clahe_lab(corrupt_u8(X_RAW_POOL[va_idx[pos]], kind, lvl, r))
    return out


# -----------------------------------------------------------------------------
# 4b. Regime B sample figures for the paper (preprocessing section).  Figures only:
#     own RNG streams, global torch RNG state is saved/restored, nothing here
#     touches training, selection, evaluation or the CSV.  Not run for Regime A
#     (clean images only).
#       fig 1: {TAG}_samples_pipeline_stages.png
#              one image per DR grade, columns = every stage of the Regime-B
#              training input pipeline:
#              acquired (crop+pad+resize) -> + corruption -> + CLAHE -> + augmentation
#       fig 2: {TAG}_samples_training_batch.png
#              a montage of images exactly as the network receives them during
#              Regime-B training (p = 0.5: a mix of clean and corrupted images,
#              random type, random severity 1-3, augmentation applied)
# -----------------------------------------------------------------------------
def _chw_to_u8(x):
    return (x.permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)


def save_regime_b_samples(n_rows=4, n_cols=6):
    ds = DRTrainSet(np.arange(N_POOL), CFG.P_CORRUPT)          # used only for its augmentation pipeline
    rng = np.random.default_rng(SAMPLE_SEED)
    with torch.random.fork_rng(devices=[]):                    # restore the global torch RNG afterwards
        torch.manual_seed(SAMPLE_SEED)

        # ---- figure 1: pipeline stages, one image per DR grade -----------------
        titles = ["1. Acquired\n(crop + pad + resize)", "2. + Corruption\n(before CLAHE)",
                  "3. + CLAHE (LAB-L)", "4. + Augmentation\n(network input)"]
        fig, ax = plt.subplots(NC, 4, figsize=(12, 3 * NC), squeeze=False)
        for c in range(NC):
            g = int(rng.choice(np.flatnonzero(Y_POOL == c)))
            kind = CFG.CORR_TYPES[c % len(CFG.CORR_TYPES)]
            lvl = 1 + (c % 3 + c // 3) % 3                      # deterministic spread of types and severities
            r = np.random.default_rng([SAMPLE_SEED, g])
            raw = X_RAW_POOL[g]
            corr = corrupt_u8(raw, kind, lvl, r)
            clahe = apply_clahe_lab(corr)
            aug = _chw_to_u8(ds.aug(to_tensor_norm(clahe)).clamp(0, 1))
            for j, im in enumerate((raw, corr, clahe, aug)):
                ax[c, j].imshow(im)
                ax[c, j].axis("off")
                if c == 0:
                    ax[c, j].set_title(titles[j], fontsize=10)
            ax[c, 0].text(0.02, 0.97, CFG.CLASS_NAMES[c], transform=ax[c, 0].transAxes, fontsize=9,
                          color="white", va="top", bbox=dict(facecolor="black", alpha=0.6, pad=2))
            ax[c, 1].text(0.02, 0.97, f"{kind} s{lvl}", transform=ax[c, 1].transAxes, fontsize=9,
                          color="white", va="top", bbox=dict(facecolor="black", alpha=0.6, pad=2))
        fig.suptitle("Regime B training pipeline: from acquired image to network input", fontsize=12)
        plt.tight_layout()
        plt.savefig(f"{CFG.FIG_DIR}/{CFG.TAG}_samples_pipeline_stages.png", dpi=200)
        plt.close()

        # ---- figure 2: montage of training inputs as drawn by the Regime-B sampler ----
        n = n_rows * n_cols
        picks = rng.choice(N_POOL, size=n, replace=False)
        fig, ax = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.9 * n_rows), squeeze=False)
        n_corr = 0
        for t, g in enumerate(picks):
            g = int(g)
            if rng.random() < CFG.P_CORRUPT:                    # same sampling rule as DRTrainSet.__getitem__
                kind = CFG.CORR_TYPES[int(rng.integers(len(CFG.CORR_TYPES)))]
                lvl = int(rng.integers(1, 4))
                img = apply_clahe_lab(corrupt_u8(X_RAW_POOL[g], kind, lvl, rng))
                tag = f"{kind} s{lvl}"
                n_corr += 1
            else:
                img = X_CLAHE_POOL[g]
                tag = "clean"
            x = _chw_to_u8(ds.aug(to_tensor_norm(img)).clamp(0, 1))
            a = ax[t // n_cols, t % n_cols]
            a.imshow(x)
            a.axis("off")
            a.set_title(f"{CFG.CLASS_NAMES[int(Y_POOL[g])]}\n{tag}", fontsize=8)
        fig.suptitle(f"Regime B: training inputs after corruption, CLAHE and augmentation "
                     f"(p_corrupt = {CFG.P_CORRUPT}; {n_corr}/{n} corrupted in this sample)", fontsize=11)
        plt.tight_layout()
        plt.savefig(f"{CFG.FIG_DIR}/{CFG.TAG}_samples_training_batch.png", dpi=200)
        plt.close()
    print(f"Regime B sample figures saved to {CFG.FIG_DIR} "
          f"({CFG.TAG}_samples_pipeline_stages.png, {CFG.TAG}_samples_training_batch.png)")


if CFG.REGIME == "B":
    try:
        save_regime_b_samples()
    except Exception as e:                                      # figures must never block a run
        print(f"WARNING: Regime B sample figures skipped ({type(e).__name__}: {e})")


# -----------------------------------------------------------------------------
# 5. Model, loss, Mixup, EMA   (identical to v3)
# -----------------------------------------------------------------------------
def mixup_batch(x, y, prob):
    if random.random() > prob:
        return x, y, y, 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    lam = float(np.random.beta(CFG.MIXUP_ALPHA, CFG.MIXUP_ALPHA))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma, label_smoothing):
        super().__init__()
        self.register_buffer("alpha", alpha.float())
        self.gamma, self.ls = gamma, label_smoothing

    def forward(self, logits, target):
        logits = logits.float()
        ce_s = F.cross_entropy(logits, target, reduction="none", label_smoothing=self.ls)
        pt   = torch.exp(-F.cross_entropy(logits, target, reduction="none"))
        return (self.alpha[target] * (1 - pt) ** self.gamma * ce_s).mean()


def class_weights(y):
    counts = np.bincount(y, minlength=NC).astype(np.float64)
    w = 1.0 / (np.maximum(counts, 1) ** CFG.WEIGHT_POWER)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


class DRNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.EfficientNet_B1_Weights.IMAGENET1K_V1 if pretrained else None
        m = torchvision.models.efficientnet_b1(weights=weights)
        self.features, self.pool = m.features, m.avgpool
        layers = []
        if CFG.DROPOUT > 0:
            layers.append(nn.Dropout(CFG.DROPOUT))
        layers += [nn.Linear(1280, CFG.HIDDEN), nn.ReLU(inplace=True)]
        if CFG.DROPOUT > 0:
            layers.append(nn.Dropout(CFG.DROPOUT))
        layers.append(nn.Linear(CFG.HIDDEN, NC))
        self.head = nn.Sequential(*layers)

    def embed(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.head(self.embed(x))


class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.module = copy.deepcopy(model).eval()
        self._src = [v for v in model.state_dict().values() if v.dtype.is_floating_point]
        self._dst = [v for v in self.module.state_dict().values() if v.dtype.is_floating_point]

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(self._dst, d)
        torch._foreach_add_(self._dst, self._src, alpha=1 - d)


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    out = []
    for x, _ in loader:
        with torch.autocast("cuda", enabled=AMP_ON):
            p = F.softmax(model(x.to(DEVICE, non_blocking=True)).float(), 1)
        out.append(p.cpu())
    return torch.cat(out).numpy()


def predict_conditions(net, raw, clahe, labels, gids):
    """Probabilities for ALL 10 conditions (no TTA)."""
    return {cond: predict_probs(net, eval_loader(CondDataset(raw, clahe, labels, gids, cond)))
            for cond in CONDITIONS}


# -----------------------------------------------------------------------------
# 6. Metrics (fast, confusion-matrix based; used for epoch selection)
# -----------------------------------------------------------------------------
W_Q = ((np.arange(NC)[:, None] - np.arange(NC)[None, :]) ** 2) / (NC - 1) ** 2


def cm_of(y, p):
    return np.bincount(np.asarray(y) * NC + np.asarray(p), minlength=NC * NC).reshape(NC, NC)


def qwk_cm(cm):
    n = cm.sum()
    E = np.outer(cm.sum(1), cm.sum(0)) / n
    return float(1.0 - (W_Q * cm).sum() / (W_Q * E).sum())


def macro_f1_cm(cm):
    tp = np.diag(cm).astype(float)
    d = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    return float(np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0).mean())


def ece_score(probs, y, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def cond_metrics(y, probs):
    pred = probs.argmax(1)
    cm = cm_of(y, pred)
    pos, tn, fp = cm[2:, 2:].sum(), cm[:2, :2].sum(), cm[:2, 2:].sum()
    fn = cm[2:, :2].sum()
    return {"acc": float(np.trace(cm) / cm.sum()), "qwk": qwk_cm(cm), "macro_f1": macro_f1_cm(cm),
            "ref_sens": float(pos / max(pos + fn, 1)), "ref_spec": float(tn / max(tn + fp, 1)),
            "ece": ece_score(probs, y),
            "nll": float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1)).mean())}


# -----------------------------------------------------------------------------
# 6b. Extended metrics = the full v3 metric set
#     (calibration, per-class, bootstrap CIs, cross-validation statistics)
# -----------------------------------------------------------------------------
INF = "noTTA"          # v4 has no TTA; this label keeps the CSV layout identical to v3


def _log(p):
    return np.log(np.clip(p, 1e-12, 1.0))


def apply_temperature(probs, T_):
    z = _log(probs) / T_
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def nll_score(probs, y):
    return float(-_log(probs)[np.arange(len(y)), y].mean())


def brier_score(probs, y):
    return float(((probs - np.eye(NC)[y]) ** 2).sum(1).mean())


def fit_temperature(probs, y):
    return float(minimize_scalar(lambda t: nll_score(apply_temperature(probs, t), y),
                                 bounds=(0.3, 5.0), method="bounded").x)


def ece_with_bins(probs, y, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bins_out = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            gap = abs(acc[m].mean() - conf[m].mean())
            ece += m.mean() * gap
            bins_out.append((float(lo), float(hi), int(m.sum()),
                             float(conf[m].mean()), float(acc[m].mean()), float(gap)))
    return float(ece), bins_out


def _sens_spec(yt, yp):
    """Binary sensitivity (TPR) and specificity (TNR); nan if a denominator is 0."""
    tp = int(((yt == 1) & (yp == 1)).sum()); fn = int(((yt == 1) & (yp == 0)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum()); fp = int(((yt == 0) & (yp == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    return sens, spec


def full_metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    ece, _ = ece_with_bins(probs, y)
    m = {
        "acc": accuracy_score(y, pred),
        "macro_precision": p, "macro_recall": r, "macro_f1": f1,
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
        "qwk": cohen_kappa_score(y, pred, weights="quadratic"),
        "ece": ece,
    }
    try:
        yb = label_binarize(y, classes=list(range(NC)))
        m["macro_auc"] = roc_auc_score(yb, probs, average="macro")
        m["macro_ap"]  = average_precision_score(yb, probs, average="macro")
    except Exception:
        m["macro_auc"] = m["macro_ap"] = float("nan")
    m["nll"]   = nll_score(probs, y)
    m["brier"] = brier_score(probs, y)
    # ---- sensitivity / specificity ----
    spec_c = [_sens_spec((y == c).astype(int), (pred == c).astype(int))[1]
              for c in range(NC) if (y == c).any()]
    m["macro_specificity"] = float(np.nanmean(spec_c))          # one-vs-rest, macro; macro sensitivity == macro_recall
    for name, thr in (("ref", 2), ("any", 1)):                  # referable DR: grade>=2 ; any DR: grade>=1
        m[f"{name}_sens"], m[f"{name}_spec"] = _sens_spec((y >= thr).astype(int), (pred >= thr).astype(int))
    try:
        m["ref_auc"] = roc_auc_score((y >= 2).astype(int), probs[:, 2:].sum(1))
    except Exception:
        m["ref_auc"] = float("nan")
    return m


METRIC_KEYS = ["acc", "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
               "qwk", "ece", "macro_auc", "macro_ap", "nll", "brier",
               "macro_specificity", "ref_sens", "ref_spec", "ref_auc", "any_sens", "any_spec"]


def per_class_metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f1, sup = precision_recall_fscore_support(
        y, pred, labels=list(range(NC)), zero_division=0)
    yb = label_binarize(y, classes=list(range(NC)))
    cm = confusion_matrix(y, pred, labels=list(range(NC)))
    rows = []
    for c in range(NC):
        pos, neg = yb[:, c].sum() > 0, yb[:, c].sum() < len(yb)
        auc = roc_auc_score(yb[:, c], probs[:, c]) if pos and neg else float("nan")
        ap  = average_precision_score(yb[:, c], probs[:, c]) if pos else float("nan")
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + tp
        spec = tn / (tn + fp) if (tn + fp) else float("nan")
        rows.append({"cls": CFG.CLASS_NAMES[c], "precision": p[c], "recall": r[c],
                     "sensitivity": r[c], "specificity": spec,
                     "f1": f1[c], "support": int(sup[c]), "auc": auc, "ap": ap})
    return rows


def confidence_summary(probs, y):
    return {"mean_confidence": float(probs.max(1).mean()),
            "accuracy": float((probs.argmax(1) == y).mean()),
            "gap_conf_minus_acc": float(probs.max(1).mean() - (probs.argmax(1) == y).mean())}


def wilson_ci(k, n, z=1.96):
    if n <= 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def stratified_bootstrap_indices(y, rng):
    parts = []
    for cls in range(NC):
        idx = np.flatnonzero(y == cls)
        if len(idx):
            parts.append(rng.choice(idx, size=len(idx), replace=True))
    return np.concatenate(parts) if parts else np.array([], dtype=int)


def bootstrap_metric_cis(y, probs, reps=1000, seed=42):
    rng = np.random.default_rng(seed)
    samples = {k: [] for k in METRIC_KEYS}
    for _ in range(reps):
        idx = stratified_bootstrap_indices(y, rng)
        m = full_metrics(y[idx], probs[idx])
        for k in METRIC_KEYS:
            v = m.get(k, np.nan)
            if np.isfinite(v):
                samples[k].append(float(v))
    out = {}
    for k in METRIC_KEYS:
        v = np.asarray(samples[k])
        out[k] = (float(np.quantile(v, .025)), float(np.quantile(v, .975))) if len(v) else (float("nan"),) * 2
    return out


def cv_stats(values, n_tr, n_va):
    """Across-fold summary. 'nb' = Nadeau-Bengio corrected t-interval, which
    accounts for the overlap of training sets between folds (the naive interval
    is too narrow). Only meaningful for metrics measured on the held-out fold."""
    v = np.asarray(values, dtype=float)
    k = len(v)
    mean, sd = float(np.nanmean(v)), float(np.nanstd(v, ddof=1))
    t = float(sps.t.ppf(0.975, k - 1))
    se_naive = sd / math.sqrt(k)
    se_nb    = sd * math.sqrt(1.0 / k + n_va / n_tr)
    return {"mean": mean, "sd": sd, "min": float(np.nanmin(v)), "max": float(np.nanmax(v)),
            "ci95_naive_low": mean - t * se_naive, "ci95_naive_high": mean + t * se_naive,
            "ci95_nb_low":    mean - t * se_nb,    "ci95_nb_high":    mean + t * se_nb}


# -----------------------------------------------------------------------------
# 7. Train one fold
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, crit):
    model.eval()
    lg_list, y_list = [], []
    for x, y in loader:
        with torch.autocast("cuda", enabled=AMP_ON):
            lg = model(x.to(DEVICE, non_blocking=True))
        lg_list.append(lg.float().cpu()); y_list.append(y)
    logits, labels = torch.cat(lg_list), torch.cat(y_list)
    m = cond_metrics(labels.numpy(), F.softmax(logits, 1).numpy())
    m["loss"] = crit(logits.to(DEVICE), labels.to(DEVICE)).item()
    return m


def train_fold(fold, tr_idx, va_idx):
    set_seed(CFG.SEED + fold)
    ytr, yva = Y_POOL[tr_idx], Y_POOL[va_idx]
    # selection set: A = clean fold, B = fixed mixed (half clean / half corrupted) fold
    Xva = X_CLAHE_POOL[va_idx] if CFG.REGIME == "A" else build_mixed_val(fold, va_idx)

    net  = DRNet().to(DEVICE)
    crit = FocalLoss(class_weights(ytr), CFG.FOCAL_GAMMA, CFG.LABEL_SMOOTHING).to(DEVICE)
    print(f"[fold {fold}] regime {CFG.REGIME} | class weights {[round(float(w), 3) for w in crit.alpha.cpu()]}")

    opt = torch.optim.AdamW(
        [{"params": net.features.parameters(), "lr": CFG.BACKBONE_LR},
         {"params": net.head.parameters(),     "lr": CFG.HEAD_LR}],
        weight_decay=CFG.WEIGHT_DECAY)
    train_loader = DataLoader(DRTrainSet(tr_idx, CFG.P_CORRUPT), batch_size=CFG.BATCH_SIZE,
                              shuffle=True, drop_last=True, num_workers=CFG.NUM_WORKERS,
                              pin_memory=True, persistent_workers=CFG.NUM_WORKERS > 0)
    val_loader = eval_loader(ArrayDataset(Xva, yva))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[CFG.BACKBONE_LR, CFG.HEAD_LR], total_steps=CFG.EPOCHS * len(train_loader),
        pct_start=CFG.PCT_START, cycle_momentum=False)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ON)
    ema = EMA(net, CFG.EMA_DECAY)

    hist = {k: [] for k in ["train_loss", "train_acc_clean", "val_acc", "val_macro_f1", "val_qwk",
                            "val_nll", "val_ece", "select_score", "epoch_sec"]}
    best_score, best_ep, best_state, best_m = -1e9, 0, None, None
    ckpt = f"{CFG.FIG_DIR}/{CFG.TAG}_fold{fold}_best.pt"

    for ep in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        net.train()
        loss_sum, seen, c_ok, c_seen = 0.0, 0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            xm, ya, yb, lam = mixup_batch(x, y, CFG.MIX_PROB)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=AMP_ON):
                logits = net(xm)
            loss = crit(logits, ya) if lam == 1.0 else lam * crit(logits, ya) + (1 - lam) * crit(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step(); ema.update()
            bs = x.size(0)
            loss_sum += loss.item() * bs; seen += bs
            if lam == 1.0:
                c_ok += (logits.argmax(1) == ya).sum().item(); c_seen += bs

        vm = eval_epoch(ema.module, val_loader, crit)
        score = vm["qwk"] + vm["macro_f1"]
        for k, v in (("train_loss", loss_sum / max(seen, 1)),
                     ("train_acc_clean", 100 * c_ok / c_seen if c_seen else float("nan")),
                     ("val_acc", 100 * vm["acc"]), ("val_macro_f1", vm["macro_f1"]), ("val_qwk", vm["qwk"]),
                     ("val_nll", vm["nll"]), ("val_ece", vm["ece"]), ("select_score", score),
                     ("epoch_sec", time.time() - t0)):
            hist[k].append(v)
        msg = (f"[f{fold} ep {ep:02d}] tr_loss {hist['train_loss'][-1]:.4f} | "
               f"EMA val acc {100*vm['acc']:.2f} F1 {vm['macro_f1']:.4f} QWK {vm['qwk']:.4f} "
               f"NLL {vm['nll']:.4f} ECE {vm['ece']:.4f}")
        if ep >= CFG.SELECT_MIN_EPOCH and score > best_score:
            best_score, best_ep, best_m = score, ep, vm
            best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
            torch.save(best_state, ckpt)
            msg += "  <-- best"
        print(msg + f" | {hist['epoch_sec'][-1]:.0f}s")

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
        torch.save(best_state, ckpt)
        best_ep, best_score, best_m = CFG.EPOCHS, hist["select_score"][-1], vm
    ema.module.load_state_dict(best_state)
    best_m = {k: float(v) for k, v in best_m.items()}
    del train_loader, val_loader, net
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ema.module, hist, best_ep, float(best_score), best_m, ckpt


# -----------------------------------------------------------------------------
# 8. Run folds, evaluate every fold model on the 10 conditions
# -----------------------------------------------------------------------------
def run_folds():
    FOLD = {}
    folds_to_run = [0] if CFG.SMOKE else list(range(K))
    for fold in folds_to_run:
        tr_idx, va_idx = FOLDS[fold]
        state_fn = f"{CFG.FIG_DIR}/{CFG.TAG}_fold{fold}_state.pkl"
        st = None
        if CFG.RESUME and os.path.exists(state_fn):
            try:
                with open(state_fn, "rb") as fh:
                    st = pickle.load(fh)
                if (st.get("cfg_sig") != CFG_SIG or not os.path.exists(st["ckpt"])
                        or not np.array_equal(st["va_idx"], va_idx)):
                    st = None
                else:
                    print(f"[fold {fold}] resumed from {state_fn}")
            except Exception:
                st = None
        if st is None:
            print(f"\n{'=' * 22} REGIME {CFG.REGIME} | FOLD {fold + 1}/{K} | train {len(tr_idx)} | val {len(va_idx)} {'=' * 22}")
            t0 = time.time()
            net, hist, best_ep, best_score, best_m, ckpt = train_fold(fold, tr_idx, va_idx)
            train_sec = time.time() - t0
            oof  = predict_conditions(net, X_RAW_POOL[va_idx], X_CLAHE_POOL[va_idx], Y_POOL[va_idx], va_idx)
            test = predict_conditions(net, X_RAW_TEST, X_CLAHE_TEST, Y_TEST, TEST_GIDS)
            st = {"cfg_sig": CFG_SIG, "va_idx": va_idx, "hist": hist, "best_ep": best_ep,
                  "best_score": best_score, "best_m": best_m, "ckpt": ckpt, "train_sec": train_sec,
                  "oof_probs": oof, "test_probs": test}
            with open(state_fn, "wb") as fh:
                pickle.dump(st, fh)
            print(f"[fold {fold}] best epoch {best_ep} | select score {best_score:.4f} | "
                  f"sel-val acc {100*best_m['acc']:.2f}% | {train_sec/60:.1f} min")
            del net
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        FOLD[fold] = st
    return FOLD


def condition_table(oof_by_cond, y):
    rows = []
    for c in CONDITIONS:
        m = cond_metrics(y, oof_by_cond[c])
        rows.append({"condition": c, **m})
    return rows


# -----------------------------------------------------------------------------
# 8a. Grad-CAM + t-SNE on the CLEAN test set (single model: the fold with the
#     best SELECTION score; same as v3)
# -----------------------------------------------------------------------------
class GradCAM:
    def __init__(self, net):
        self.net = net.eval()
        self.acts = None
        self.grads = None
        self.handle = net.features[-1].register_forward_hook(self._fwd)

    def _fwd(self, m, i, o):
        if o.requires_grad:
            self.acts = o
            o.register_hook(lambda g: setattr(self, "grads", g))

    def remove(self):
        self.handle.remove()

    def __call__(self, x, cls=None):
        self.net.zero_grad(set_to_none=True)
        logits = self.net(x)
        cls = int(logits.argmax(1).item()) if cls is None else int(cls)
        logits[0, cls].backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls


def save_gradcam_figure(net, X, y, pred, tag):
    net = net.to(DEVICE).float().eval()
    cam_fn = GradCAM(net)
    picks = []
    for c in range(NC):
        idx = np.where((y == c) & (pred == c))[0]
        if len(idx):
            picks.append((idx[0], "correct"))
    wrong = np.where(y != pred)[0]
    picks += [(i, "WRONG") for i in wrong[:4]]
    if not picks:
        cam_fn.remove()
        return
    fig, ax = plt.subplots(2, len(picks), figsize=(3 * len(picks), 6), squeeze=False)
    for j, (i, kind) in enumerate(picks):
        x = ((torch.from_numpy(X[i]).permute(2, 0, 1).float() / 255
              - IMNET_MEAN) / IMNET_STD).unsqueeze(0).to(DEVICE)
        cam, cam_pred = cam_fn(x)
        heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8),
                                              cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay = (0.55 * X[i].astype(np.float32) + 0.45 * heat.astype(np.float32)
                   ).clip(0, 255).astype(np.uint8)
        ax[0, j].imshow(X[i])
        ax[0, j].set_title(f"True: {CFG.CLASS_NAMES[y[i]]}", fontsize=9)
        ax[0, j].axis("off")
        ax[1, j].imshow(overlay)
        ax[1, j].set_title(f"Pred: {CFG.CLASS_NAMES[cam_pred]} ({kind})", fontsize=9)
        ax[1, j].axis("off")
    plt.tight_layout()
    plt.savefig(f"{CFG.FIG_DIR}/{tag}.png", dpi=120)
    plt.close()
    cam_fn.remove()


@torch.no_grad()
def extract_embeddings(net, X):
    net = net.to(DEVICE).eval()
    feats = []
    for xb, _ in eval_loader(ArrayDataset(X, np.zeros(len(X), dtype=int))):
        with torch.autocast("cuda", enabled=AMP_ON):
            feats.append(net.embed(xb.to(DEVICE)).float().cpu())
    return torch.cat(feats).numpy()


def space_stats(Z, y, metric, suffix):
    stats_ = {f"silhouette_gt_{suffix}": float(silhouette_score(Z, y, metric=metric))}
    present = [c for c in range(NC) if (y == c).any()]
    cent = {c: Z[y == c].mean(0) for c in present}
    stats_[f"mean_inter_centroid_distance_{suffix}"] = float(np.mean([
        np.linalg.norm(cent[a] - cent[b]) for i, a in enumerate(present) for b in present[i + 1:]]))
    for c in present:
        stats_[f"intra_class_spread_{CFG.CLASS_NAMES[c]}_{suffix}"] = float(
            np.linalg.norm(Z[y == c] - cent[c], axis=1).mean())
    return stats_


def embedding_report(FOLD):
    """Grad-CAM figure, t-SNE figure and embedding statistics on the clean test set.
    A failure here is reported but never prevents the CSV from being written."""
    vis_fold = int(max(FOLD, key=lambda f: FOLD[f]["best_score"]))      # chosen on selection score, not test
    print(f"\nGrad-CAM / t-SNE use fold {vis_fold} (highest selection score), clean test set.")
    stats_ = {}
    try:
        net = DRNet(pretrained=False)
        net.load_state_dict(torch.load(FOLD[vis_fold]["ckpt"], map_location="cpu"))
        net = net.to(DEVICE).eval()
        pred = FOLD[vis_fold]["test_probs"]["clean"].argmax(1)
    except Exception as e:
        print(f"WARNING: could not load the visualisation model ({type(e).__name__}: {e}); "
              f"Grad-CAM / t-SNE skipped.")
        return stats_, vis_fold
    try:
        save_gradcam_figure(net, X_CLAHE_TEST, Y_TEST, pred, tag=f"{CFG.TAG}_gradcam_fold{vis_fold}")
    except Exception as e:
        print(f"WARNING: Grad-CAM skipped ({type(e).__name__}: {e})")
    try:
        emb = extract_embeddings(net, X_CLAHE_TEST)
        stats_ = space_stats(emb, Y_TEST, "cosine", "original_1280d")
        perplexity = float(min(30, max(2, (len(emb) - 1) // 3)))
        Z2 = TSNE(n_components=2, perplexity=perplexity, init="pca",
                  random_state=CFG.SEED).fit_transform(emb)
        stats_.update(space_stats(Z2, Y_TEST, "euclidean", "tsne_coords"))
        fig, ax = plt.subplots(1, 2, figsize=(14, 6))
        for c in range(NC):
            ax[0].scatter(*Z2[Y_TEST == c].T, s=12, label=CFG.CLASS_NAMES[c])
            ax[1].scatter(*Z2[pred == c].T, s=12, label=CFG.CLASS_NAMES[c])
        ax[0].set_title(f"t-SNE GT (fold {vis_fold} model, clean test)")
        ax[1].set_title("t-SNE pred")
        ax[0].legend(); ax[1].legend()
        plt.tight_layout()
        plt.savefig(f"{CFG.FIG_DIR}/{CFG.TAG}_tsne_fold{vis_fold}.png", dpi=120)
        plt.close()
        print("Grad-CAM and t-SNE figures saved.")
    except Exception as e:
        print(f"WARNING: t-SNE / embedding statistics skipped ({type(e).__name__}: {e})")
    return stats_, vis_fold


# -----------------------------------------------------------------------------
# 8b. Full report: every v3 metric, for every condition, into ONE CSV per regime
# -----------------------------------------------------------------------------
ROWS = []


def add(section, metric, value, split="", inference="", condition="", cls="", epoch="", note="", fold=""):
    ROWS.append({"section": section, "fold": fold, "split": split, "inference": inference,
                 "condition": condition, "class": cls, "epoch": epoch, "metric": metric,
                 "value": value, "note": note})


def fci(ci):
    return f"[{ci[0]:.4f}, {ci[1]:.4f}]"


def full_report(FOLD, oof, test_folds, run_sec):
    ROWS.clear()
    BLOCKS, BLK = [], {}

    def make_block(fold, split, cond, y, raw, cal, note=""):
        b = {"fold": fold, "split": split, "cond": cond, "y": y, "raw": raw, "cal": cal, "note": note,
             "m_raw": full_metrics(y, raw), "m_cal": full_metrics(y, cal)}
        BLOCKS.append(b)
        BLK[(fold, split, cond)] = b
        return b

    # ---- per-fold blocks (held-out fold = "val", fixed test set = "test") ----
    T_FOLD = {f: {} for f in range(K)}
    for f in range(K):
        st = FOLD[f]
        yva = Y_POOL[st["va_idx"]]
        for c in CONDITIONS:
            Tk = fit_temperature(st["oof_probs"][c], yva)          # fit on this fold's held-out data, this condition
            T_FOLD[f][c] = Tk
            make_block(f, "val", c, yva, st["oof_probs"][c],
                       apply_temperature(st["oof_probs"][c], Tk),
                       note="T fit on this held-out fold under this condition (in-sample, optimistic)")
            make_block(f, "test", c, Y_TEST, st["test_probs"][c],
                       apply_temperature(st["test_probs"][c], Tk),
                       note="T from this fold's held-out data (same condition); test never used for fitting")

    # ---- pooled out-of-fold blocks with cross-fitted temperature ----
    T_CROSS = {f: {} for f in range(K)}
    for c in CONDITIONS:
        oof_cal = np.zeros_like(oof[c])
        for f in range(K):
            other = FOLD_OF != f
            Tx = fit_temperature(oof[c][other], Y_POOL[other])      # fit on the OTHER folds
            T_CROSS[f][c] = Tx
            oof_cal[~other] = apply_temperature(oof[c][~other], Tx)
        make_block("all", "oof", c, Y_POOL, oof[c], oof_cal,
                   note="calibrated = cross-fitted T (honest)")

    # ---- 5-model ensemble on the fixed test set ----
    for c in CONDITIONS:
        ens_raw = np.mean(test_folds[c], axis=0)
        ens_cal = np.mean([BLK[(f, "test", c)]["cal"] for f in range(K)], axis=0)
        make_block("ensemble", "test", c, Y_TEST, ens_raw, ens_cal,
                   note="raw = mean of fold probs; calibrated = mean of per-fold calibrated probs")

    # ---- bootstrap CIs (OOF + ensemble), parallel over blocks ----
    print(f"\nBootstrapping ({CFG.BOOTSTRAP_REPS} reps x {len(CONDITIONS)} conditions x OOF/ensemble x raw/cal)...")
    jobs = [(b, v) for b in BLOCKS if b["fold"] in ("all", "ensemble") for v in ("raw", "cal")]
    res = Parallel(n_jobs=CFG.N_JOBS)(
        delayed(bootstrap_metric_cis)(b["y"], b[v], CFG.BOOTSTRAP_REPS, CFG.SEED)
        for b, v in tqdm(jobs, desc="bootstrap"))
    for (b, v), r in zip(jobs, res):
        b["ci_" + v] = r

    # ---- across-fold statistics ----
    n_tr = float(np.mean([len(FOLDS[f][0]) for f in range(K)]))
    n_va = float(np.mean([len(FOLDS[f][1]) for f in range(K)]))
    CV = {}
    for split in ("val", "test"):
        for c in CONDITIONS:
            for variant in ("raw", "cal"):
                for k in METRIC_KEYS:
                    vals = [BLK[(f, split, c)]["m_" + variant][k] for f in range(K)]
                    CV[(split, c, variant, k)] = cv_stats(vals, n_tr, n_va)

    # ------------------------------ console ------------------------------------
    print("\n" + "=" * 118)
    print(f"REGIME {CFG.REGIME}: POOLED OUT-OF-FOLD with stratified-bootstrap 95% CI "
          f"(n={N_POOL}, {CFG.BOOTSTRAP_REPS} reps). cal = cross-fitted temperature scaling")
    for c in CONDITIONS:
        b = BLK[("all", "oof", c)]
        m, ci, mc = b["m_raw"], b["ci_raw"], b["m_cal"]
        print(f"{c:11s} | acc {m['acc']:.4f} {fci(ci['acc'])} | QWK {m['qwk']:.4f} {fci(ci['qwk'])} | "
              f"F1 {m['macro_f1']:.4f} {fci(ci['macro_f1'])}")
        print(f"{'':11s} | ref-sens {m['ref_sens']:.4f} {fci(ci['ref_sens'])} ref-spec {m['ref_spec']:.4f} "
              f"{fci(ci['ref_spec'])} ref-AUC {m['ref_auc']:.4f} | macro-AUC {m['macro_auc']:.4f} | "
              f"ECE raw {m['ece']:.4f} -> cal {mc['ece']:.4f} | NLL raw {m['nll']:.4f} -> cal {mc['nll']:.4f}")

    print("\n" + "=" * 118)
    print(f"REGIME {CFG.REGIME}: FIXED TEST SET, 5-model ENSEMBLE with stratified-bootstrap 95% CI")
    for c in CONDITIONS:
        b = BLK[("ensemble", "test", c)]
        m, ci, mc = b["m_raw"], b["ci_raw"], b["m_cal"]
        print(f"{c:11s} | acc {m['acc']:.4f} {fci(ci['acc'])} | QWK {m['qwk']:.4f} {fci(ci['qwk'])} | "
              f"F1 {m['macro_f1']:.4f} {fci(ci['macro_f1'])} | ref-sens {m['ref_sens']:.4f} "
              f"ref-spec {m['ref_spec']:.4f} | ECE raw {m['ece']:.4f} -> cal {mc['ece']:.4f}")

    print("\n" + "=" * 118)
    print(f"REGIME {CFG.REGIME}: ACROSS-FOLD SUMMARY (mean ± SD). Held-out CI = Nadeau-Bengio corrected 95% t-interval.")
    for c in CONDITIONS:
        sq, sa, tq = CV[("val", c, "raw", "qwk")], CV[("val", c, "raw", "acc")], CV[("test", c, "raw", "qwk")]
        print(f"{c:11s} | held-out QWK {sq['mean']:.4f}±{sq['sd']:.4f} [{sq['ci95_nb_low']:.4f}, {sq['ci95_nb_high']:.4f}] | "
              f"held-out acc {sa['mean']:.4f}±{sa['sd']:.4f} | test (single models) QWK {tq['mean']:.4f}±{tq['sd']:.4f}")

    # ------------------------------ CSV rows -----------------------------------
    emb_stats, vis_fold = embedding_report(FOLD)

    cfg_vars = {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}
    switch_keys = {"REGIME", "GRID_ONLY", "SMOKE"}
    corr_keys = {"BASE_RES", "CORR_TYPES", "DEFOCUS_SIGMA", "HAZE_T", "HAZE_A", "NOISE_STD",
                 "P_CORRUPT", "EVAL_SEED", "VALMIX_SEED"}
    path_keys = {"ROOT", "TRAIN_DIR", "VAL_DIR", "TEST_DIR", "TRAIN_CSV", "VAL_CSV", "TEST_CSV",
                 "OUT_DIR", "CACHE_DIR", "FIG_DIR", "TAG"}
    for k, v in cfg_vars.items():
        note = ("run switch" if k in switch_keys else
                "v4 corruption model (locked after Figure 1)" if k in corr_keys else
                "path / identity" if k in path_keys else
                "analysis only (not part of the config signature)" if k in ANALYSIS_ONLY_KEYS else
                "v4 data pipeline: crop -> pad -> resize -> [corruption] -> CLAHE" if k in
                {"IMG_SIZE", "CROP_TOL", "CROP_MIN_FRAC", "CLAHE_CLIP", "CLAHE_TILE"} else
                "v3 recipe (unchanged)")
        add("config", k, v, note=note)
    add("config", "CLASS_WEIGHT_IMPLEMENTATION", "inv_freq^WEIGHT_POWER_mean1",
        note="weights = 1/count^0.5 rescaled to class-mean 1, recomputed per fold from its training labels")
    add("config", "PREPROCESS_ORDER", "crop -> pad_to_square -> resize -> [corruption] -> CLAHE(LAB-L) -> augmentation -> normalise")
    add("config", "CONDITIONS", ";".join(CONDITIONS))
    add("config", "FOLD_SEED_RULE", "SEED + fold_index", note="master seed 42; stratified split random_state=SPLIT_SEED")
    add("config", "CV_DESIGN", "pooled train+val, stratified 5-fold; test untouched",
        note="test never used for epoch selection or temperature fitting")
    add("config", "SELECTION_SET",
        "clean held-out fold" if CFG.REGIME == "A" else "fixed mixed held-out fold (half clean, half corrupted)",
        note="epoch selection by QWK + macro-F1 on the EMA model")
    add("config", "INFERENCE", INF, note="no TTA in v4; column kept for layout compatibility with v3")

    add("run_info", "torch_version", torch.__version__)
    add("run_info", "python_version", platform.python_version())
    add("run_info", "gpus_visible", "; ".join(GPU_NAMES) if GPU_NAMES else "cpu")
    add("run_info", "config_signature", CFG_SIG)
    add("run_info", "regime", CFG.REGIME)
    add("run_info", "pool_size", N_POOL)
    add("run_info", "test_size", int(len(Y_TEST)))
    add("run_info", "total_run_seconds_this_session", round(run_sec, 1),
        note="folds resumed from cache contribute ~0 s")
    add("run_info", "vis_fold_gradcam_tsne", vis_fold, note="fold with best selection score; clean test set")
    for f in range(K):
        st = FOLD[f]
        add("run_info", "best_epoch", st["best_ep"], fold=f)
        add("run_info", "best_select_score_qwk_plus_macro_f1", st["best_score"], fold=f)
        add("run_info", "val_acc_at_best_epoch_ema", st["best_m"]["acc"], fold=f)
        add("run_info", "val_qwk_at_best_epoch_ema", st["best_m"]["qwk"], fold=f)
        add("run_info", "val_macro_f1_at_best_epoch_ema", st["best_m"]["macro_f1"], fold=f)
        add("run_info", "training_seconds", round(st["train_sec"], 1), fold=f)
        add("run_info", "n_train", int(len(FOLDS[f][0])), fold=f)
        add("run_info", "n_val", int(len(FOLDS[f][1])), fold=f)
        for k, v in st["best_m"].items():
            add("selection_val_at_best", k, v, fold=f, split="selection_val",
                note="selection set at the best epoch (EMA model): clean fold in Regime A, mixed fold in Regime B")

    for i in range(N_POOL):
        add("fold_assignment", str(ID_POOL[i]), int(FOLD_OF[i]), cls=CFG.CLASS_NAMES[Y_POOL[i]])

    for f in range(K):
        for i, ep in enumerate(range(1, CFG.EPOCHS + 1)):
            for k in FOLD[f]["hist"]:
                add("history", k, FOLD[f]["hist"][k][i], epoch=ep, fold=f,
                    note="selection set (clean fold in A, mixed fold in B)" if k.startswith("val_") else "")

    def report_block(b):
        fold, split, cond, y = b["fold"], b["split"], b["cond"], b["y"]
        kw = dict(fold=fold, split=split, inference=INF, condition=cond)
        for variant in ("raw", "cal"):
            probs, m = b[variant], b["m_" + variant]
            sfx = "" if variant == "raw" else "_calibrated"
            note = b["note"] if variant == "cal" else ""
            for k in METRIC_KEYS:
                add("overall" + sfx, k, m[k], note=note, **kw)
            lo, hi = wilson_ci(int((probs.argmax(1) == y).sum()), len(y))
            add("overall" + sfx, "acc_wilson_ci95_low",  lo, **kw)
            add("overall" + sfx, "acc_wilson_ci95_high", hi, **kw)
            for row in per_class_metrics(y, probs):
                for k in ("precision", "recall", "sensitivity", "specificity", "f1", "support", "auc", "ap"):
                    add("per_class" + sfx, k, row[k], cls=row["cls"], **kw)
            cm = confusion_matrix(y, probs.argmax(1), labels=range(NC))
            for i, tname in enumerate(CFG.CLASS_NAMES):
                for j, pname in enumerate(CFG.CLASS_NAMES):
                    add("confusion" + sfx, f"pred_{pname}", int(cm[i, j]), cls=tname, **kw)
            _, bins = ece_with_bins(probs, y)
            for lo_, hi_, cnt, mc, ac, gp in bins:
                add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_count",     cnt, **kw)
                add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_mean_conf", mc,  **kw)
                add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_accuracy",  ac,  **kw)
                add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_gap",       gp,  **kw)
            for k, v in confidence_summary(probs, y).items():
                add("calibration_summary" + sfx, k, v, **kw)
            ci = b.get("ci_" + variant)
            if ci:
                for metric, (lo_, hi_) in ci.items():
                    add("bootstrap_ci" + sfx, f"{metric}_ci95_low",  lo_, **kw)
                    add("bootstrap_ci" + sfx, f"{metric}_ci95_high", hi_, **kw)

    for b in BLOCKS:
        report_block(b)

    for f in range(K):
        for c in CONDITIONS:
            add("temperature_scaling", "temperature_fold_val_fit", T_FOLD[f][c], fold=f, inference=INF, condition=c,
                note="fit on this fold's held-out data under this condition; applied to this fold's val and test")
            add("temperature_scaling", "temperature_cross_fit_oof", T_CROSS[f][c], fold=f, inference=INF, condition=c,
                note="fit on the other folds' OOF (same condition); used only for honest OOF calibration")

    for (split, c, variant, k), s in CV.items():
        sec = "cv_summary" if variant == "raw" else "cv_summary_calibrated"
        for stat, val in s.items():
            if split == "test" and stat.startswith("ci95"):
                continue                                   # test CI across folds ignores test-sampling noise
            add(sec, f"{k}_{stat}", val, fold="cv", split=split, inference=INF, condition=c,
                note=("Nadeau-Bengio corrected" if stat.startswith("ci95_nb") else
                      "naive t-interval (too narrow, folds overlap)" if stat.startswith("ci95_naive") else
                      "variation across fold-trained models on the fixed test set" if split == "test" else ""))

    for k, v in emb_stats.items():
        add("embedding", k, v, split="test", inference=INF, condition="clean", fold=vis_fold)

    # ---- references (constants copied from v3; clean-condition benchmarks) ----
    NO, TT = "noTTA", "TTA"
    BASELINE_V1 = {
        ("val",  NO): {"acc": 0.8607, "macro_f1": 0.7324, "weighted_f1": 0.8589, "qwk": 0.9179, "ece": 0.0654},
        ("val",  TT): {"acc": 0.8607, "macro_f1": 0.7573, "weighted_f1": 0.8607, "qwk": 0.9204, "ece": 0.0666},
        ("test", NO): {"acc": 0.8251, "macro_f1": 0.6389, "weighted_f1": 0.8261, "qwk": 0.9030, "ece": 0.0351},
        ("test", TT): {"acc": 0.8197, "macro_f1": 0.6285, "weighted_f1": 0.8199, "qwk": 0.9036, "ece": 0.0484},
    }
    BASELINE_V2 = {
        ("val",  NO): {"acc": 0.8552, "macro_f1": 0.7325, "weighted_f1": 0.8518, "qwk": 0.9158, "ece": 0.0790},
        ("val",  TT): {"acc": 0.8388, "macro_f1": 0.7064, "weighted_f1": 0.8350, "qwk": 0.9041, "ece": 0.0745},
        ("test", NO): {"acc": 0.8525, "macro_f1": 0.6863, "weighted_f1": 0.8476, "qwk": 0.9062, "ece": 0.0822},
        ("test", TT): {"acc": 0.8579, "macro_f1": 0.6969, "weighted_f1": 0.8543, "qwk": 0.9104, "ece": 0.0587},
    }
    for sec, ref, nt in (("baseline_v1_reference", BASELINE_V1, "v1 pipeline, 224px, single split, seed 42"),
                         ("baseline_v2_reference", BASELINE_V2, "v2 pipeline (steps 1-5), 320px, single split, seed 42")):
        for (split, inf), r in ref.items():
            for k, v in r.items():
                add(sec, k, v, split=split, inference=inf, condition="clean", note=nt)

    PAPER_OVERALL = {
        ("val",  NO): {"acc": 0.8407, "macro_f1": 0.6962},
        ("test", NO): {"acc": 0.8361, "macro_f1": 0.6434, "weighted_f1": 0.8262,
                       "macro_precision": 0.7122, "macro_recall": 0.6217,
                       "macro_auc": 0.8949, "macro_ap": 0.6836, "ece": 0.0327},
        ("test", TT): {"acc": 0.8497, "macro_f1": 0.6627, "weighted_f1": 0.8400,
                       "macro_precision": 0.7354, "macro_recall": 0.6375,
                       "macro_auc": 0.9099, "macro_ap": 0.6767, "ece": 0.0239},
    }
    for (split, inf), ref in PAPER_OVERALL.items():
        for k, v in ref.items():
            add("paper_reference", k, v, split=split, inference=inf, condition="clean", note="paper Tables 3/4/7")

    PAPER_CLASS = {
        NO: {"No_DR": (0.9751, 0.9849, 0.9800), "Mild": (0.6154, 0.5333, 0.5714),
             "Moderate": (0.6847, 0.8736, 0.7677), "Severe": (0.4286, 0.3529, 0.3871),
             "Proliferative": (0.8571, 0.3636, 0.5106)},
        TT: {"No_DR": (0.9799, 0.9799, 0.9799), "Mild": (0.7500, 0.6000, 0.6667),
             "Moderate": (0.6957, 0.9195, 0.7921), "Severe": (0.3846, 0.2941, 0.3333),
             "Proliferative": (0.8667, 0.3939, 0.5417)},
    }
    for inf, table in PAPER_CLASS.items():
        for cname, (p, r, f1) in table.items():
            add("paper_reference_per_class", "precision", p,  split="test", inference=inf, condition="clean", cls=cname)
            add("paper_reference_per_class", "recall",    r,  split="test", inference=inf, condition="clean", cls=cname)
            add("paper_reference_per_class", "f1",        f1, split="test", inference=inf, condition="clean", cls=cname)

    csv_path = os.path.join(CFG.OUT_DIR, f"{CFG.TAG}_results.csv")
    pd.DataFrame(ROWS, columns=["section", "fold", "split", "inference", "condition", "class",
                                "epoch", "metric", "value", "note"]).to_csv(csv_path, index=False)
    return csv_path, len(ROWS)


# -----------------------------------------------------------------------------
# 8c. Finish: smoke summary, probability pickle, console table, full CSV
# -----------------------------------------------------------------------------
def finish(FOLD, run_sec=0.0):
    if CFG.SMOKE:
        va = FOLDS[0][1]
        print("\nSMOKE TEST (fold 0 only, held-out fold, 3 epochs) - numbers are NOT results:")
        print(f"{'condition':11s} | acc    QWK    ref-sens")
        for r in condition_table(FOLD[0]["oof_probs"], Y_POOL[va]):
            print(f"{r['condition']:11s} | {r['acc']:.3f}  {r['qwk']:.3f}  {r['ref_sens']:.3f}")
        print("Smoke OK if: no errors, table prints, and (after 3 epochs) corrupted rows are not identical to 'clean'.")
        return

    oof = {c: np.zeros((N_POOL, NC), np.float32) for c in CONDITIONS}
    for f in range(K):
        for c in CONDITIONS:
            oof[c][FOLD[f]["va_idx"]] = FOLD[f]["oof_probs"][c]
    test_folds = {c: [FOLD[f]["test_probs"][c] for f in range(K)] for c in CONDITIONS}

    RES = {"regime": CFG.REGIME, "conditions": CONDITIONS, "n_folds": K, "cfg_sig": CFG_SIG,
           "Y_POOL": Y_POOL, "Y_TEST": Y_TEST, "FOLD_OF": FOLD_OF, "ID_POOL": ID_POOL,
           "oof": oof, "test_folds": test_folds,
           "fold_info": [{"best_ep": FOLD[f]["best_ep"], "best_score": FOLD[f]["best_score"],
                          "best_m": FOLD[f]["best_m"], "train_sec": FOLD[f]["train_sec"]} for f in range(K)],
           "hist": [FOLD[f]["hist"] for f in range(K)],
           "corruption": {k: getattr(CFG, k) for k in ("CORR_TYPES", "DEFOCUS_SIGMA", "HAZE_T", "HAZE_A",
                                                       "NOISE_STD", "P_CORRUPT", "EVAL_SEED", "VALMIX_SEED",
                                                       "IMG_SIZE", "BASE_RES")}}
    pkl_path = os.path.join(CFG.OUT_DIR, f"{CFG.TAG}_probs.pkl")
    with open(pkl_path, "wb") as fh:
        pickle.dump(RES, fh)
    print(f"\nSaved: {pkl_path}")

    print(f"\n{'=' * 110}\nREGIME {CFG.REGIME}: pooled out-of-fold results per condition (n={N_POOL}) and fixed test set (5-model ensemble)")
    print(f"{'condition':11s} | OOF acc  QWK    F1     ref-sens ref-spec ECE   | fold QWK mean±SD | test-ens acc  QWK")
    for c in CONDITIONS:
        m = cond_metrics(Y_POOL, oof[c])
        fq = [cond_metrics(Y_POOL[FOLD_OF == f], oof[c][FOLD_OF == f])["qwk"] for f in range(K)]
        ens = np.mean(test_folds[c], axis=0)
        mt = cond_metrics(Y_TEST, ens)
        print(f"{c:11s} | {m['acc']:.4f}  {m['qwk']:.4f} {m['macro_f1']:.4f} {m['ref_sens']:.4f}   {m['ref_spec']:.4f}   "
              f"{m['ece']:.4f} | {np.mean(fq):.4f}±{np.std(fq, ddof=1):.4f} | {mt['acc']:.4f}   {mt['qwk']:.4f}")

    csv_path, n_rows = full_report(FOLD, oof, test_folds, run_sec)
    print(f"\nSaved single CSV: {csv_path}")
    print(f"Rows: {n_rows}")
    print("NEXT:", "run again with REGIME='B'" if CFG.REGIME == "A" else
          "run dr_robustness_analysis.py (needs BOTH v4_robust_A_probs.pkl and v4_robust_B_probs.pkl)")
    print("Done. Send me the CSV + console log.")


if CFG.GRID_ONLY:
    print("\nGRID_ONLY: caches built and Figure 1 saved. Inspect the figures, lock the corruption values in CFG, then set GRID_ONLY=False.")
    if not SANITY_OK:
        print("WARNING: some corruption is not monotonic in severity - fix before continuing.")
else:
    _t_run = time.time()
    FOLD_RESULTS = run_folds()
    finish(FOLD_RESULTS, time.time() - _t_run)

REGIME A | p_corrupt 0.0 | smoke False | grid_only False | device cuda | torch 2.11.0+cu128 | cfg 39d1cff543c8
Split integrity OK.


cache train:   0%|          | 0/2930 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/train_v4_raw_crop_pad_resize_s320_fbddaee74c44.npy


cache val:   0%|          | 0/366 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/val_v4_raw_crop_pad_resize_s320_b9bb8db97582.npy


...

cache saved: /tmp/dr_cache/test_v4_raw_crop_pad_resize_s320_b6e0148962cc.npy
pool n=3296 counts [1606, 340, 912, 176, 262] | test n=366 counts [199, 30, 87, 17, 33]

Corruption sanity (mean |corrupted - clean| on 0-255 scale, before CLAHE; must grow with severity):
  defocus  s1   1.69 | s2   2.79 | s3   3.65 | monotonic True
  haze     s1  29.49 | s2  58.96 | s3  88.26 | monotonic True
  noise    s1   5.27 | s2  10.31 | s3  16.78 | monotonic True
Figure 1 saved to /kaggle/working/figures_v4

====================== REGIME A | FOLD 1/5 | train 2636 | val 660 ======================
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 183MB/s]


[fold 0] regime A | class weights [0.5, 1.086, 0.664, 1.514, 1.236]
[f0 ep 01] tr_loss 0.7226 | EMA val acc 74.85 F1 0.4384 QWK 0.7869 NLL 0.7462 ECE 0.1431 | 94s
[f0 ep 02] tr_loss 0.5090 | EMA val acc 76.82 F1 0.5551 QWK 0.8182 NLL 0.5601 ECE 0.0410 | 21s
[f0 ep 03] tr_loss 0.4386 | EMA val acc 80.30 F1 0.6392 QWK 0.8807 NLL 0.5301 ECE 0.0519 | 21s
[f0 ep 04] tr_loss 0.4293 | EMA val acc 83.33 F1 0.6853 QWK 0.9009 NLL 0.4954 ECE 0.0564 | 22s
[f0 ep 05] tr_loss 0.3601 | EMA val acc 81.36 F1 0.6465 QWK 0.8955 NLL 0.4726 ECE 0.0175 | 21s
[f0 ep 06] tr_loss 0.3194 | EMA val acc 82.58 F1 0.6643 QWK 0.9005 NLL 0.4592 ECE 0.0200 | 22s
[f0 ep 07] tr_loss 0.3174 | EMA val acc 82.88 F1 0.6695 QWK 0.8995 NLL 0.4577 ECE 0.0152 | 22s
[f0 ep 08] tr_loss 0.2667 | EMA val acc 83.48 F1 0.6881 QWK 0.9080 NLL 0.4540 ECE 0.0248  <-- best | 22s
[f0 ep 09] tr_loss 0.2565 | EMA val acc 83.18 F1 0.6837 QWK 0.9043 NLL 0.4573 ECE 0.0333 | 22s
[f0 ep 10] tr_loss 0.2328 | EMA val acc 83.64 F1 0.6937 QWK 0.9082 

bootstrap:   0%|          | 0/40 [00:00<?, ?it/s]


REGIME A: POOLED OUT-OF-FOLD with stratified-bootstrap 95% CI (n=3296, 1000 reps). cal = cross-fitted temperature scaling
clean       | acc 0.8410 [0.8298, 0.8513] | QWK 0.9052 [0.8956, 0.9149] | F1 0.7080 [0.6867, 0.7286]
            | ref-sens 0.9148 [0.9000, 0.9304] ref-spec 0.9440 [0.9353, 0.9527] ref-AUC 0.9804 | macro-AUC 0.9446 | ECE raw 0.0410 -> cal 0.0150 | NLL raw 0.4690 -> cal 0.4613
defocus_s1  | acc 0.8022 [0.7910, 0.8122] | QWK 0.8710 [0.8573, 0.8838] | F1 0.6171 [0.5934, 0.6407]
            | ref-sens 0.9696 [0.9593, 0.9785] ref-spec 0.8592 [0.8484, 0.8705] ref-AUC 0.9679 | macro-AUC 0.9041 | ECE raw 0.0218 -> cal 0.0238 | NLL raw 0.6089 -> cal 0.6099
defocus_s2  | acc 0.6990 [0.6872, 0.7112] | QWK 0.7251 [0.7021, 0.7471] | F1 0.4723 [0.4516, 0.4934]
            | ref-sens 0.8333 [0.8141, 0.8541] ref-spec 0.8941 [0.8828, 0.9054] ref-AUC 0.9438 | macro-AUC 0.8726 | ECE raw 0.0320 -> cal 0.0291 | NLL raw 0.8092 -> cal 0.8101
defocus_s3  | acc 0.5273 [0.5182, 0.5373] | QW

In [1]:
# =============================================================================
# APTOS 2019 DR grading | EfficientNet-B1 | Pipeline v3 = v2 (steps 1-6) + 5-fold CV
#
#   *** NO-CLAHE ABLATION COPY ***  Identical to the v3 pipeline except that CLAHE is
#   switched off (USE_CLAHE = False). Preprocessing is therefore: crop -> pad -> resize.
#   TAG, PREP_VERSION and the PREPROCESS_ORDER label were renamed so that caches, fold
#   states, checkpoints, figures and the results CSV never overwrite the CLAHE run.
#
#   Training recipe, preprocessing, loss, EMA, selection and temperature scaling
#   are IDENTICAL to v2. What is new:
#
#   DESIGN
#   - train.csv + valid.csv are POOLED (3294 images) and split into 5 stratified
#     folds. Each fold model trains on 4/5 and uses the held-out 1/5 for exactly
#     what v2 used "val" for (epoch selection by QWK+macro-F1 on the EMA model,
#     and fitting the temperature).
#   - The original test set (366 images) is NEVER used for selection or
#     calibration. Each fold model predicts it once; results are reported per
#     fold (mean/SD) and as a 5-model ensemble (reported separately).
#   - Master seed 42; fold k uses seed 42+k. No multi-seed repeats.
#
#   STATISTICS
#   - Out-of-fold (OOF) predictions over all 3294 images -> pooled metrics with
#     stratified-bootstrap 95% CIs (1000 reps).
#   - Cross-fitted temperature scaling for OOF calibration (T for fold k is fit
#     on the OTHER folds' OOF predictions) -> honest OOF ECE/NLL/Brier.
#   - Per-fold mean, SD, min/max; naive and Nadeau-Bengio corrected t-intervals
#     across folds (val side).
#   - Test: per-fold models (mean/SD) + ensemble with bootstrap CIs.
#   - Paired bootstrap TTA - noTTA on OOF predictions (is TTA worth it?).
#   - Fold results are cached to disk so an interrupted run can resume.
#
#   Everything is written to ONE CSV (column "fold" added).
# =============================================================================

import os, time, json, random, math, warnings, platform, gc, hashlib, copy, pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF

from scipy import stats as sps
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, average_precision_score,
    cohen_kappa_score, silhouette_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")


# -----------------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------------
class CFG:
    # ---------- paths ----------
    ROOT      = "/kaggle/input/datasets/mariaherrerot/aptos2019"
    TRAIN_DIR = f"{ROOT}/train_images/train_images"
    VAL_DIR   = f"{ROOT}/val_images/val_images"
    TEST_DIR  = f"{ROOT}/test_images/test_images"
    TRAIN_CSV = f"{ROOT}/train_1.csv"
    VAL_CSV   = f"{ROOT}/valid.csv"
    TEST_CSV  = f"{ROOT}/test.csv"
    OUT_DIR   = "/kaggle/working"
    CACHE_DIR = "/tmp/dr_cache"
    FIG_DIR   = f"{OUT_DIR}/figures_v3"

    # ---------- experiment identity ----------
    TAG  = "v3_cv5_seed42_noCLAHE"
    SEED = 42

    # ---------- v3: cross-validation ----------
    N_FOLDS = 5
    RESUME  = True                         # reuse finished folds if config unchanged

    # ---------- data ----------
    IMG_SIZE    = 320
    NUM_CLASSES = 5
    CLASS_NAMES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative"]

    # ---------- paper-specified (unchanged) ----------
    EPOCHS          = 30
    BATCH_SIZE      = 16
    WEIGHT_DECAY    = 1e-4
    LABEL_SMOOTHING = 0.05

    # ---------- preprocessing ----------
    USE_CROP       = True
    CROP_TOL       = 7
    CROP_MIN_FRAC  = 0.05
    USE_CLAHE      = False
    CLAHE_CLIP     = 2.0
    CLAHE_TILE     = 8

    # ---------- optimisation ----------
    BACKBONE_LR = 2e-4
    HEAD_LR     = 1e-3
    PCT_START   = 0.1
    GRAD_CLIP   = 1.0
    EMA_DECAY   = 0.998

    # ---------- regularisation / imbalance ----------
    FOCAL_GAMMA  = 1.0
    WEIGHT_POWER = 0.5
    MIXUP_ALPHA  = 0.4
    MIX_PROB     = 0.25
    DROPOUT      = 0.2
    HIDDEN       = 512

    # ---------- model selection ----------
    SELECT_MIN_EPOCH = 8

    # ---------- other assumptions ----------
    TTA_ANGLE      = 10
    AUG_DEGREES    = 20
    AUG_TRANSLATE  = 0.05
    AUG_SCALE      = (0.9, 1.1)
    AUG_BRIGHTNESS = 0.15
    AUG_CONTRAST   = 0.15

    USE_AMP     = True
    NUM_WORKERS = 4
    N_JOBS      = 4

    INFER_NO_TTA = "noTTA"
    INFER_TTA    = "TTA"

    BOOTSTRAP_REPS = 1000


os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.CACHE_DIR, exist_ok=True)
os.makedirs(CFG.FIG_DIR, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
AMP_ON    = CFG.USE_AMP and DEVICE.type == "cuda"
MODES     = (CFG.INFER_NO_TTA, CFG.INFER_TTA)
K         = CFG.N_FOLDS


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def cfg_signature():
    d = {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]


CFG_SIG = cfg_signature()
set_seed(CFG.SEED)
print(f"Device {DEVICE} | GPUs {GPU_NAMES} | torch {torch.__version__} | py {platform.python_version()} | cfg {CFG_SIG}")


# -----------------------------------------------------------------------------
# 2. Data loading, split integrity, preprocessing cache
# -----------------------------------------------------------------------------
def load_df(csv_path):
    df = pd.read_csv(csv_path)
    id_col  = next((c for c in df.columns if c.lower() in
                    ("id_code", "image", "id", "image_id", "filename")), df.columns[0])
    lab_col = next((c for c in df.columns if c.lower() in
                    ("diagnosis", "label", "level", "dr_grade")), df.columns[1])
    df = df[[id_col, lab_col]].rename(columns={id_col: "id", lab_col: "label"})
    df["id"]    = df["id"].astype(str)
    df["label"] = df["label"].astype(int)
    assert df["label"].isin(range(CFG.NUM_CLASSES)).all(), f"bad labels in {csv_path}"
    return df


TRAIN_DF = load_df(CFG.TRAIN_CSV)
VAL_DF   = load_df(CFG.VAL_CSV)
TEST_DF  = load_df(CFG.TEST_CSV)

for name, df in (("train", TRAIN_DF), ("val", VAL_DF), ("test", TEST_DF)):
    assert df["id"].is_unique, f"duplicate IDs in {name}"
tr, va, te = map(set, (TRAIN_DF["id"], VAL_DF["id"], TEST_DF["id"]))
assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), "split overlap"
print("Split integrity OK.")

PAPER_COUNTS = {
    "train": [1434, 300, 808, 154, 234],
    "val":   [171, 40, 103, 22, 28],
    "test":  [199, 30, 87, 17, 33],
}
for name, df in (("train", TRAIN_DF), ("val", VAL_DF), ("test", TEST_DF)):
    got = np.bincount(df["label"].values, minlength=CFG.NUM_CLASSES).tolist()
    print(f"{name:5s} n={len(df):5d} | counts {got} | paper {PAPER_COUNTS[name]} | "
          f"{'MATCH' if got == PAPER_COUNTS[name] else 'DIFFERS'}")


def find_image(img_dir, img_id):
    for ext in ("", ".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        p = os.path.join(img_dir, f"{img_id}{ext}")
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"{img_id!r} not found in {img_dir}")


def crop_retina(rgb, tol=CFG.CROP_TOL, min_frac=CFG.CROP_MIN_FRAC):
    mask = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) > tol
    if mask.mean() < min_frac:
        return rgb
    h, w = mask.shape
    rows = np.flatnonzero(mask.sum(1) > max(2, 0.005 * w))
    cols = np.flatnonzero(mask.sum(0) > max(2, 0.005 * h))
    if len(rows) == 0 or len(cols) == 0:
        return rgb
    return rgb[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def pad_to_square(rgb):
    h, w = rgb.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    return cv2.copyMakeBorder(rgb, top, s - h - top, left, s - w - left,
                              cv2.BORDER_CONSTANT, value=(0, 0, 0))


def apply_clahe_lab(rgb, clip=CFG.CLAHE_CLIP, tile=CFG.CLAHE_TILE):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)


def load_and_preprocess(path, size):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"cannot read {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if CFG.USE_CROP:
        rgb = crop_retina(rgb)
    if CFG.USE_CLAHE:
        rgb = apply_clahe_lab(rgb)
    rgb = pad_to_square(rgb)
    interp = cv2.INTER_AREA if rgb.shape[0] >= size else cv2.INTER_CUBIC
    return cv2.resize(rgb, (size, size), interpolation=interp)


PREP_VERSION = "v3_crop_pad_noclahe"        # new name -> never reuses the CLAHE image caches


def cache_fingerprint(df, paths, csv_path, tag):
    h = hashlib.sha1()
    h.update(json.dumps({
        "tag": tag, "prep": PREP_VERSION,
        "use_crop": CFG.USE_CROP, "crop_tol": CFG.CROP_TOL, "crop_min_frac": CFG.CROP_MIN_FRAC,
        "use_clahe": CFG.USE_CLAHE, "clahe_clip": CFG.CLAHE_CLIP, "clahe_tile": CFG.CLAHE_TILE,
        "img_size": CFG.IMG_SIZE, "n": len(df),
    }, sort_keys=True).encode())
    with open(csv_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    for p in paths:
        st = os.stat(p)
        h.update(f"{os.path.basename(p)}|{st.st_size}|{st.st_mtime_ns}".encode())
    return h.hexdigest()[:12]


def build_cache(df, img_dir, tag, csv_path):
    paths = [find_image(img_dir, i) for i in df["id"]]
    sig   = cache_fingerprint(df, paths, csv_path, tag)
    fn    = os.path.join(CFG.CACHE_DIR, f"{tag}_{PREP_VERSION}_s{CFG.IMG_SIZE}_{sig}.npy")
    if os.path.exists(fn):
        arr = np.load(fn)
        if arr.shape == (len(df), CFG.IMG_SIZE, CFG.IMG_SIZE, 3) and arr.dtype == np.uint8:
            print(f"cache hit: {fn}")
            return arr
        os.remove(fn)
    arr = Parallel(n_jobs=CFG.N_JOBS)(
        delayed(load_and_preprocess)(p, CFG.IMG_SIZE) for p in tqdm(paths, desc=f"cache {tag}"))
    arr = np.stack(arr).astype(np.uint8)
    np.save(fn, arr)
    print(f"cache saved: {fn}")
    return arr


X_TRAIN = build_cache(TRAIN_DF, CFG.TRAIN_DIR, "train", CFG.TRAIN_CSV)
X_VAL   = build_cache(VAL_DF,   CFG.VAL_DIR,   "val",   CFG.VAL_CSV)
X_TEST  = build_cache(TEST_DF,  CFG.TEST_DIR,  "test",  CFG.TEST_CSV)
Y_TEST  = TEST_DF["label"].values

# ---- v3: pool train + val for cross-validation; test stays untouched ----------
X_POOL  = np.concatenate([X_TRAIN, X_VAL])
Y_POOL  = np.concatenate([TRAIN_DF["label"].values, VAL_DF["label"].values])
ID_POOL = np.concatenate([TRAIN_DF["id"].values, VAL_DF["id"].values])
del X_TRAIN, X_VAL
gc.collect()
N_POOL = len(Y_POOL)
print(f"pool n={N_POOL} counts {np.bincount(Y_POOL, minlength=CFG.NUM_CLASSES).tolist()} | "
      f"test n={len(Y_TEST)} | X_POOL {X_POOL.shape}")

SKF   = StratifiedKFold(n_splits=K, shuffle=True, random_state=CFG.SEED)
FOLDS = list(SKF.split(np.zeros(N_POOL), Y_POOL))
FOLD_OF = np.empty(N_POOL, dtype=int)
for f, (_, va_i) in enumerate(FOLDS):
    FOLD_OF[va_i] = f
    print(f"fold {f}: train {len(FOLDS[f][0])} | val {len(va_i)} | "
          f"val counts {np.bincount(Y_POOL[va_i], minlength=CFG.NUM_CLASSES).tolist()}")


def save_preprocess_preview(X, y, tag, per_class=2):
    fig, ax = plt.subplots(CFG.NUM_CLASSES, per_class, figsize=(3 * per_class, 3 * CFG.NUM_CLASSES),
                           squeeze=False)
    rng = np.random.default_rng(0)
    for c in range(CFG.NUM_CLASSES):
        idx = np.flatnonzero(y == c)
        pick = rng.choice(idx, size=min(per_class, len(idx)), replace=False) if len(idx) else []
        for j in range(per_class):
            ax[c, j].axis("off")
            if j < len(pick):
                ax[c, j].imshow(X[pick[j]])
                ax[c, j].set_title(CFG.CLASS_NAMES[c], fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{CFG.FIG_DIR}/{tag}.png", dpi=100)
    plt.close()


save_preprocess_preview(X_POOL, Y_POOL, f"{CFG.TAG}_preprocess_preview")


# -----------------------------------------------------------------------------
# 3. Dataset, augmentations, model, loss, Mixup, TTA, EMA   (identical to v2)
# -----------------------------------------------------------------------------
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
BLACK_FILL = (-IMNET_MEAN / IMNET_STD).flatten().tolist()


class DRDataset(Dataset):
    def __init__(self, images, labels, train):
        self.images, self.labels, self.train = images, labels, train
        self.aug = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.5),
            T.RandomAffine(degrees=CFG.AUG_DEGREES,
                           translate=(CFG.AUG_TRANSLATE, CFG.AUG_TRANSLATE),
                           scale=CFG.AUG_SCALE),
            T.ColorJitter(brightness=CFG.AUG_BRIGHTNESS, contrast=CFG.AUG_CONTRAST),
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).permute(2, 0, 1).float() / 255.0
        if self.train:
            img = self.aug(img).clamp(0, 1)
        img = (img - IMNET_MEAN) / IMNET_STD
        return img, int(self.labels[i])


def make_loader(X, y, train):
    return DataLoader(
        DRDataset(X, y, train),
        batch_size=CFG.BATCH_SIZE if train else 64,
        shuffle=train, drop_last=train,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        persistent_workers=(train and CFG.NUM_WORKERS > 0),
    )


def mixup_batch(x, y, prob):
    if random.random() > prob:
        return x, y, y, 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    lam = float(np.random.beta(CFG.MIXUP_ALPHA, CFG.MIXUP_ALPHA))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma, label_smoothing):
        super().__init__()
        self.register_buffer("alpha", alpha.float())
        self.gamma, self.ls = gamma, label_smoothing

    def forward(self, logits, target):
        logits = logits.float()
        ce_s = F.cross_entropy(logits, target, reduction="none", label_smoothing=self.ls)
        pt   = torch.exp(-F.cross_entropy(logits, target, reduction="none"))
        return (self.alpha[target] * (1 - pt) ** self.gamma * ce_s).mean()


def class_weights(y):
    counts = np.bincount(y, minlength=CFG.NUM_CLASSES).astype(np.float64)
    w = 1.0 / (np.maximum(counts, 1) ** CFG.WEIGHT_POWER)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)


class DRNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.EfficientNet_B1_Weights.IMAGENET1K_V1 if pretrained else None
        m = torchvision.models.efficientnet_b1(weights=weights)
        self.features, self.pool = m.features, m.avgpool
        layers = []
        if CFG.DROPOUT > 0:
            layers.append(nn.Dropout(CFG.DROPOUT))
        layers += [nn.Linear(1280, CFG.HIDDEN), nn.ReLU(inplace=True)]
        if CFG.DROPOUT > 0:
            layers.append(nn.Dropout(CFG.DROPOUT))
        layers.append(nn.Linear(CFG.HIDDEN, CFG.NUM_CLASSES))
        self.head = nn.Sequential(*layers)

    def embed(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.head(self.embed(x))


class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.module = copy.deepcopy(model).eval()
        self._src = [v for v in model.state_dict().values() if v.dtype.is_floating_point]
        self._dst = [v for v in self.module.state_dict().values() if v.dtype.is_floating_point]

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(self._dst, d)
        torch._foreach_add_(self._dst, self._src, alpha=1 - d)


def tta_views(x, mode):
    if mode is None:
        return [x]
    views = [x, torch.flip(x, dims=[3]), torch.flip(x, dims=[2])]
    if mode == "TTA":
        views += [TF.rotate(x,  CFG.TTA_ANGLE, fill=BLACK_FILL),
                  TF.rotate(x, -CFG.TTA_ANGLE, fill=BLACK_FILL)]
    return views


@torch.no_grad()
def predict_probs(model, loader, mode=None):
    model.eval()
    out = []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", enabled=AMP_ON):
            p = torch.stack([F.softmax(model(v).float(), 1) for v in tta_views(x, mode)]).mean(0)
        out.append(p.cpu())
    return torch.cat(out).numpy() if out else np.empty((0, CFG.NUM_CLASSES), np.float32)


def predict_both(net, X, y):
    loader = make_loader(X, y, False)
    out = {CFG.INFER_NO_TTA: predict_probs(net, loader, None),
           CFG.INFER_TTA:    predict_probs(net, loader, "TTA")}
    del loader
    return out


# -----------------------------------------------------------------------------
# 4. Metrics, calibration, bootstrap, CV statistics
# -----------------------------------------------------------------------------
def _log(p):
    return np.log(np.clip(p, 1e-12, 1.0))


def apply_temperature(probs, T_):
    z = _log(probs) / T_
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def nll_score(probs, y):
    return float(-_log(probs)[np.arange(len(y)), y].mean())


def brier_score(probs, y):
    return float(((probs - np.eye(CFG.NUM_CLASSES)[y]) ** 2).sum(1).mean())


def fit_temperature(probs, y):
    return float(minimize_scalar(lambda t: nll_score(apply_temperature(probs, t), y),
                                 bounds=(0.3, 5.0), method="bounded").x)


def ece_score(probs, y, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bins_out = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            gap = abs(acc[m].mean() - conf[m].mean())
            ece += m.mean() * gap
            bins_out.append((float(lo), float(hi), int(m.sum()),
                             float(conf[m].mean()), float(acc[m].mean()), float(gap)))
    return float(ece), bins_out


def _sens_spec(yt, yp):
    """Binary sensitivity (TPR) and specificity (TNR); nan if a denominator is 0."""
    tp = int(((yt == 1) & (yp == 1)).sum()); fn = int(((yt == 1) & (yp == 0)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum()); fp = int(((yt == 0) & (yp == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    return sens, spec


def full_metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    ece, _ = ece_score(probs, y)
    m = {
        "acc": accuracy_score(y, pred),
        "macro_precision": p, "macro_recall": r, "macro_f1": f1,
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
        "qwk": cohen_kappa_score(y, pred, weights="quadratic"),
        "ece": ece,
    }
    try:
        yb = label_binarize(y, classes=list(range(CFG.NUM_CLASSES)))
        m["macro_auc"] = roc_auc_score(yb, probs, average="macro")
        m["macro_ap"]  = average_precision_score(yb, probs, average="macro")
    except Exception:
        m["macro_auc"] = m["macro_ap"] = float("nan")
    m["nll"]   = nll_score(probs, y)
    m["brier"] = brier_score(probs, y)
    # ---- sensitivity / specificity ----
    spec_c = [_sens_spec((y == c).astype(int), (pred == c).astype(int))[1]
              for c in range(CFG.NUM_CLASSES) if (y == c).any()]
    m["macro_specificity"] = float(np.nanmean(spec_c))          # one-vs-rest, macro; macro sensitivity == macro_recall
    for name, thr in (("ref", 2), ("any", 1)):                  # referable DR: grade>=2 ; any DR: grade>=1
        m[f"{name}_sens"], m[f"{name}_spec"] = _sens_spec((y >= thr).astype(int), (pred >= thr).astype(int))
    try:
        m["ref_auc"] = roc_auc_score((y >= 2).astype(int), probs[:, 2:].sum(1))
    except Exception:
        m["ref_auc"] = float("nan")
    return m


METRIC_KEYS = ["acc", "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
               "qwk", "ece", "macro_auc", "macro_ap", "nll", "brier",
               "macro_specificity", "ref_sens", "ref_spec", "ref_auc", "any_sens", "any_spec"]


def per_class_metrics(y, probs):
    pred = probs.argmax(1)
    p, r, f1, sup = precision_recall_fscore_support(
        y, pred, labels=list(range(CFG.NUM_CLASSES)), zero_division=0)
    yb = label_binarize(y, classes=list(range(CFG.NUM_CLASSES)))
    rows = []
    for c in range(CFG.NUM_CLASSES):
        pos, neg = yb[:, c].sum() > 0, yb[:, c].sum() < len(yb)
        auc = roc_auc_score(yb[:, c], probs[:, c]) if pos and neg else float("nan")
        ap  = average_precision_score(yb[:, c], probs[:, c]) if pos else float("nan")
        cm = confusion_matrix(y, pred, labels=list(range(CFG.NUM_CLASSES)))
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + tp
        spec = tn / (tn + fp) if (tn + fp) else float("nan")
        rows.append({"cls": CFG.CLASS_NAMES[c], "precision": p[c], "recall": r[c],
                     "sensitivity": r[c], "specificity": spec,
                     "f1": f1[c], "support": int(sup[c]), "auc": auc, "ap": ap})
    return rows


def confidence_summary(probs, y):
    return {"mean_confidence": float(probs.max(1).mean()),
            "accuracy": float((probs.argmax(1) == y).mean()),
            "gap_conf_minus_acc": float(probs.max(1).mean() - (probs.argmax(1) == y).mean())}


def wilson_ci(k, n, z=1.96):
    if n <= 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def stratified_bootstrap_indices(y, rng):
    parts = []
    for cls in range(CFG.NUM_CLASSES):
        idx = np.flatnonzero(y == cls)
        if len(idx):
            parts.append(rng.choice(idx, size=len(idx), replace=True))
    return np.concatenate(parts) if parts else np.array([], dtype=int)


def bootstrap_metric_cis(y, probs, reps=1000, seed=42):
    rng = np.random.default_rng(seed)
    samples = {k: [] for k in METRIC_KEYS}
    for _ in range(reps):
        idx = stratified_bootstrap_indices(y, rng)
        m = full_metrics(y[idx], probs[idx])
        for k in METRIC_KEYS:
            v = m.get(k, np.nan)
            if np.isfinite(v):
                samples[k].append(float(v))
    out = {}
    for k in METRIC_KEYS:
        v = np.asarray(samples[k])
        out[k] = (float(np.quantile(v, .025)), float(np.quantile(v, .975))) if len(v) else (float("nan"),) * 2
    return out


QUICK_KEYS = ("acc", "macro_f1", "qwk", "ece", "nll")


def quick_metrics(y, probs):
    pred = probs.argmax(1)
    return {"acc": float((pred == y).mean()),
            "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
            "qwk": float(cohen_kappa_score(y, pred, weights="quadratic")),
            "ece": ece_score(probs, y)[0],
            "nll": nll_score(probs, y)}


def paired_bootstrap(y, probs_a, probs_b, reps=1000, seed=42, keys=QUICK_KEYS):
    """Paired stratified bootstrap of metric(A) - metric(B) on the SAME resamples."""
    rng = np.random.default_rng(seed)
    full_a, full_b = quick_metrics(y, probs_a), quick_metrics(y, probs_b)
    diffs = {k: [] for k in keys}
    for _ in range(reps):
        idx = stratified_bootstrap_indices(y, rng)
        a, b = quick_metrics(y[idx], probs_a[idx]), quick_metrics(y[idx], probs_b[idx])
        for k in keys:
            diffs[k].append(a[k] - b[k])
    out = {}
    for k in keys:
        v = np.asarray(diffs[k])
        out[k] = {"diff": full_a[k] - full_b[k],
                  "lo": float(np.quantile(v, .025)), "hi": float(np.quantile(v, .975)),
                  "p_gt0": float((v > 0).mean())}
    return out


def cv_stats(values, n_tr, n_va):
    """Across-fold summary. 'nb' = Nadeau-Bengio corrected t-interval, which
    accounts for the overlap of training sets between folds (the naive interval
    is too narrow). Only meaningful for metrics measured on the held-out fold."""
    v = np.asarray(values, dtype=float)
    k = len(v)
    mean, sd = float(np.nanmean(v)), float(np.nanstd(v, ddof=1))
    t = float(sps.t.ppf(0.975, k - 1))
    se_naive = sd / math.sqrt(k)
    se_nb    = sd * math.sqrt(1.0 / k + n_va / n_tr)
    return {"mean": mean, "sd": sd, "min": float(np.nanmin(v)), "max": float(np.nanmax(v)),
            "ci95_naive_low": mean - t * se_naive, "ci95_naive_high": mean + t * se_naive,
            "ci95_nb_low":    mean - t * se_nb,    "ci95_nb_high":    mean + t * se_nb}


# -----------------------------------------------------------------------------
# 5. Training one fold (identical recipe to v2)
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, crit):
    model.eval()
    lg_list, y_list = [], []
    for x, y in loader:
        with torch.autocast("cuda", enabled=AMP_ON):
            lg = model(x.to(DEVICE, non_blocking=True))
        lg_list.append(lg.float().cpu())
        y_list.append(y)
    logits, labels = torch.cat(lg_list), torch.cat(y_list)
    probs = F.softmax(logits, 1).numpy()
    m = full_metrics(labels.numpy(), probs)
    m["loss"] = crit(logits.to(DEVICE), labels.to(DEVICE)).item()
    m["nll"]  = F.cross_entropy(logits, labels).item()
    return m


def train_fold(fold, Xtr, ytr, Xva, yva):
    set_seed(CFG.SEED + fold)
    net  = DRNet().to(DEVICE)
    crit = FocalLoss(class_weights(ytr), CFG.FOCAL_GAMMA, CFG.LABEL_SMOOTHING).to(DEVICE)
    print(f"[fold {fold}] class weights:", [round(float(w), 3) for w in crit.alpha.cpu()])

    opt = torch.optim.AdamW(
        [{"params": net.features.parameters(), "lr": CFG.BACKBONE_LR},
         {"params": net.head.parameters(),     "lr": CFG.HEAD_LR}],
        weight_decay=CFG.WEIGHT_DECAY)
    train_loader = make_loader(Xtr, ytr, True)
    val_loader   = make_loader(Xva, yva, False)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[CFG.BACKBONE_LR, CFG.HEAD_LR],
        total_steps=CFG.EPOCHS * len(train_loader),
        pct_start=CFG.PCT_START, cycle_momentum=False)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ON)
    ema = EMA(net, CFG.EMA_DECAY)

    hist = {k: [] for k in [
        "train_loss", "train_acc", "train_acc_clean",
        "val_loss", "val_nll", "val_acc", "val_macro_f1", "val_qwk", "val_ece",
        "val_acc_raw", "val_macro_f1_raw", "val_qwk_raw",
        "select_score", "lr_backbone", "lr_head", "epoch_sec"]}
    best_score, best_ep, best_state, best_m = -1e9, 0, None, None
    ckpt = f"{CFG.FIG_DIR}/{CFG.TAG}_fold{fold}_best.pt"

    for ep in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        net.train()
        loss_sum, correct, seen = 0.0, 0, 0
        c_correct, c_seen = 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            xm, ya, yb, lam = mixup_batch(x, y, CFG.MIX_PROB)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=AMP_ON):
                logits = net(xm)
            loss = crit(logits, ya) if lam == 1.0 else \
                   lam * crit(logits, ya) + (1 - lam) * crit(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update()

            bs = x.size(0)
            loss_sum += loss.item() * bs
            tgt = ya if lam >= 0.5 else yb
            n_ok = (logits.argmax(1) == tgt).sum().item()
            correct += n_ok; seen += bs
            if lam == 1.0:
                c_correct += n_ok; c_seen += bs

        vm_ema = eval_epoch(ema.module, val_loader, crit)
        vm_raw = eval_epoch(net,        val_loader, crit)
        score  = vm_ema["qwk"] + vm_ema["macro_f1"]

        hist["train_loss"].append(loss_sum / max(seen, 1))
        hist["train_acc"].append(100 * correct / max(seen, 1))
        hist["train_acc_clean"].append(100 * c_correct / c_seen if c_seen else float("nan"))
        hist["val_loss"].append(vm_ema["loss"])
        hist["val_nll"].append(vm_ema["nll"])
        hist["val_acc"].append(100 * vm_ema["acc"])
        hist["val_macro_f1"].append(vm_ema["macro_f1"])
        hist["val_qwk"].append(vm_ema["qwk"])
        hist["val_ece"].append(vm_ema["ece"])
        hist["val_acc_raw"].append(100 * vm_raw["acc"])
        hist["val_macro_f1_raw"].append(vm_raw["macro_f1"])
        hist["val_qwk_raw"].append(vm_raw["qwk"])
        hist["select_score"].append(score)
        hist["lr_backbone"].append(opt.param_groups[0]["lr"])
        hist["lr_head"].append(opt.param_groups[1]["lr"])
        hist["epoch_sec"].append(time.time() - t0)

        msg = (f"[f{fold} ep {ep:02d}] tr_loss {hist['train_loss'][-1]:.4f} "
               f"tr_acc_clean {hist['train_acc_clean'][-1]:.1f} | "
               f"EMA val_acc {100*vm_ema['acc']:.2f} F1 {vm_ema['macro_f1']:.4f} "
               f"QWK {vm_ema['qwk']:.4f} NLL {vm_ema['nll']:.4f} ECE {vm_ema['ece']:.4f} | "
               f"raw acc {100*vm_raw['acc']:.2f} QWK {vm_raw['qwk']:.4f} | "
               f"lr {opt.param_groups[0]['lr']:.1e}/{opt.param_groups[1]['lr']:.1e}")
        if ep >= CFG.SELECT_MIN_EPOCH and score > best_score:
            best_score, best_ep, best_m = score, ep, vm_ema
            best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
            torch.save(best_state, ckpt)
            msg += "  <-- best"
        print(msg + f" | {hist['epoch_sec'][-1]:.0f}s")

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
        torch.save(best_state, ckpt)
        best_ep, best_score, best_m = CFG.EPOCHS, hist["select_score"][-1], vm_ema
    ema.module.load_state_dict(best_state)
    best_m = {k: float(v) for k, v in best_m.items()}
    del train_loader, val_loader, net
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return ema.module, hist, best_ep, best_score, best_m, ckpt


# -----------------------------------------------------------------------------
# 6. Run the 5 folds (each fold model predicts held-out fold + test, once)
# -----------------------------------------------------------------------------
FOLD = {}
t_all = time.time()
for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    state_fn = f"{CFG.FIG_DIR}/{CFG.TAG}_fold{fold}_state.pkl"
    st = None
    if CFG.RESUME and os.path.exists(state_fn):
        try:
            with open(state_fn, "rb") as fh:
                st = pickle.load(fh)
            if (st.get("cfg_sig") != CFG_SIG or not os.path.exists(st["ckpt"])
                    or not np.array_equal(st["va_idx"], va_idx)):
                st = None
            else:
                print(f"[fold {fold}] resumed from {state_fn}")
        except Exception:
            st = None

    if st is None:
        print(f"\n{'=' * 25} FOLD {fold + 1}/{K} | train {len(tr_idx)} | val {len(va_idx)} {'=' * 25}")
        t0 = time.time()
        Xtr, ytr = X_POOL[tr_idx], Y_POOL[tr_idx]
        Xva, yva = X_POOL[va_idx], Y_POOL[va_idx]
        net, hist, best_ep, best_score, best_m, ckpt = train_fold(fold, Xtr, ytr, Xva, yva)
        train_sec = time.time() - t0
        val_probs  = predict_both(net, Xva, yva)
        test_probs = predict_both(net, X_TEST, Y_TEST)     # this model's one test evaluation
        st = {"cfg_sig": CFG_SIG, "va_idx": va_idx, "hist": hist, "best_ep": best_ep,
              "best_score": float(best_score), "best_m": best_m, "ckpt": ckpt,
              "train_sec": train_sec, "val_probs": val_probs, "test_probs": test_probs}
        with open(state_fn, "wb") as fh:
            pickle.dump(st, fh)
        print(f"[fold {fold}] best epoch {best_ep} | select score {best_score:.4f} | "
              f"val acc {100 * best_m['acc']:.2f}% | {train_sec / 60:.1f} min")
        del net, Xtr, Xva
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    FOLD[fold] = st

TOTAL_SEC = time.time() - t_all
print(f"\nAll folds done in {TOTAL_SEC / 60:.1f} min")


# -----------------------------------------------------------------------------
# 7. Statistics: per-fold, OOF, ensemble, bootstrap CIs, paired bootstrap
# -----------------------------------------------------------------------------
BLOCKS = []


def make_block(fold, split, inf, y, raw, cal, note=""):
    b = {"fold": fold, "split": split, "inf": inf, "y": y, "raw": raw, "cal": cal, "note": note,
         "m_raw": full_metrics(y, raw), "m_cal": full_metrics(y, cal)}
    BLOCKS.append(b)
    return b


T_FOLD = {f: {} for f in range(K)}
for f in range(K):
    st = FOLD[f]
    yva = Y_POOL[st["va_idx"]]
    for inf in MODES:
        Tk = fit_temperature(st["val_probs"][inf], yva)           # fit on this fold's val only
        T_FOLD[f][inf] = Tk
        make_block(f, "val", inf, yva, st["val_probs"][inf],
                   apply_temperature(st["val_probs"][inf], Tk),
                   note="T fit on this val fold (in-sample, optimistic)")
        make_block(f, "test", inf, Y_TEST, st["test_probs"][inf],
                   apply_temperature(st["test_probs"][inf], Tk),
                   note="T from this fold's val; test never used for fitting")

# ---- OOF predictions over all pooled images --------------------------------
OOF_RAW = {inf: np.zeros((N_POOL, CFG.NUM_CLASSES), np.float32) for inf in MODES}
for f in range(K):
    for inf in MODES:
        OOF_RAW[inf][FOLD[f]["va_idx"]] = FOLD[f]["val_probs"][inf]

T_CROSS = {f: {} for f in range(K)}
OOF_CAL = {inf: np.zeros_like(OOF_RAW[inf]) for inf in MODES}
for inf in MODES:
    for f in range(K):
        other = FOLD_OF != f
        Tx = fit_temperature(OOF_RAW[inf][other], Y_POOL[other])  # fit on the OTHER folds
        T_CROSS[f][inf] = Tx
        OOF_CAL[inf][~other] = apply_temperature(OOF_RAW[inf][~other], Tx)
    make_block("all", "oof", inf, Y_POOL, OOF_RAW[inf], OOF_CAL[inf],
               note="calibrated = cross-fitted T (honest)")

# ---- 5-model ensemble on the fixed test set --------------------------------
ENS_RAW, ENS_CAL = {}, {}
for inf in MODES:
    ENS_RAW[inf] = np.mean([FOLD[f]["test_probs"][inf] for f in range(K)], axis=0)
    ENS_CAL[inf] = np.mean([[b for b in BLOCKS if b["fold"] == f and b["split"] == "test"
                             and b["inf"] == inf][0]["cal"] for f in range(K)], axis=0)
    make_block("ensemble", "test", inf, Y_TEST, ENS_RAW[inf], ENS_CAL[inf],
               note="raw = mean of fold probs; calibrated = mean of per-fold calibrated probs")

# ---- bootstrap CIs (OOF + ensemble) ----------------------------------------
print("\nBootstrapping (this takes a few minutes)...")
for b in BLOCKS:
    if b["fold"] in ("all", "ensemble"):
        b["ci_raw"] = bootstrap_metric_cis(b["y"], b["raw"], reps=CFG.BOOTSTRAP_REPS, seed=CFG.SEED)
        b["ci_cal"] = bootstrap_metric_cis(b["y"], b["cal"], reps=CFG.BOOTSTRAP_REPS, seed=CFG.SEED)

# ---- paired bootstrap: TTA - noTTA on OOF (raw) ----------------------------
PAIRED_TTA = paired_bootstrap(Y_POOL, OOF_RAW[CFG.INFER_TTA], OOF_RAW[CFG.INFER_NO_TTA],
                              reps=CFG.BOOTSTRAP_REPS, seed=CFG.SEED)

# ---- across-fold statistics -------------------------------------------------
N_TR = float(np.mean([len(FOLDS[f][0]) for f in range(K)]))
N_VA = float(np.mean([len(FOLDS[f][1]) for f in range(K)]))
CV = {}
for split in ("val", "test"):
    for inf in MODES:
        for variant in ("raw", "cal"):
            for k in METRIC_KEYS:
                vals = [[b for b in BLOCKS if b["fold"] == f and b["split"] == split
                         and b["inf"] == inf][0]["m_" + variant][k] for f in range(K)]
                CV[(split, inf, variant, k)] = cv_stats(vals, N_TR, N_VA)


def blk(fold, split, inf):
    return [b for b in BLOCKS if b["fold"] == fold and b["split"] == split and b["inf"] == inf][0]


def fm(m, keys=("acc", "macro_f1", "qwk", "ece", "nll")):
    names = {"acc": "acc", "macro_f1": "F1", "qwk": "QWK", "ece": "ECE", "nll": "NLL"}
    return " | ".join(f"{names[k]} {m[k]:.4f}" for k in keys)


def fci(ci):
    return f"[{ci[0]:.4f}, {ci[1]:.4f}]"


def sens_line(m, ci):
    return (f"{'':10s} macro-spec {m['macro_specificity']:.4f} {fci(ci['macro_specificity'])} | "
            f"referable-DR sens {m['ref_sens']:.4f} {fci(ci['ref_sens'])} "
            f"spec {m['ref_spec']:.4f} {fci(ci['ref_spec'])} AUC {m['ref_auc']:.4f} | "
            f"any-DR sens {m['any_sens']:.4f} spec {m['any_spec']:.4f}")


print("\n" + "=" * 100)
print("PER-FOLD RESULTS (raw probabilities; T = fold temperature)")
for f in range(K):
    print(f"fold {f}: best epoch {FOLD[f]['best_ep']} | T noTTA {T_FOLD[f][CFG.INFER_NO_TTA]:.3f} "
          f"TTA {T_FOLD[f][CFG.INFER_TTA]:.3f}")
    for inf in MODES:
        print(f"   val  {inf:5s} | {fm(blk(f, 'val', inf)['m_raw'])}")
        print(f"   test {inf:5s} | {fm(blk(f, 'test', inf)['m_raw'])}"
              f"  || cal ECE {blk(f, 'test', inf)['m_cal']['ece']:.4f} NLL {blk(f, 'test', inf)['m_cal']['nll']:.4f}")

print("\n" + "=" * 100)
print("ACROSS-FOLD SUMMARY (mean ± SD). Val CI = Nadeau-Bengio corrected 95% t-interval.")
for inf in MODES:
    for split in ("val", "test"):
        parts = []
        for k, nm in (("acc", "acc"), ("macro_f1", "F1"), ("qwk", "QWK"), ("ece", "ECE"), ("nll", "NLL")):
            s = CV[(split, inf, "raw", k)]
            txt = f"{nm} {s['mean']:.4f}±{s['sd']:.4f}"
            if split == "val" and k in ("acc", "qwk"):
                txt += f" [{s['ci95_nb_low']:.4f}, {s['ci95_nb_high']:.4f}]"
            parts.append(txt)
        print(f"{split:4s} {inf:5s} | " + " | ".join(parts))
        s = CV[(split, inf, "cal", "ece")]
        n = CV[(split, inf, "cal", "nll")]
        print(f"{'':10s} calibrated: ECE {s['mean']:.4f}±{s['sd']:.4f} | NLL {n['mean']:.4f}±{n['sd']:.4f}")

print("\n" + "=" * 100)
print("OUT-OF-FOLD (all 3294 pooled images) with stratified-bootstrap 95% CI")
for inf in MODES:
    b = blk("all", "oof", inf)
    for variant, label in (("raw", "raw"), ("cal", "cross-fitted calibrated")):
        m, ci = b["m_" + variant], b["ci_" + variant]
        print(f"OOF {inf:5s} {label:24s} | "
              f"acc {m['acc']:.4f} {fci(ci['acc'])} | F1 {m['macro_f1']:.4f} {fci(ci['macro_f1'])} | "
              f"QWK {m['qwk']:.4f} {fci(ci['qwk'])} | ECE {m['ece']:.4f} {fci(ci['ece'])} | NLL {m['nll']:.4f}")
        print(sens_line(m, ci))
    cs = confidence_summary(b["raw"], Y_POOL)
    print(f"{'':10s} raw mean_conf {cs['mean_confidence']:.4f} acc {cs['accuracy']:.4f} "
          f"conf-acc {cs['gap_conf_minus_acc']:+.4f}")

print("\n" + "=" * 100)
print("TEST SET: 5-model ENSEMBLE with stratified-bootstrap 95% CI (reported separately from single models)")
for inf in MODES:
    b = blk("ensemble", "test", inf)
    for variant, label in (("raw", "raw"), ("cal", "calibrated")):
        m, ci = b["m_" + variant], b["ci_" + variant]
        print(f"ENS {inf:5s} {label:10s} | "
              f"acc {m['acc']:.4f} {fci(ci['acc'])} | F1 {m['macro_f1']:.4f} {fci(ci['macro_f1'])} | "
              f"QWK {m['qwk']:.4f} {fci(ci['qwk'])} | ECE {m['ece']:.4f} {fci(ci['ece'])} | NLL {m['nll']:.4f}")
        print(sens_line(m, ci))
    cs = confidence_summary(b["cal"], Y_TEST)
    print(f"{'':10s} calibrated conf-acc {cs['gap_conf_minus_acc']:+.4f} "
          f"(+ overconfident, - underconfident)")

print("\n" + "=" * 100)
print("PAIRED BOOTSTRAP on OOF: TTA minus noTTA (CI excluding 0 => real difference)")
for k in QUICK_KEYS:
    r = PAIRED_TTA[k]
    print(f"  {k:9s} diff {r['diff']:+.4f}  95% CI [{r['lo']:+.4f}, {r['hi']:+.4f}]  P(diff>0)={r['p_gt0']:.3f}")


# -----------------------------------------------------------------------------
# 8. Grad-CAM + t-SNE (single model: the fold with the best VALIDATION score)
# -----------------------------------------------------------------------------
class GradCAM:
    def __init__(self, net):
        self.net = net.eval()
        self.acts = None
        self.grads = None
        self.handle = net.features[-1].register_forward_hook(self._fwd)

    def _fwd(self, m, i, o):
        if o.requires_grad:
            self.acts = o
            o.register_hook(lambda g: setattr(self, "grads", g))

    def remove(self):
        self.handle.remove()

    def __call__(self, x, cls=None):
        self.net.zero_grad(set_to_none=True)
        logits = self.net(x)
        cls = int(logits.argmax(1).item()) if cls is None else int(cls)
        logits[0, cls].backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls


def save_gradcam_figure(net, X, y, pred, tag):
    net = net.to(DEVICE).float().eval()
    cam_fn = GradCAM(net)
    picks = []
    for c in range(CFG.NUM_CLASSES):
        idx = np.where((y == c) & (pred == c))[0]
        if len(idx):
            picks.append((idx[0], "correct"))
    wrong = np.where(y != pred)[0]
    picks += [(i, "WRONG") for i in wrong[:4]]
    if not picks:
        cam_fn.remove()
        return
    fig, ax = plt.subplots(2, len(picks), figsize=(3 * len(picks), 6), squeeze=False)
    for j, (i, kind) in enumerate(picks):
        x = ((torch.from_numpy(X[i]).permute(2, 0, 1).float() / 255
              - IMNET_MEAN) / IMNET_STD).unsqueeze(0).to(DEVICE)
        cam, cam_pred = cam_fn(x)
        heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8),
                                              cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay = (0.55 * X[i].astype(np.float32) + 0.45 * heat.astype(np.float32)
                   ).clip(0, 255).astype(np.uint8)
        ax[0, j].imshow(X[i])
        ax[0, j].set_title(f"True: {CFG.CLASS_NAMES[y[i]]}", fontsize=9)
        ax[0, j].axis("off")
        ax[1, j].imshow(overlay)
        ax[1, j].set_title(f"Pred: {CFG.CLASS_NAMES[cam_pred]} ({kind})", fontsize=9)
        ax[1, j].axis("off")
    plt.tight_layout()
    plt.savefig(f"{CFG.FIG_DIR}/{tag}.png", dpi=120)
    plt.close()
    cam_fn.remove()


@torch.no_grad()
def extract_embeddings(net, X):
    net = net.to(DEVICE).eval()
    feats = []
    for xb, _ in make_loader(X, np.zeros(len(X), dtype=int), False):
        with torch.autocast("cuda", enabled=AMP_ON):
            feats.append(net.embed(xb.to(DEVICE)).float().cpu())
    return torch.cat(feats).numpy()


def space_stats(Z, y, metric, suffix):
    stats_ = {f"silhouette_gt_{suffix}": float(silhouette_score(Z, y, metric=metric))}
    present = [c for c in range(CFG.NUM_CLASSES) if (y == c).any()]
    cent = {c: Z[y == c].mean(0) for c in present}
    stats_[f"mean_inter_centroid_distance_{suffix}"] = float(np.mean([
        np.linalg.norm(cent[a] - cent[b]) for i, a in enumerate(present) for b in present[i + 1:]]))
    for c in present:
        stats_[f"intra_class_spread_{CFG.CLASS_NAMES[c]}_{suffix}"] = float(
            np.linalg.norm(Z[y == c] - cent[c], axis=1).mean())
    return stats_


VIS_FOLD = int(max(FOLD, key=lambda f: FOLD[f]["best_score"]))      # chosen on val score, not test
print(f"\nGrad-CAM / t-SNE use fold {VIS_FOLD} (highest validation selection score).")
VIS_NET = DRNet(pretrained=False)
VIS_NET.load_state_dict(torch.load(FOLD[VIS_FOLD]["ckpt"], map_location="cpu"))
VIS_NET = VIS_NET.to(DEVICE).eval()
VIS_PRED = FOLD[VIS_FOLD]["test_probs"][CFG.INFER_TTA].argmax(1)

save_gradcam_figure(VIS_NET, X_TEST, Y_TEST, VIS_PRED, tag=f"{CFG.TAG}_gradcam_fold{VIS_FOLD}")

EMB       = extract_embeddings(VIS_NET, X_TEST)
EMB_STATS = space_stats(EMB, Y_TEST, "cosine", "original_1280d")
perplexity = float(min(30, max(2, (len(EMB) - 1) // 3)))
Z2 = TSNE(n_components=2, perplexity=perplexity, init="pca",
          random_state=CFG.SEED).fit_transform(EMB)
EMB_STATS.update(space_stats(Z2, Y_TEST, "euclidean", "tsne_coords"))

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for c in range(CFG.NUM_CLASSES):
    ax[0].scatter(*Z2[Y_TEST == c].T, s=12, label=CFG.CLASS_NAMES[c])
    ax[1].scatter(*Z2[VIS_PRED == c].T, s=12, label=CFG.CLASS_NAMES[c])
ax[0].set_title(f"t-SNE GT (fold {VIS_FOLD} model)")
ax[1].set_title("t-SNE pred")
ax[0].legend(); ax[1].legend()
plt.tight_layout()
plt.savefig(f"{CFG.FIG_DIR}/{CFG.TAG}_tsne_fold{VIS_FOLD}.png", dpi=120)
plt.close()
print("Grad-CAM and t-SNE figures saved.")


# -----------------------------------------------------------------------------
# 9. Write EVERYTHING to ONE CSV
# -----------------------------------------------------------------------------
ROWS = []


def add(section, metric, value, split="", inference="", cls="", epoch="", note="", fold=""):
    ROWS.append({"section": section, "fold": fold, "split": split, "inference": inference,
                 "class": cls, "epoch": epoch, "metric": metric, "value": value, "note": note})


V_NOTES = {
    "IMG_SIZE": "v2 step 1 (v1/paper: 224)",
    "USE_CROP": "v2 step 2", "CROP_TOL": "v2 step 2", "CROP_MIN_FRAC": "v2 step 2",
    "BACKBONE_LR": "v2 step 3", "HEAD_LR": "v2 step 3", "PCT_START": "v2 step 3",
    "GRAD_CLIP": "v2 step 3", "EMA_DECAY": "v2 step 3",
    "FOCAL_GAMMA": "v2 step 4 (v1/paper: 2.0)", "WEIGHT_POWER": "v2 step 4 (v1: 1.0)",
    "MIXUP_ALPHA": "v2 step 4 (CutMix removed)", "MIX_PROB": "v2 step 4 (v1: 0.5)",
    "DROPOUT": "v2 step 4 (v1: 0.0)",
    "SELECT_MIN_EPOCH": "v2 step 5; selection score = val QWK + macro-F1 on EMA model",
    "N_FOLDS": "v3 new: stratified K-fold on pooled train+val",
    "RESUME": "v3 new",
}
assumption_keys = {
    "CLAHE_CLIP", "CLAHE_TILE", "HIDDEN", "TTA_ANGLE", "AUG_DEGREES", "AUG_TRANSLATE",
    "AUG_SCALE", "AUG_BRIGHTNESS", "AUG_CONTRAST", "USE_AMP", "USE_CLAHE",
    "NUM_WORKERS", "N_JOBS", "BOOTSTRAP_REPS", "SEED", "TAG",
    "OUT_DIR", "CACHE_DIR", "FIG_DIR", "ROOT",
    "TRAIN_DIR", "VAL_DIR", "TEST_DIR", "TRAIN_CSV", "VAL_CSV", "TEST_CSV",
}
for k, v in vars(CFG).items():
    if not k.startswith("_") and not callable(v):
        add("config", k, v,
            note=V_NOTES.get(k, "assumption" if k in assumption_keys else "paper-specified"))
add("config", "CLASS_WEIGHT_IMPLEMENTATION", "inv_freq^WEIGHT_POWER_mean1",
    note="weights = 1/count^0.5 rescaled to class-mean 1, recomputed per fold from its training labels")
add("config", "PREPROCESS_ORDER", "crop -> pad_to_square -> resize (CLAHE disabled)")
add("config", "FOLD_SEED_RULE", "SEED + fold_index", note="master seed 42; stratified split random_state=42")
add("config", "CV_DESIGN", "pooled train+val, stratified 5-fold; test untouched",
    note="test never used for epoch selection or temperature fitting")

add("run_info", "torch_version", torch.__version__)
add("run_info", "python_version", platform.python_version())
add("run_info", "gpus_visible", "; ".join(GPU_NAMES) if GPU_NAMES else "cpu")
add("run_info", "config_signature", CFG_SIG)
add("run_info", "pool_size", N_POOL)
add("run_info", "test_size", int(len(Y_TEST)))
add("run_info", "total_run_seconds_this_session", round(TOTAL_SEC, 1),
    note="folds resumed from cache contribute ~0 s")
add("run_info", "vis_fold_gradcam_tsne", VIS_FOLD, note="fold with best validation selection score")
for f in range(K):
    st = FOLD[f]
    add("run_info", "best_epoch", st["best_ep"], fold=f)
    add("run_info", "best_select_score_qwk_plus_macro_f1", st["best_score"], fold=f)
    add("run_info", "val_acc_at_best_epoch_ema", st["best_m"]["acc"], fold=f)
    add("run_info", "val_qwk_at_best_epoch_ema", st["best_m"]["qwk"], fold=f)
    add("run_info", "val_macro_f1_at_best_epoch_ema", st["best_m"]["macro_f1"], fold=f)
    add("run_info", "training_seconds", round(st["train_sec"], 1), fold=f)
    add("run_info", "n_train", int(len(FOLDS[f][0])), fold=f)
    add("run_info", "n_val", int(len(FOLDS[f][1])), fold=f)

for i in range(N_POOL):
    add("fold_assignment", str(ID_POOL[i]), int(FOLD_OF[i]), cls=CFG.CLASS_NAMES[Y_POOL[i]])

for f in range(K):
    for i, ep in enumerate(range(1, CFG.EPOCHS + 1)):
        for k in FOLD[f]["hist"]:
            add("history", k, FOLD[f]["hist"][k][i], epoch=ep, fold=f)


def report_block(b):
    fold, split, inf, y = b["fold"], b["split"], b["inf"], b["y"]
    kw = dict(fold=fold, split=split, inference=inf)
    for variant in ("raw", "cal"):
        probs, m = b[variant], b["m_" + variant]
        sfx = "" if variant == "raw" else "_calibrated"
        note = b["note"] if variant == "cal" else ""
        for k in METRIC_KEYS:
            add("overall" + sfx, k, m[k], note=note, **kw)
        lo, hi = wilson_ci(int((probs.argmax(1) == y).sum()), len(y))
        add("overall" + sfx, "acc_wilson_ci95_low",  lo, **kw)
        add("overall" + sfx, "acc_wilson_ci95_high", hi, **kw)
        for row in per_class_metrics(y, probs):
            for k in ("precision", "recall", "sensitivity", "specificity", "f1", "support", "auc", "ap"):
                add("per_class" + sfx, k, row[k], cls=row["cls"], **kw)
        cm = confusion_matrix(y, probs.argmax(1), labels=range(CFG.NUM_CLASSES))
        for i, tname in enumerate(CFG.CLASS_NAMES):
            for j, pname in enumerate(CFG.CLASS_NAMES):
                add("confusion" + sfx, f"pred_{pname}", int(cm[i, j]), cls=tname, **kw)
        _, bins = ece_score(probs, y)
        for lo_, hi_, cnt, mc, ac, gp in bins:
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_count",     cnt, **kw)
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_mean_conf", mc,  **kw)
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_accuracy",  ac,  **kw)
            add("calibration_bin" + sfx, f"bin_{lo_:.1f}-{hi_:.1f}_gap",       gp,  **kw)
        for k, v in confidence_summary(probs, y).items():
            add("calibration_summary" + sfx, k, v, **kw)
        ci = b.get("ci_" + variant)
        if ci:
            for metric, (lo_, hi_) in ci.items():
                add("bootstrap_ci" + sfx, f"{metric}_ci95_low",  lo_, **kw)
                add("bootstrap_ci" + sfx, f"{metric}_ci95_high", hi_, **kw)


for b in BLOCKS:
    report_block(b)

# temperatures
for f in range(K):
    for inf in MODES:
        add("temperature_scaling", "temperature_fold_val_fit", T_FOLD[f][inf], fold=f, inference=inf,
            note="fit on this fold's val; applied to this fold's val and test")
        add("temperature_scaling", "temperature_cross_fit_oof", T_CROSS[f][inf], fold=f, inference=inf,
            note="fit on the other folds' OOF; used only for honest OOF calibration")

# across-fold statistics
for (split, inf, variant, k), s in CV.items():
    sec = "cv_summary" if variant == "raw" else "cv_summary_calibrated"
    for stat, val in s.items():
        if split == "test" and stat.startswith("ci95"):
            continue                                   # test CI across folds ignores test-sampling noise
        add(sec, f"{k}_{stat}", val, fold="cv", split=split, inference=inf,
            note=("Nadeau-Bengio corrected" if stat.startswith("ci95_nb") else
                  "naive t-interval (too narrow, folds overlap)" if stat.startswith("ci95_naive") else
                  "variation across fold-trained models on the fixed test set" if split == "test" else ""))

# paired bootstrap TTA - noTTA
for k, r in PAIRED_TTA.items():
    add("paired_bootstrap_oof", f"{k}_diff_TTA_minus_noTTA", r["diff"], fold="all", split="oof")
    add("paired_bootstrap_oof", f"{k}_ci95_low",  r["lo"],    fold="all", split="oof")
    add("paired_bootstrap_oof", f"{k}_ci95_high", r["hi"],    fold="all", split="oof")
    add("paired_bootstrap_oof", f"{k}_p_diff_gt0", r["p_gt0"], fold="all", split="oof")

for k, v in EMB_STATS.items():
    add("embedding", k, v, split="test", inference=CFG.INFER_TTA, fold=VIS_FOLD)

# ---- references --------------------------------------------------------------
BASELINE_V1 = {
    ("val",  CFG.INFER_NO_TTA): {"acc": 0.8607, "macro_f1": 0.7324, "weighted_f1": 0.8589, "qwk": 0.9179, "ece": 0.0654},
    ("val",  CFG.INFER_TTA):    {"acc": 0.8607, "macro_f1": 0.7573, "weighted_f1": 0.8607, "qwk": 0.9204, "ece": 0.0666},
    ("test", CFG.INFER_NO_TTA): {"acc": 0.8251, "macro_f1": 0.6389, "weighted_f1": 0.8261, "qwk": 0.9030, "ece": 0.0351},
    ("test", CFG.INFER_TTA):    {"acc": 0.8197, "macro_f1": 0.6285, "weighted_f1": 0.8199, "qwk": 0.9036, "ece": 0.0484},
}
BASELINE_V2 = {
    ("val",  CFG.INFER_NO_TTA): {"acc": 0.8552, "macro_f1": 0.7325, "weighted_f1": 0.8518, "qwk": 0.9158, "ece": 0.0790},
    ("val",  CFG.INFER_TTA):    {"acc": 0.8388, "macro_f1": 0.7064, "weighted_f1": 0.8350, "qwk": 0.9041, "ece": 0.0745},
    ("test", CFG.INFER_NO_TTA): {"acc": 0.8525, "macro_f1": 0.6863, "weighted_f1": 0.8476, "qwk": 0.9062, "ece": 0.0822},
    ("test", CFG.INFER_TTA):    {"acc": 0.8579, "macro_f1": 0.6969, "weighted_f1": 0.8543, "qwk": 0.9104, "ece": 0.0587},
}
for sec, ref, nt in (("baseline_v1_reference", BASELINE_V1, "v1 pipeline, 224px, single split, seed 42"),
                     ("baseline_v2_reference", BASELINE_V2, "v2 pipeline (steps 1-5), 320px, single split, seed 42")):
    for (split, inf), r in ref.items():
        for k, v in r.items():
            add(sec, k, v, split=split, inference=inf, note=nt)

PAPER_OVERALL = {
    ("val",  CFG.INFER_NO_TTA): {"acc": 0.8407, "macro_f1": 0.6962},
    ("test", CFG.INFER_NO_TTA): {"acc": 0.8361, "macro_f1": 0.6434, "weighted_f1": 0.8262,
                                 "macro_precision": 0.7122, "macro_recall": 0.6217,
                                 "macro_auc": 0.8949, "macro_ap": 0.6836, "ece": 0.0327},
    ("test", CFG.INFER_TTA):    {"acc": 0.8497, "macro_f1": 0.6627, "weighted_f1": 0.8400,
                                 "macro_precision": 0.7354, "macro_recall": 0.6375,
                                 "macro_auc": 0.9099, "macro_ap": 0.6767, "ece": 0.0239},
}
for (split, inf), ref in PAPER_OVERALL.items():
    for k, v in ref.items():
        add("paper_reference", k, v, split=split, inference=inf, note="paper Tables 3/4/7")

PAPER_CLASS = {
    CFG.INFER_NO_TTA: {
        "No_DR": (0.9751, 0.9849, 0.9800), "Mild": (0.6154, 0.5333, 0.5714),
        "Moderate": (0.6847, 0.8736, 0.7677), "Severe": (0.4286, 0.3529, 0.3871),
        "Proliferative": (0.8571, 0.3636, 0.5106),
    },
    CFG.INFER_TTA: {
        "No_DR": (0.9799, 0.9799, 0.9799), "Mild": (0.7500, 0.6000, 0.6667),
        "Moderate": (0.6957, 0.9195, 0.7921), "Severe": (0.3846, 0.2941, 0.3333),
        "Proliferative": (0.8667, 0.3939, 0.5417),
    },
}
for inf, table in PAPER_CLASS.items():
    for cname, (p, r, f1) in table.items():
        add("paper_reference_per_class", "precision", p,  split="test", inference=inf, cls=cname)
        add("paper_reference_per_class", "recall",    r,  split="test", inference=inf, cls=cname)
        add("paper_reference_per_class", "f1",        f1, split="test", inference=inf, cls=cname)

CSV_PATH = os.path.join(CFG.OUT_DIR, f"{CFG.TAG}_results.csv")
pd.DataFrame(ROWS, columns=["section", "fold", "split", "inference", "class",
                            "epoch", "metric", "value", "note"]
             ).to_csv(CSV_PATH, index=False)
print(f"\nSaved single CSV: {CSV_PATH}")
print(f"Rows: {len(ROWS)}")
print("Done. Send me the CSV + console log.")

Device cuda | GPUs ['Tesla T4', 'Tesla T4'] | torch 2.11.0+cu128 | py 3.13.15 | cfg 73b07f048655
Split integrity OK.
train n= 2930 | counts [1434, 300, 808, 154, 234] | paper [1434, 300, 808, 154, 234] | MATCH
val   n=  366 | counts [172, 40, 104, 22, 28] | paper [171, 40, 103, 22, 28] | DIFFERS
test  n=  366 | counts [199, 30, 87, 17, 33] | paper [199, 30, 87, 17, 33] | MATCH


cache train:   0%|          | 0/2930 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/train_v3_crop_pad_noclahe_s320_7c8b32220dea.npy


cache val:   0%|          | 0/366 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/val_v3_crop_pad_noclahe_s320_94ad448327e2.npy


cache test:   0%|          | 0/366 [00:00<?, ?it/s]

cache saved: /tmp/dr_cache/test_v3_crop_pad_noclahe_s320_b0eba48336f4.npy
pool n=3296 counts [1606, 340, 912, 176, 262] | test n=366 | X_POOL (3296, 320, 320, 3)
fold 0: train 2636 | val 660 | val counts [321, 68, 183, 36, 52]
fold 1: train 2637 | val 659 | val counts [321, 68, 183, 35, 52]
fold 2: train 2637 | val 659 | val counts [321, 68, 182, 35, 53]
fold 3: train 2637 | val 659 | val counts [321, 68, 182, 35, 53]
fold 4: train 2637 | val 659 | val counts [322, 68, 182, 35, 52]

========================= FOLD 1/5 | train 2636 | val 660 =========================
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 164MB/s]


[fold 0] class weights: [0.5, 1.086, 0.664, 1.514, 1.236]
[f0 ep 01] tr_loss 0.7196 tr_acc_clean 62.2 | EMA val_acc 76.67 F1 0.4767 QWK 0.7915 NLL 0.7586 ECE 0.1752 | raw acc 76.82 QWK 0.8154 | lr 5.6e-05/2.8e-04 | 110s
[f0 ep 02] tr_loss 0.4864 tr_acc_clean 76.0 | EMA val_acc 79.09 F1 0.5927 QWK 0.8613 NLL 0.5621 ECE 0.0606 | raw acc 78.48 QWK 0.8830 | lr 1.5e-04/7.6e-04 | 26s
[f0 ep 03] tr_loss 0.4555 tr_acc_clean 77.2 | EMA val_acc 81.67 F1 0.6037 QWK 0.8871 NLL 0.4910 ECE 0.0408 | raw acc 79.85 QWK 0.8922 | lr 2.0e-04/1.0e-03 | 27s
[f0 ep 04] tr_loss 0.3911 tr_acc_clean 81.2 | EMA val_acc 83.03 F1 0.6679 QWK 0.9098 NLL 0.4689 ECE 0.0428 | raw acc 82.42 QWK 0.8943 | lr 2.0e-04/1.0e-03 | 27s
[f0 ep 05] tr_loss 0.3607 tr_acc_clean 82.1 | EMA val_acc 83.48 F1 0.6747 QWK 0.9122 NLL 0.4352 ECE 0.0434 | raw acc 81.36 QWK 0.9079 | lr 2.0e-04/9.9e-04 | 27s
[f0 ep 06] tr_loss 0.3215 tr_acc_clean 84.5 | EMA val_acc 84.70 F1 0.7064 QWK 0.9159 NLL 0.4263 ECE 0.0354 | raw acc 80.91 QWK 0.8920 | 